<a id="part-i"></a>
<div style="padding:30px 32px;border-radius:18px;background:linear-gradient(125deg,#101827 0%,#182e4b 52%,#134d56 100%);color:#f8fafc">
  <div style="font-size:12px;letter-spacing:3px;color:#67e8f9;font-weight:700">PART I · FORENSIC TEXT ANALYSIS</div>
  <h1 style="font-size:31px;line-height:1.2;color:#ffffff;margin:16px 0 10px">🧠 Name That LLM<br>The Laboratory of 18 Voices</h1>
  <p style="font-size:17px;line-height:1.6;color:#cbd5e1;margin:0">Deep Forensic EDA: separating the model's writing signature from the prompt's subject</p>
</div>

<div style="border:1px solid #D5E4EB;background:#F5FAFC;border-radius:12px;padding:18px;margin:18px 0 20px">
  <div style="font-size:11px;color:#527285;letter-spacing:1.4px;font-weight:800;margin-bottom:13px">RESEARCH MAP</div>
  <div style="display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:9px">
    <a href="#suspects" style="padding:12px;border:1px solid #D2E1E9;border-radius:8px;background:#fff;text-decoration:none;color:#225F76;font-weight:700;font-size:13px">01 · The Suspects</a>
    <a href="#rhythm" style="padding:12px;border:1px solid #D2E1E9;border-radius:8px;background:#fff;text-decoration:none;color:#225F76;font-weight:700;font-size:13px">02 · Rhythm</a>
    <a href="#punctuation" style="padding:12px;border:1px solid #D2E1E9;border-radius:8px;background:#fff;text-decoration:none;color:#225F76;font-weight:700;font-size:13px">03 · Punctuation</a>
    <a href="#lineup" style="padding:12px;border:1px solid #D2E1E9;border-radius:8px;background:#fff;text-decoration:none;color:#225F76;font-weight:700;font-size:13px">04 · The Lineup</a>
    <a href="#evidence-test" style="padding:12px;border:1px solid #D2E1E9;border-radius:8px;background:#fff;text-decoration:none;color:#225F76;font-weight:700;font-size:13px">05 · Testing the Evidence</a>
    <a href="#part-ii" style="padding:12px;border:1px solid #D2E1E9;border-radius:8px;background:#fff;text-decoration:none;color:#225F76;font-weight:700;font-size:13px">Part II · The Ensemble</a>
    <a href="#version-trail" style="padding:12px;border:1px solid #D2E1E9;border-radius:8px;background:#fff;text-decoration:none;color:#225F76;font-weight:700;font-size:13px">Version Trail</a>
    <a href="#credits" style="padding:12px;border:1px solid #D2E1E9;border-radius:8px;background:#fff;text-decoration:none;color:#225F76;font-weight:700;font-size:13px">Credits & Sources</a>
  </div>
</div>

> **Eighteen performers. One score.** A prompt sets the melody, but each model has its own rhythm, vocabulary and emphasis. The mystery is not *what* a response says but *which model* could have written it.

Five investigations examine the training structure, response lengths, punctuation habits, answers from different models to **the same prompt**, and how validation splits change the conclusions. Matching the prompt helps separate task content from a model's writing style.

**Our guiding question:** does a stylistic clue hold across independent prompts, or does it only seem convincing in a few selected examples?


In [ ]:

# 00 · Laboratory setup and data contract
from pathlib import Path
from IPython.display import display, HTML
import html, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

plt.rcParams.update({
    "font.family": "DejaVu Sans", "figure.dpi": 120, "savefig.dpi": 150,
    "font.size": 10, "axes.titlesize": 13, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#CBD5E1", "axes.labelcolor": "#334155",
    "xtick.color": "#64748B", "ytick.color": "#334155",
    "grid.color": "#E2E8F0", "figure.facecolor": "white", "axes.facecolor": "white",
})

INK, TEAL, BLUE, GOLD, RED = "#17253B", "#0F9D92", "#5365D8", "#E6A23C", "#D86170"
FAMILY_PALETTE = {
    "GPT": "#5365D8",
    "Claude": "#CD705A",
    "Gemini": "#0F9D92",
    "GLM": "#A56CC1",
    "Qwen": "#E6A23C",
    "Kimi": "#78909C",
}

def family_of(name):
    name = name.lower()
    return next(
        (fam for token, fam in [
            ("gpt", "GPT"), ("claude", "Claude"),
            ("gemini", "Gemini"), ("glm", "GLM"),
            ("qwen", "Qwen"), ("kimi", "Kimi")
        ] if name.startswith(token)),
        "Other"
    )

def locate_official(filename):
    roots = [
        Path("/kaggle/input/competitions/name-that-llm"),
        Path("/kaggle/input/name-that-llm"),
        Path("/mnt/data"),
        Path("."),
    ]
    candidates = [r / filename for r in roots]

    if filename == "train.csv":
        candidates.insert(-1, Path("/mnt/data/train (2).csv"))

    hit = next((p for p in candidates if p.exists()), None)
    if hit is None:
        raise FileNotFoundError(
            f"Official {filename} not found. "
            "Attach the Name That LLM competition dataset."
        )
    return hit

# EDA uses training data only.
train_path = locate_official("train.csv")
train = pd.read_csv(train_path)

assert {"id", "prompt_id", "text", "model"}.issubset(train.columns)
assert train["id"].is_unique
assert not train[["id", "prompt_id", "text", "model"]].isna().any().any()

train["family"] = train["model"].map(family_of)
train["n_chars"] = train["text"].str.len()
train["n_words"] = train["text"].str.split().str.len()

model_order = sorted(
    train["model"].unique(),
    key=lambda x: (list(FAMILY_PALETTE).index(family_of(x)), x)
)

def card(label, value, foot=""):
    return (
        f'<div style="background:#f3f7fb;border:1px solid #e0e7ef;'
        f'border-radius:13px;padding:14px 16px;flex:1;min-width:130px">'
        f'<div style="color:#64748b;font-size:12px">{label}</div>'
        f'<div style="font-size:27px;font-weight:750;color:#17253b;'
        f'margin:4px 0">{value}</div>'
        f'<div style="font-size:11px;color:#64748b">{foot}</div></div>'
    )

def panel(title, text, color=TEAL):
    display(HTML(
        f'<div style="border-left:4px solid {color};border-radius:7px;'
        f'background:#f4f8fb;padding:13px 17px;margin:12px 0">'
        f'<b style="color:#17253b">{title}</b>'
        f'<div style="font-size:13px;line-height:1.6;color:#475569;'
        f'margin-top:5px">{text}</div></div>'
    ))

prompt_counts = train.groupby("prompt_id")["model"].nunique()
missing_cells = (
    len(prompt_counts) * train["model"].nunique()
) - len(train)

median_chars = int(train["n_chars"].median())
median_words = int(train["n_words"].median())

display(HTML(
    '<div style="display:flex;flex-wrap:wrap;gap:10px;margin:16px 0">'
    + ''.join([
        card(
            "Responses examined",
            f"{len(train):,}".replace(",", " "),
            "official training set"
        ),
        card("Models", train["model"].nunique(), "attribution classes"),
        card("Unique prompts", train["prompt_id"].nunique(), "matched comparisons"),
        card("Median length", f"{median_chars}", "characters per response"),
    ])
    + '</div>'
))

print("Train:", train_path, "| shape:", train.shape)
print("Missing prompt × model combinations:", missing_cells)
print("Exact duplicate texts:", int(train["text"].duplicated().sum()))


<a id="suspects"></a>
## 01. The Suspects — An Almost Perfect Chessboard

Imagine **150 musical scores** performed by **18 different musicians**. Nearly every musician plays every piece. For authorship attribution, that is an unusually informative arrangement: we can compare model responses to the same task rather than mistaking the topic for the author's style.

**The question.** How complete is this experiment? Are the classes so imbalanced that simply favoring a frequent author could help?

The first panel counts responses from each model. The second is a `prompt_id × model` board: a colored square marks a response that exists; a missing square marks an absent generation.

**Why it matters:** this structure makes paired stylometric comparisons possible. It also changes what a random train/validation split means — something we will examine in the final experiment.


In [ ]:
# 01 · Class coverage and the prompt-by-model board
coverage = pd.crosstab(train["model"], train["prompt_id"]).reindex(model_order)
counts = train["model"].value_counts().reindex(model_order)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={"width_ratios": [1.05, 2.0]})
colors = [FAMILY_PALETTE[family_of(m)] for m in model_order]
ax1.barh(model_order, counts, color=colors, height=.67)
ax1.invert_yaxis()
ax1.set_xlim(0, max(counts) * 1.12)
ax1.set_xlabel("Responses per model")
ax1.set_title("A · Class balance", loc="left", fontweight="bold", color=INK, pad=15)
ax1.axvline(150, color="#AAB8C8", ls="--", lw=1)
for i, count in enumerate(counts):
    ax1.text(count + 1, i, str(count), va="center", fontsize=8.5, color=INK)

ax2.imshow((coverage.to_numpy() > 0).astype(int), aspect="auto", interpolation="nearest",
           cmap=ListedColormap(["#E57A83", "#0F9D92"]), vmin=0, vmax=1)
ax2.set_yticks(range(len(model_order)), labels=model_order, fontsize=8)
xt = np.linspace(0, coverage.shape[1] - 1, 7, dtype=int)
ax2.set_xticks(xt, [coverage.columns[j] for j in xt], rotation=0)
ax2.set_xlabel("Prompt ID (p000 to p149)")
ax2.set_title("B · Which model answered which prompt", loc="left", fontweight="bold", color=INK, pad=15)
plt.tight_layout(w_pad=3)
plt.show()

missing = np.argwhere(coverage.to_numpy() == 0)
if len(missing):
    detail = "; ".join(f"{coverage.index[i]} × {coverage.columns[j]}" for i, j in missing)
    panel("One missing pairing", html.escape(detail), GOLD)
else:
    panel("Complete prompt board", "Every prompt has a response from every class.")
print("Prompts with all models:", int((prompt_counts == train['model'].nunique()).sum()), "/", len(prompt_counts))

<a id="rhythm"></a>
## 02. Length — A Signature or Simply the Task's Demands?

One musician rushes through a passage; another lets every note linger. Text has a rhythm too: how many words a model uses to develop an idea, and how long its answers become.

But a long answer may simply reflect a demanding prompt. So we examine two views of the same signal:

- **Absolute length:** the median number of characters produced by each model.
- **Within-prompt residual:** the response length minus the mean length of answers to that prompt. This approximately removes the shared influence of topic and difficulty.

The intervals in the right-hand chart estimate uncertainty in the mean across prompts (normal approximation, ±1.96 SE); they do not show the full range of responses. **We are looking for persistent tendencies, not declaring the most verbose model identified.**


In [ ]:
# 02 · Raw length versus within-prompt length residuals
train["length_vs_prompt"] = train["n_chars"] - train.groupby("prompt_id")["n_chars"].transform("mean")
length_stats = train.groupby("model")["n_chars"].agg(median="median", low=lambda z: z.quantile(.25), high=lambda z: z.quantile(.75))
residual_stats = train.groupby("model")["length_vs_prompt"].agg(["mean", "std", "count"])
models = sorted(model_order, key=lambda m: length_stats.loc[m, "median"])
colors = [FAMILY_PALETTE[family_of(m)] for m in models]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15.5, 7.2), gridspec_kw={"width_ratios": [1.03, 1]})
ypos = np.arange(len(models))
q25 = length_stats.loc[models, "low"].to_numpy()
q75 = length_stats.loc[models, "high"].to_numpy()
med = length_stats.loc[models, "median"].to_numpy()
ax1.hlines(ypos, q25, q75, color=colors, linewidth=6, alpha=.45)
ax1.scatter(med, ypos, color=colors, s=42, zorder=3, edgecolors="white", linewidths=.65)
ax1.set_yticks(ypos, models, fontsize=8.5)
ax1.invert_yaxis()
ax1.set_xlabel("Characters per response · median and IQR")
ax1.set_title("A · Absolute length", loc="left", fontweight="bold", color=INK, pad=16)

mean = residual_stats.loc[models, "mean"].to_numpy()
err = 1.96 * residual_stats.loc[models, "std"].to_numpy() / np.sqrt(residual_stats.loc[models, "count"].to_numpy())
ax2.barh(ypos, mean, color=colors, alpha=.87, height=.58)
ax2.errorbar(mean, ypos, xerr=err, fmt="none", ecolor=INK, alpha=.8, capsize=2, lw=1)
ax2.axvline(0, color=INK, lw=1.2)
ax2.set_yticks(ypos, [""] * len(ypos))
ax2.invert_yaxis()
ax2.set_xlabel("Length residual relative to prompt mean (characters)")
ax2.set_title("B · After controlling for prompt", loc="left", fontweight="bold", color=INK, pad=16)
plt.tight_layout(w_pad=3)
plt.show()

fastest, slowest = models[0], models[-1]
panel("The length signature", f"Shortest median: <b>{html.escape(fastest)}</b> — {med[0]:.0f} chars; "
      f"longest: <b>{html.escape(slowest)}</b> — {med[-1]:.0f} chars. "
      "The right-hand chart shows how much of the difference remains when responses to identical prompts are compared.")

<a id="punctuation"></a>
## 03. The Fingerprints That Survive Formatting Removal

In a detective film, fingerprints may settle the case. Here the obvious ones have almost vanished: **no Markdown headings, bullet lists, fenced code blocks, or line breaks** appear in the training responses. They have been flattened into continuous text. A classifier counting Markdown headings alone would find nothing useful.

Yet subtler habits remain: **em dashes**, semicolons, colons, commas, word length, and sentence boundaries.

We count punctuation marks per **1,000 characters**, long words per **100 words**, and then subtract each prompt's mean feature value. The result is a matrix of stylistic residuals, less dominated by the subject and the overall length of the response.

The **heatmap** reveals the direction of each model's characteristic deviation. The chart beside it shows the share of variance associated with model identity after prompt centering (`η²`).


In [ ]:
# 03 · Punctuation, lexical rhythm, and prompt-centered effect size
s = train["text"].fillna("")
char_denom = train["n_chars"].clip(lower=1)
word_denom = train["n_words"].clip(lower=1)

def per_1k(symbol):
    return 1000 * s.str.count(re.escape(symbol)) / char_denom

signals = pd.DataFrame({
    "Em dash —": per_1k("—"),
    "Semicolon": per_1k(";"),
    "Colon": per_1k(":"),
    "Comma": per_1k(","),
    "Period": per_1k("."),
    "Hyphen -": per_1k("-"),
    "Double quote": per_1k('"'),
    "Question mark": per_1k("?"),
    "Long words ≥9": 100 * s.str.count(r"\b[A-Za-z]{9,}\b") / word_denom,
    "Sentence boundaries": 1000 * s.str.count(r"[.!?]+") / char_denom,
})
centered = signals - signals.groupby(train["prompt_id"]).transform("mean")
mean_by_model = centered.groupby(train["model"]).mean().reindex(model_order)
counts_by_model = train["model"].value_counts().reindex(model_order)

def model_eta_squared(series):
    grand = series.mean()
    between = (((series.groupby(train["model"]).mean() - grand) ** 2)
               * counts_by_model).sum()
    total = ((series - grand) ** 2).sum()
    return float(between / total) if total > 1e-12 else 0.0

ranked = pd.Series({feature: model_eta_squared(centered[feature]) for feature in signals}).sort_values(ascending=False)
top = ranked.head(8).index.tolist()
std = centered[top].std().replace(0, 1)
z = (mean_by_model[top] / std).clip(-1.4, 1.4)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={"width_ratios": [1.75, .95]})
im = ax1.imshow(z.to_numpy(), cmap="RdBu_r", vmin=-1.4, vmax=1.4, aspect="auto")
ax1.set_yticks(range(len(model_order)), model_order, fontsize=8.2)
ax1.set_xticks(range(len(top)), [t.replace(" ", "\n", 1) for t in top], fontsize=8)
ax1.set_title("A · Style signatures after prompt centering", loc="left", fontweight="bold", color=INK, pad=15)
cb = fig.colorbar(im, ax=ax1, fraction=.022, pad=.015)
cb.set_label("Deviation / SD", fontsize=9)
ax2.barh(top[::-1], ranked.loc[top[::-1]].to_numpy(), color=[TEAL if i == 0 else BLUE for i in range(len(top))], height=.62)
ax2.set_xlim(0, max(.10, ranked.max() * 1.15))
ax2.set_xlabel("Prompt-centered η²")
ax2.set_title("B · Which features distinguish models", loc="left", fontweight="bold", color=INK, pad=15)
for i, f in enumerate(top[::-1]):
    ax2.text(ranked.loc[f] + .008, i, f"{ranked.loc[f]:.3f}", va="center", fontsize=9, color=INK)
plt.tight_layout(w_pad=3)
plt.show()

zero_markdown = {"Markdown headings": int(s.str.contains(r"(?m)^#{1,6}\s").sum()),
                 "Bullet lists": int(s.str.contains(r"(?m)^\s*[-*+]\s+\S").sum()),
                 "Code blocks": int(s.str.contains('```', regex=False).sum()),
                 "Line breaks": int(s.str.contains("\n", regex=False).sum())}
print("Formatting markers in 2,699 responses:", zero_markdown)
top_feature = ranked.index[0]
panel("The strongest clue", f"The strongest stylometric dimension in these measurements is "
      f"<b>{html.escape(top_feature)}</b> (η² = {ranked.iloc[0]:.3f}). "
      "This is a substantial between-class difference in prompt-centered training data, "
      "not a guarantee of accurate attribution by itself.")

<a id="lineup"></a>
## 04. The Lineup — One Prompt, Eighteen Voices

Picture all eighteen suspects receiving **exactly the same question**. Their answers will inevitably discuss similar things, making differences in phrasing and structure easier to notice.

Instead of selecting the most extreme prompt, we choose one with a **typical spread of response lengths**. Below are genuine excerpts from six different model families, alongside the lengths of all eighteen answers.

These examples are qualitative evidence, not a classification score. They can inspire hypotheses, but only repeated measurements across many prompts can test them.


In [ ]:
# 04 · Paired responses: one shared prompt, six model families
complete_prompts = prompt_counts[prompt_counts == train["model"].nunique()].index
spread = train[train["prompt_id"].isin(complete_prompts)].groupby("prompt_id")["n_chars"].agg(
    lambda values: values.quantile(.90) - values.quantile(.10)
)
selected_prompt = (spread - spread.median()).abs().idxmin()
case = train.loc[train["prompt_id"] == selected_prompt].copy()
case["em_dashes"] = case["text"].str.count("—")
case = case.set_index("model")

fig, ax = plt.subplots(figsize=(13.8, 5.4))
ordered_case = case.sort_values("n_chars")
ax.barh(ordered_case.index, ordered_case["n_chars"],
        color=[FAMILY_PALETTE[family_of(m)] for m in ordered_case.index], height=.68)
ax.invert_yaxis()
ax.set_xlabel("Response length (characters)")
ax.set_title(f"Prompt {selected_prompt} · different tempos for the same score", loc="left", fontweight="bold", color=INK, pad=16)
for i, (_, row) in enumerate(ordered_case.iterrows()):
    ax.text(row["n_chars"] + 8, i, f"{row['n_chars']} · em dashes {row['em_dashes']}", fontsize=8, va="center", color=INK)
ax.set_xlim(0, ordered_case["n_chars"].max() * 1.22)
plt.tight_layout()
plt.show()

examples = []
for fam in FAMILY_PALETTE:
    candidates = [m for m in case.index if family_of(m) == fam]
    if not candidates:
        continue
    chosen = sorted(candidates)[0]
    row = case.loc[chosen]
    quote = html.escape(row["text"][:275].strip()) + ("…" if len(row["text"]) > 275 else "")
    color = FAMILY_PALETTE[fam]
    examples.append(
        f'<div style="border:1px solid #e0e7ef;border-top:4px solid {color};background:#f8fafc;'
        f'border-radius:12px;padding:12px 15px;min-width:240px;flex:1">'
        f'<div style="font-size:12px;font-weight:700;color:{color}">{html.escape(chosen)}</div>'
        f'<div style="font-size:11px;color:#64748b;margin:4px 0 10px">'
        f'{int(row["n_chars"])} characters · em dashes: {int(row["em_dashes"])}</div>'
        f'<div style="font-size:12px;line-height:1.6;color:#243247">“{quote}”</div></div>')
display(HTML('<div style="display:flex;flex-wrap:wrap;gap:10px;margin:16px 0">' + ''.join(examples) + '</div>'))
panel("What stays constant", f"All displayed responses share the same <code>prompt_id = {selected_prompt}</code>. "
      "The original prompt text is not included in the official CSV; we do not attempt to reconstruct it.")

<a id="evidence-test"></a>
## 05. The Validation Trap — When the Familiar Rule Reverses

When several authors respond to the same prompts, their texts share a substantial amount of content. In a random split, some responses to a prompt appear in training while others appear in validation. That is not the same as testing on **unseen prompts**.

We can measure the difference by holding the detector fixed and changing only the split:

| Experiment | Train / validation | What it measures |
|---|---|---|
| **A. Random stratified** | Responses to one prompt can appear on both sides | Generalization to other answers, potentially on familiar topics |
| **B. Prompt-grouped** | Every response to a prompt stays in a single fold | Generalization to previously unseen prompts |

Our detector is deliberately simple: **character 3–4-gram TF-IDF + Linear SVM**, evaluated with three folds. The vectorizer is refitted inside each training fold. We report out-of-fold Balanced Accuracy and confusion between the 18 models. **We do not assume which split will score higher:** this dataset's unusual structure may produce a counterintuitive result.


In [ ]:
# 05 · Random vs prompt-grouped CV + forensic confusion matrix (CPU)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.svm import LinearSVC
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, recall_score

texts = train["text"].to_numpy()
y = train["model"].to_numpy()
groups = train["prompt_id"].to_numpy()

strategies = {
    "Random stratified": StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    "Prompt-grouped": StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42),
}
results, predictions = {}, {}
for name, cv in strategies.items():
    oof = np.empty(len(train), dtype=object)
    folds = []
    indices = cv.split(texts, y, groups if name == "Prompt-grouped" else None)
    for fold, (tr_idx, va_idx) in enumerate(indices, 1):
        if name == "Prompt-grouped":
            assert not set(groups[tr_idx]).intersection(groups[va_idx])
        vectorizer = TfidfVectorizer(
            analyzer="char", ngram_range=(3, 4), min_df=3,
            max_features=22000, sublinear_tf=True, dtype=np.float32,
        )
        x_tr = vectorizer.fit_transform(texts[tr_idx])
        x_va = vectorizer.transform(texts[va_idx])
        clf = LinearSVC(C=1.0, dual="auto", class_weight="balanced", max_iter=1500)
        clf.fit(x_tr, y[tr_idx])
        oof[va_idx] = clf.predict(x_va)
        folds.append(balanced_accuracy_score(y[va_idx], oof[va_idx]))
        print(f"{name:<19} | fold {fold} | Balanced Accuracy = {folds[-1]:.4f}")
    results[name] = {"oof": balanced_accuracy_score(y, oof), "folds": folds}
    predictions[name] = oof

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={"width_ratios": [.95, 1.65]})
labels = list(results)
values = [results[k]["oof"] for k in labels]
ax1.barh(labels[::-1], values[::-1], color=[TEAL, BLUE], height=.5)
for i, value in enumerate(values[::-1]):
    ax1.text(value + .007, i, f"{value:.3f}", va="center", fontsize=15, fontweight="bold", color=INK)
ax1.set_xlim(0, max(values) * 1.34)
ax1.set_xlabel("OOF Balanced Accuracy")
ax1.set_title("A · One detector, two CV strategies", loc="left", fontweight="bold", color=INK, pad=17)
ax1.grid(axis="x", alpha=.5)

cm = confusion_matrix(y, predictions["Prompt-grouped"], labels=model_order)
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
im = ax2.imshow(cm_norm, cmap="Blues", vmin=0, vmax=max(.33, cm_norm.max()), aspect="auto")
ax2.set_xticks(range(len(model_order)), model_order, rotation=90, fontsize=7)
ax2.set_yticks(range(len(model_order)), model_order, fontsize=7)
ax2.set_xlabel("Predicted model")
ax2.set_ylabel("True model")
ax2.set_title("B · Confusion under prompt-grouped CV", loc="left", fontweight="bold", color=INK, pad=17)
fig.colorbar(im, ax=ax2, shrink=.74, fraction=.03, label="Fraction of class responses")
plt.tight_layout(w_pad=3.5)
plt.show()

pair_errors = []
for i, actual in enumerate(model_order):
    for j, guessed in enumerate(model_order):
        if i != j and cm[i, j] > 0:
            pair_errors.append((int(cm[i, j]), actual, guessed))
pair_errors.sort(reverse=True)
error_rows = ''.join(
    f'<tr><td style="padding:5px 10px">{html.escape(a)}</td>'
    f'<td style="padding:5px 10px">{html.escape(b)}</td>'
    f'<td style="padding:5px 10px;text-align:right">{n}</td></tr>'
    for n, a, b in pair_errors[:5]
)
display(HTML('<div style="max-width:680px;margin:12px 0">'
             '<div style="font-weight:700;color:#17253b;margin:8px 0">Five most frequent directed confusions</div>'
             '<table style="border-collapse:collapse;width:100%;font-size:12px">'
             '<thead><tr style="background:#e8f0f7"><th align="left">True author</th>'
             '<th align="left">Predicted as</th><th align="right">Count</th></tr></thead>'
             f'<tbody>{error_rows}</tbody></table></div>'))

r, g = results["Random stratified"]["oof"], results["Prompt-grouped"]["oof"]
if g > r:
    finding = (f"In this experiment, prompt-grouped CV scored <b>{g:.3f}</b>, "
               f"versus <b>{r:.3f}</b>. The surprising result: "
               "here the random split <i>decreases</i>, rather than increasing this TF-IDF detector’s score. "
               "One possible explanation is competing labels among content-similar responses "
               "to a single prompt; this remains a hypothesis to examine.")
else:
    finding = (f"In this experiment, random stratified CV scored <b>{r:.3f}</b>, "
               f"versus prompt-grouped <b>{g:.3f}</b>. The split strategy materially changes the estimate; "
               "grouped CV better reflects evaluation on unseen prompts.")
panel("The laboratory’s final finding", finding, TEAL)


<a id="part-ii"></a>
<div style="padding:28px 32px;border-radius:18px;background:linear-gradient(125deg,#112034 0%,#203759 50%,#1d5860 100%);color:#f8fafc;margin:22px 0 14px">
 <div style="font-size:12px;letter-spacing:3px;color:#7ee6de;font-weight:700">PART II · MODEL ATTRIBUTION</div>
 <h1 style="font-size:29px;line-height:1.25;color:#ffffff;margin:14px 0 9px">⚖️ From Clues to a Verdict</h1>
 <p style="font-size:16px;line-height:1.55;color:#d5e4f1;margin:0">A clean LoRA ensemble, native embedding readouts and calibrated probability fusion</p>
 <div style="margin-top:18px;font-size:12px;color:#b5ede9">Sparse Text &nbsp;·&nbsp; Clean LoRA &nbsp;·&nbsp; ATLAS &nbsp;·&nbsp; GSQ &nbsp;·&nbsp; Qwen3 Embedding &nbsp;·&nbsp; Sinkhorn</div>
</div>

The forensic experiments above investigate author signals while controlling for the prompt. The inference stack takes a different route: it combines fitted sparse-text evidence, multidepth LoRA features, a five-fold native embedding expert and final class-distribution calibration.

Its executable pipeline follows the public **0.684 Clean LoRA + Qwen3-Embedding** reference, with the independent forensic investigation retained as Part I.


## 06. A Larger Orchestra — Clean LoRA and Native Embedding

A response carries several overlapping signals: lexical habits, structure, semantic context and patterns visible in deeper transformer layers. The inference ensemble combines five backbone readers with prefit text statistics, a Qwen3-Embedding-8B expert and a specialized Qwen3.8-27B stage.

A central distinction from the previous Sovereign Titan stack is the **clean Qwen3-14B adapter blend**: the `n14b_clean` branch uses the public checkpoints `ntl-q14-lora-v4` and `lora_qwen3_14b` with coefficients **0.72 / 0.28**. The final fusion uses **40% native embedding**, **3% GSQ inside the parent**, and a **15% complementary stack**.


## 07. The Evidence Locker — Assembling Fitted Assets

The execution begins by locating publicly mounted models and adapters. Required files are resolved by their contents or manifest names rather than assuming a single Kaggle directory layout.

The fitted `ntl666-prefit` bundle is reconstructed from declared archive parts. Its internal file list and provenance record are checked before the readout stages begin. A tiny public sample activates the lightweight format path; a full competition run enters the inference pipeline.


In [ ]:
import glob
import json
import os
import re
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd

NB_T0 = time.time()
RUN_LIMIT_S = 8.0 * 3600

wheels = glob.glob('/kaggle/input/**/bitsandbytes-*.whl', recursive=True)
if wheels:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '-q'] + wheels, capture_output=True, text=True)

configs = sorted(glob.glob('/kaggle/input/**/config.json', recursive=True))


def model_dir(*parts):
    hits = [os.path.dirname(c) for c in configs if all(p in c.lower() for p in parts)]
    assert hits, parts
    return hits[0]


def adapter_dir(name):
    hits = glob.glob(f'/kaggle/input/**/{name}/heads.pt', recursive=True)
    assert hits, name
    return os.path.dirname(hits[0])


def meta_dir(name):
    hits = glob.glob(f'/kaggle/input/**/{name}/meta.json', recursive=True)
    assert hits, name
    return os.path.dirname(hits[0]), json.load(open(hits[0]))['cfg']


def first(pattern):
    hits = glob.glob(pattern, recursive=True)
    assert hits, pattern
    return hits[0]


DATA = next(p for p in ['/kaggle/input/competitions/name-that-llm', '/kaggle/input/name-that-llm'] if os.path.exists(p + '/train.csv'))
FAST_COMMIT = len(pd.read_csv(DATA + '/test.csv')) < 50
print('FAST_COMMIT', FAST_COMMIT)

open_pool = pd.read_parquet(first('/kaggle/input/**/ntl_open_pool.parquet'), columns=['text', 'model', 'flag_train_prompt_regen', 'match_weight'])
open_pool = open_pool[~open_pool.flag_train_prompt_regen.astype(bool)]
open_pool[['text', 'model']].to_parquet('/kaggle/working/open_pool.parquet')
POOL = '/kaggle/working/open_pool.parquet'
print('svc pool rows', len(open_pool))


def draw(g, n):
    w = np.nan_to_num(g.match_weight.to_numpy(dtype=np.float64))
    return g.take(np.random.RandomState(0).choice(len(g), size=n, replace=False, p=w / w.sum()).astype(np.intp))


matched_counts = pd.read_parquet(first('/kaggle/input/**/nlm_matched_clean.parquet'), columns=['model']).model.value_counts()
extra = pd.concat([draw(g, min(len(g), int(matched_counts.get(m, 0)))) for m, g in open_pool.groupby('model')])
EXTRA = '/kaggle/working/extra_matched.parquet'
extra[['text', 'model']].to_parquet(EXTRA)
print('external rows', len(extra), 'matched_clean rows', int(matched_counts.sum()))
del open_pool, extra

B8 = model_dir('/qwen-3/', '/8b-base/')
B14 = model_dir('/qwen-3/', '/14b-base/')
B25_14 = model_dir('/qwen2.5/', '/14b/')
BL = model_dir('/llama-3-1-8b-base/')
SHARDS = {'model-00001-of-00004.safetensors': 4976698672, 'model-00002-of-00004.safetensors': 4999802720,
          'model-00003-of-00004.safetensors': 4915916176, 'model-00004-of-00004.safetensors': 1168138808}
assert all(os.path.getsize(os.path.join(BL, f)) == n for f, n in SHARDS.items()), BL
if not FAST_COMMIT:
    shutil.copytree(BL, '/tmp/llama-3.1/8b', dirs_exist_ok=True)
    BL = '/tmp/llama-3.1/8b'

AQ8V5 = [adapter_dir(f'q8v5_f{k}') for k in range(5)] + [adapter_dir('q8v5')]
AQ14V5 = [adapter_dir(f'q14v5_f{k}') for k in range(5)] + [adapter_dir('q14v5')]
ALL8V5 = [adapter_dir(f'll8v5_f{k}') for k in range(5)] + [adapter_dir('ll8v5')]
N14B, N14BC = meta_dir('lora_qwen3_14b')
Q25_14, Q25_14C = meta_dir('lora_qwen25_14b')
OQ1 = os.path.dirname(first('/kaggle/input/**/ntl-q14-lora/heads.safetensors'))
OQ4 = os.path.dirname(first('/kaggle/input/**/ntl-q14-lora-v4/heads.safetensors'))

NATIVE = os.path.dirname(first('/kaggle/input/**/ntl-native-q8-multidepth-fivefold-0679/manifest.json'))
NATIVE_BASE = model_dir('/qwen-3-embedding/', '/8b/')
assert all(os.path.exists(f'{NATIVE}/fold{k}/trainables.safetensors') and os.path.exists(f'{NATIVE}/calibration/fold{k}.json') for k in range(5)), NATIVE
assert os.path.exists(f'{NATIVE}/base/model.safetensors.index.json') and os.path.exists(f'{NATIVE_BASE}/model.safetensors.index.json'), (NATIVE, NATIVE_BASE)

LB_ROOT = os.path.dirname(first('/kaggle/input/**/ntl666-cache-parts.json'))
GSQ_ASSETS = LB_ROOT
PREFIT = '/kaggle/temp/ntl666-prefit'
import zipfile
from pathlib import PurePosixPath
parts_meta = json.loads(open(f'{LB_ROOT}/ntl666-cache-parts.json').read())
assert parts_meta['format'] == 'ntl666-cache-parts-v1'
assert [x['name'] for x in parts_meta['parts']] == [f'ntl666-prefit.part{i:03d}.bin' for i in range(len(parts_meta['parts']))]
os.makedirs('/kaggle/temp', exist_ok=True)
shutil.rmtree(PREFIT, ignore_errors=True)
prefit_zip = '/kaggle/temp/ntl666-prefit.zip'
try:
    with open(prefit_zip, 'wb') as joined:
        for part in parts_meta['parts']:
            assert os.path.getsize(f"{LB_ROOT}/{part['name']}") == part['bytes'], part['name']
            with open(f"{LB_ROOT}/{part['name']}", 'rb') as stream:
                shutil.copyfileobj(stream, joined, 16 * 1024 * 1024)
    assert os.path.getsize(prefit_zip) == parts_meta['archive_bytes']
    with zipfile.ZipFile(prefit_zip) as archive:
        entries = archive.infolist()
        assert len(entries) == parts_meta['files'] and len({e.filename for e in entries}) == len(entries)
        for entry in entries:
            path = PurePosixPath(entry.filename)
            assert not path.is_absolute() and '..' not in path.parts and chr(92) not in entry.filename and path.parts[0] == 'ntl666-prefit' and not entry.is_dir()
        assert archive.read('ntl666-prefit/COMPLETE') == b'ntl666-v14-data-numeric-v1' + chr(10).encode()
        provenance = json.loads(archive.read('ntl666-prefit/provenance.json'))
        assert provenance['status'] == 'COMPLETE' and provenance['pool_rows'] == 190588 and provenance['dataset_version'] == 13
        archive.extractall('/kaggle/temp')
finally:
    if os.path.exists(prefit_zip):
        os.remove(prefit_zip)
for name in ('core/svc4', 'core/tfidf', 'core/distort'):
    assert os.path.isdir(f'{PREFIT}/{name}'), name
assert os.path.exists(f'{GSQ_ASSETS}/gsq3-base/qwen38-27b-gsq3/config.json') and os.path.exists(f'{GSQ_ASSETS}/gsq120k-trained/gsq120k-inference/gsq/final.safetensors')
assert len(glob.glob(f'{GSQ_ASSETS}/name-that-llm-artifacts/qwen38-offline-linux/*.whl')) >= 6

DEB = os.path.dirname(first('/kaggle/input/**/deberta_v3_large/meta.json'))
if not FAST_COMMIT:
    shutil.copytree(DEB, '/kaggle/working/deberta_v3_large', dirs_exist_ok=True)
    DEB = '/kaggle/working/deberta_v3_large'

for name, value in dict(B8=B8, B14=B14, B25_14=B25_14, BL=BL, N14B=N14B, Q25_14=Q25_14, OQ1=OQ1, OQ4=OQ4, NATIVE=NATIVE, NATIVE_BASE=NATIVE_BASE, PREFIT=PREFIT, GSQ_ASSETS=GSQ_ASSETS, DEB=DEB).items():
    print(f'{name:10s} {value}')
print('setup', round(time.time() - NB_T0), 's')


## 08. An Orderly Examination — Concurrent Worker Stages

Large models share finite GPU memory. The worker engine coordinates feature extraction, releases resources between branches and passes intermediate arrays to the next stage without keeping all backbones resident at once.


In [ ]:
%%writefile pipe.py
import gc
import os
import threading
import time
from collections import deque

import numpy as np


def _flat(x):
    if isinstance(x, dict):
        parts = [x[k] for k in sorted(x)]
    else:
        parts = x if isinstance(x, (tuple, list)) else [x]
    return np.concatenate([np.ravel(np.asarray(p, dtype=np.float64)) for p in parts])


def _agree(batches, work, workers, tag, log):
    probe = [batches[len(batches) // 2 + i] for i in range(workers)]
    try:
        ref = [work(b) for b in probe]
        if any(r is None for r in ref):
            return False
        got = [None] * workers
        err = []

        def go(i):
            try:
                got[i] = work(probe[i])
            except Exception as e:
                err.append(e)

        threads = [threading.Thread(target=go, args=(i,)) for i in range(workers)]
        [t.start() for t in threads]
        [t.join() for t in threads]
        ok = not err and all(g is not None and np.abs(_flat(r) - _flat(g)).max() <= 2e-2 * (np.abs(_flat(r)).mean() + 1e-6) for r, g in zip(ref, got))
    except Exception as e:
        log(f'{tag}: concurrent check raised {e!r:.100}')
        ok = False
    log(f'{tag}: concurrent check {"passed" if ok else "failed, single worker"}')
    return ok


def pipelined(batches, work, store, fallback, workers=2, deadline=None, tag='pipe', log=print):
    batches = list(batches)
    t0 = time.time()
    workers = min(workers, int(os.environ.get('NTL_PIPE_WORKERS', workers)))
    if workers < 2 or len(batches) < 6 or not _agree(batches, work, workers, tag, log):
        workers = 1
    queue = deque(batches)
    redo = []
    lock = threading.Lock()
    stop = threading.Event()

    def loop():
        while not stop.is_set():
            if deadline is not None and time.time() >= deadline:
                return
            with lock:
                if not queue:
                    return
                b = queue.popleft()
            try:
                r = work(b)
            except Exception:
                with lock:
                    redo.append(b)
                stop.set()
                return
            with lock:
                if r is None:
                    redo.append(b)
                else:
                    store(b, r)

    if workers == 1:
        while queue and not (deadline is not None and time.time() >= deadline):
            b = queue.popleft()
            r = work(b)
            if r is None:
                redo.append(b)
            else:
                store(b, r)
    else:
        threads = [threading.Thread(target=loop) for _ in range(workers)]
        [t.start() for t in threads]
        [t.join() for t in threads]
    rest = redo + list(queue)
    if rest:
        gc.collect()
        try:
            import torch
            torch.cuda.empty_cache()
        except Exception:
            pass
    for b in rest:
        if deadline is not None and time.time() >= deadline:
            break
        fallback(b)
    log(f'{tag}: {len(batches)} batches, {workers} worker(s), {len(redo)} redone sequentially, {time.time() - t0:.0f}s')


def spans_devices(model):
    return len({p.device for p in model.parameters()}) > 1


## 09. Gaussian Readouts — Calibrating Hidden Representations

Transformer activations from selected depths enter class-conditional readouts. Gaussian density scores provide a second view of the same passage, complementing learned linear and twin-head projections at ambiguous class boundaries.


In [ ]:
%%writefile gauss.py
import itertools
import numpy as np
K = 18
DEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,
           "iters": 5, "repel": True, "sp_scale": 1.0}

def _lse(A, axis=None, keepdims=False):
    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
    if keepdims: return out
    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

def log_softmax(L): return L - _lse(L, axis=1, keepdims=True)

def _lw_shrinkage(X):
    from sklearn.covariance import ledoit_wolf_shrinkage
    return float(ledoit_wolf_shrinkage(X, assume_centered=True))

def _shrunk(S, a):
    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)

def _two_way(Z, y, pinv, G, iters=4):
    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)
    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))
    for _ in range(iters):
        R = Z - P[pinv]
        for c in cls: mu[c] = R[y == c].mean(0)
        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]
    return mu, P, cnt_p

def fit(X, y, prompts, cfg=None):
    c = dict(DEFAULT); c.update(cfg or {})
    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape
    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)
    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx
    if c["red"] == "none":
        V = np.eye(D)
    else:
        if c["red"] == "wpca":
            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]
        elif c["red"] == "pca": R = Xs
        else: raise ValueError(c["red"])
        d = int(min(c["d"], N - G - 1, D))
        if N < D:
            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]
            V = (R.T @ U) / np.sqrt(w)
        else:
            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T
    Z = Xs @ V; d = Z.shape[1]
    mu, P, ng = _two_way(Z, y, pinv, G)
    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = E.T @ E / dof
    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)
    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]
    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)
    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li
    W = V @ A.T
    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,
            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}

def transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]

_MASK = {}
def _inj_mask(n, C):
    if (n, C) not in _MASK:
        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)
        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]
        _MASK[(n, C)] = bad
    return _MASK[(n, C)]

def singleton_ll(U, m, lam):
    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

def joint_terms(U, m, lam):
    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]
    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)
    return a, B

def joint_table(a, B):
    n, C = a.shape; Q = np.zeros([C] * n)
    for i in range(n):
        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)
    for i, j in itertools.combinations(range(n), 2):
        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)
    return Q

def exact_marginals(Q, T=1.0, inj=True):
    n = Q.ndim; C = Q.shape[0]; Qt = Q / T
    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)
    lz = _lse(Qt); out = np.empty((n, C))
    for i in range(n):
        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz
    return out

def em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):
    n = len(U); v = lam / (1 + (n - 1) * lam)
    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
    q = np.exp(log_softmax(L))
    for _ in range(iters):
        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v
        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
        if repel:
            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)
        q = np.exp(log_softmax(L))
    return log_softmax(L)

LAM_DIFF = 1e6
def lam_g(prm, cfg=None):
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})
    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]

def group_logpost(U, prm, T1=None, Tg=None, cfg=None):
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)
    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg
    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)
    if n <= c["n_exact"]:
        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])
    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])

def scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))
    sizes = np.bincount(lab); s1 = sizes[lab] == 1
    T1 = prm["T1"] if T1 is None else T1
    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)
    for g in np.where(sizes >= 2)[0]:
        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)
    return out


## 10. Reading Between the Layers — ATLAS Inference

A single final hidden state does not capture every stylistic feature. The ATLAS stage extracts multiple transformer depths — **2, 4, 6, 8, 12, 16, 20 and 24** — and evaluates their trained heads. Several backbone families contribute complementary representations.


In [ ]:
%%writefile atlas_infer.py
import gc
import json
import os
import re
import threading
import time

import numpy as np
import torch
from safetensors.torch import load_file
from torch import nn
from torch.nn import functional as F

from pipe import pipelined

TEMPERATURES = (1.0, 2.0, 4.0, 8.0)
HEAD_CHUNK_TOKENS = 64


class Route(threading.local):
    labels = None


class SourceLinear(nn.Module):
    def __init__(self, base, sources, rank, alpha, route):
        super().__init__()
        self.base = base
        self.route = route
        self.scaling = alpha / rank
        dev = base.weight.device
        self.a = nn.Parameter(torch.zeros(rank, base.in_features, device=dev, dtype=torch.float32), requires_grad=False)
        self.b = nn.Parameter(torch.zeros(sources, base.out_features, rank, device=dev, dtype=torch.float32), requires_grad=False)

    def forward(self, x):
        labels = self.route.labels
        if labels is None or labels.numel() != x.shape[0]:
            raise ValueError('source label missing for a batch row')
        out = self.base(x)
        delta = torch.bmm(F.linear(x.float(), self.a), self.b.index_select(0, labels.to(self.b.device)).transpose(1, 2))
        return out + (delta * self.scaling).to(out.dtype)


def score_columns(logp, mask):
    lp = logp.float()
    valid = mask.float().unsqueeze(1)
    mean_lp = (lp * valid).sum(-1) / valid.sum(-1).clamp_min(1)
    gap = (lp.unsqueeze(1) - lp.unsqueeze(2)).clamp_min(0).sum(2)
    cols = [mean_lp] + [(torch.exp(-gap / t) * valid).sum(-1) / valid.sum(-1).clamp_min(1) for t in TEMPERATURES]
    return torch.stack(cols, -1)


class AtlasModel(nn.Module):
    def __init__(self, base, labels, rank, alpha):
        super().__init__()
        self.base = base
        self.labels = list(labels)
        self.route = Route()
        self.base.requires_grad_(False)
        self.base.config.use_cache = False
        hit = [(n, m) for n, m in self.base.named_modules() if n.split('.')[-1] in ('q_proj', 'k_proj', 'v_proj', 'o_proj') and isinstance(m, nn.Linear)]
        if not hit:
            raise ValueError('no attention projections found')
        for name, m in hit:
            parent, leaf = name.rsplit('.', 1)
            setattr(self.base.get_submodule(parent), leaf, SourceLinear(m, len(labels), rank, alpha, self.route))
        emb = self.base.get_input_embeddings().weight
        self.control = nn.Parameter(torch.zeros(len(labels), 1, emb.shape[1], device=emb.device, dtype=torch.float32), requires_grad=False)

    @property
    def device(self):
        return self.control.device

    def load_state(self, state):
        own = {n: p for n, p in self.named_parameters() if n.endswith(('.a', '.b')) or n == 'control'}
        if set(own) != set(state):
            raise ValueError(f'adapter key mismatch: {sorted(set(own) ^ set(state))[:6]}')
        with torch.no_grad():
            for n, p in own.items():
                t = state[n]
                if tuple(t.shape) != tuple(p.shape) or not torch.isfinite(t).all():
                    raise ValueError(f'bad adapter tensor {n}')
                p.copy_(t.to(device=p.device, dtype=p.dtype))

    def token_logp(self, ids, mask, labels):
        self.route.labels = labels
        tok = self.base.get_input_embeddings()(ids)
        ctrl = self.control.index_select(0, labels).to(tok.dtype)
        embeds = torch.cat([ctrl, tok], 1)
        attn = torch.cat([mask.new_ones(len(ids), 1), mask], 1)
        hid = self.base.model(inputs_embeds=embeds, attention_mask=attn, use_cache=False).last_hidden_state[:, 1:-1]
        tgt = ids[:, 1:]
        head = self.base.get_output_embeddings()
        parts = []
        for s in range(0, hid.shape[1], HEAD_CHUNK_TOKENS):
            h, y = hid[:, s:s + HEAD_CHUNK_TOKENS], tgt[:, s:s + HEAD_CHUNK_TOKENS]
            logits = head(h.to(head.weight.device)).float()
            parts.append(-F.cross_entropy(logits.reshape(-1, logits.shape[-1]), y.to(logits.device).reshape(-1), reduction='none').reshape(y.shape))
        return torch.cat(parts, 1).to(ids.device)

    @torch.no_grad()
    def score(self, ids, mask, chunk=3):
        outs = []
        for first in range(0, len(self.labels), chunk):
            last = min(len(self.labels), first + chunk)
            src = torch.arange(first, last, device=ids.device).repeat_interleave(len(ids))
            lp = self.token_logp(ids.repeat(last - first, 1), mask.repeat(last - first, 1), src)
            outs.append(lp.reshape(last - first, len(ids), -1).transpose(0, 1))
        cols = score_columns(torch.cat(outs, 1), mask[:, 1:])
        return cols, torch.isfinite(cols).flatten(1).all(1)


def split_device_map(base_path, n_gpu, head_weight=2.4):
    from accelerate import init_empty_weights
    from transformers import AutoConfig, AutoModelForCausalLM
    if n_gpu < 2:
        return {'': 0}
    cfg = AutoConfig.from_pretrained(base_path)
    with init_empty_weights():
        meta = AutoModelForCausalLM.from_config(cfg, torch_dtype=torch.float16)
    first = int(round((cfg.num_hidden_layers + head_weight) / 2.0))
    dm = {'model.embed_tokens': 0, 'model.rotary_emb': 0, 'model.norm': 1, 'lm_head': 1}
    for i in range(cfg.num_hidden_layers):
        dm[f'model.layers.{i}'] = 0 if i < first else 1
    tensors = [n for n, _ in meta.named_parameters()] + [n for n, _ in meta.named_buffers()]
    missing = [n for n in tensors if not any(n == k or n.startswith(k + '.') for k in dm)]
    if missing:
        raise RuntimeError(f'device map does not cover {missing[:3]}')
    return dm


class Scorer:
    def __init__(self, base_path, bundle_dir, log=print):
        self.log = log
        meta = json.load(open(os.path.join(bundle_dir, 'atlas_bundle.json'), encoding='utf-8'))
        self.labels = list(meta['labels'])
        cfg = meta['config']
        n_gpu = torch.cuda.device_count()
        t0 = time.time()
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.tok = AutoTokenizer.from_pretrained(base_path)
        self.tok.padding_side = 'right'
        if self.tok.pad_token_id is None:
            self.tok.pad_token = self.tok.eos_token
        try:
            dm = split_device_map(base_path, n_gpu)
        except Exception as e:
            self.log(f'explicit device map failed ({e!r:.100}), using device_map=auto')
            dm = 'auto'
        base = AutoModelForCausalLM.from_pretrained(base_path, dtype=torch.float16, attn_implementation='sdpa', device_map=dm)
        if any(p.device.type != 'cuda' for p in base.parameters()):
            raise RuntimeError('ATLAS base did not fit on the GPUs')
        self.model = AtlasModel(base, self.labels, cfg['rank'], cfg['alpha'])
        self.model.load_state(load_file(os.path.join(bundle_dir, 'adapters.safetensors')))
        self.model.eval()
        per_dev = {}
        for p in base.parameters():
            per_dev[str(p.device)] = per_dev.get(str(p.device), 0) + p.numel() * p.element_size()
        self.max_len = int(cfg['max_length'])
        head_path = os.path.join(bundle_dir, 'atlas_head.json')
        self.head = json.load(open(head_path)) if os.path.exists(head_path) else None
        lite_path = os.path.join(bundle_dir, 'atlas_head_meanlp.json')
        self.lite = json.load(open(lite_path)) if os.path.exists(lite_path) else None
        self.log(f'atlas loaded in {time.time() - t0:.0f}s, weights per device: ' + ', '.join(f'{k} {v / 2 ** 30:.1f} GiB' for k, v in sorted(per_dev.items())))
        self.n_gpu = n_gpu

    def encode(self, texts):
        return self.tok(list(texts), truncation=True, max_length=self.max_len)['input_ids']

    def _pack(self, enc, rows):
        pk = self.tok.pad({'input_ids': [enc[int(i)] for i in rows]}, padding=True, return_tensors='pt')
        return pk['input_ids'].to(self.model.device), pk['attention_mask'].to(self.model.device)

    def _chunks(self, enc, rows, batch, chunk, max_tokens):
        lens = np.array([len(enc[int(i)]) for i in rows])
        order = np.asarray(rows)[np.argsort(-lens, kind='stable')]
        out, a = [], 0
        while a < len(order):
            width = max(1, len(enc[int(order[a])]))
            size = max(1, min(batch, max_tokens // (chunk * width)))
            out.append(order[a:a + size])
            a += size
        return out

    def _score_rows(self, enc, rows, chunk):
        ids, mask = self._pack(enc, rows)
        p, ok = self.model.score(ids, mask, chunk)
        return p.float().cpu().numpy(), ok.cpu().numpy()

    def run(self, enc, rows, deadline, batch=4, chunk=3, max_tokens=14000, workers=2):
        out = np.zeros((len(enc), len(self.labels), 5), np.float32)
        done = np.zeros(len(enc), dtype=bool)
        chunks = self._chunks(enc, rows, batch, chunk, max_tokens)
        total = float(sum(len(enc[int(i)]) for i in rows))
        stat = {'invalid': 0, 'oom': 0, 'tokens': 0, 'rows': 0, 'next': 200, 't0': time.time()}

        def work(item):
            return self._score_rows(enc, item, chunk)

        def store(item, res):
            p, ok = res
            good = ok & np.isfinite(p).reshape(len(p), -1).all(1)
            out[item[good]] = p[good]
            done[item[good]] = True
            stat['invalid'] += int((~good).sum())
            stat['rows'] += len(item)
            stat['tokens'] += sum(len(enc[int(i)]) for i in item)
            if stat['rows'] >= stat['next']:
                stat['next'] += 200
                spent = time.time() - stat['t0']
                self.log(f"atlas {stat['rows']}/{len(rows)} rows, {spent:.0f}s elapsed, about {spent / stat['tokens'] * (total - stat['tokens']):.0f}s left")

        def fallback(item, c=chunk):
            try:
                store(item, self._score_rows(enc, item, c))
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                stat['oom'] += 1
                if len(item) > 1:
                    half = len(item) // 2
                    fallback(item[:half], c)
                    fallback(item[half:], c)
                elif c > 1:
                    fallback(item, 1)
                else:
                    stat['invalid'] += 1

        pipelined(chunks, work, store, fallback, workers if self.n_gpu >= 2 else 1, deadline, 'atlas', self.log)
        self.log(f"atlas scored {int(done.sum())}/{len(rows)} rows in {time.time() - stat['t0']:.0f}s (invalid {stat['invalid']}, oom retries {stat['oom']})")
        return out, done

    def run_candidates(self, enc, cand, deadline, max_seqs=16, max_tokens=8192, workers=1):
        n, k = cand.shape
        rows = np.repeat(np.arange(n), k)
        labels = cand.ravel()
        lens = np.array([len(enc[int(r)]) for r in rows])
        order = np.argsort(-lens, kind='stable')
        batches, a = [], 0
        while a < len(order):
            size = max(1, min(max_seqs, max_tokens // max(1, int(lens[order[a]]))))
            batches.append(order[a:a + size])
            a += size
        out = np.full(n * k, np.nan, np.float32)
        stat = {'done': 0, 'next': 2000, 't0': time.time(), 'oom': 0}

        @torch.no_grad()
        def work(item):
            pk = self.tok.pad({'input_ids': [enc[int(rows[i])] for i in item]}, padding=True, return_tensors='pt')
            ids, mask = pk['input_ids'].to(self.model.device), pk['attention_mask'].to(self.model.device)
            lp = self.model.token_logp(ids, mask, torch.as_tensor(labels[item], device=ids.device))
            valid = mask[:, 1:].float()
            return ((lp * valid).sum(1) / valid.sum(1).clamp_min(1)).float().cpu().numpy()

        def store(item, vals):
            out[item] = np.where(np.isfinite(vals), vals, np.nan)
            stat['done'] += len(item)
            if stat['done'] >= stat['next']:
                stat['next'] += 2000
                spent = time.time() - stat['t0']
                self.log(f"atlas lite {stat['done']}/{len(order)} sequences, {spent:.0f}s elapsed, about {spent / stat['done'] * (len(order) - stat['done']):.0f}s left")

        def fallback(item):
            try:
                store(item, work(item))
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                stat['oom'] += 1
                if len(item) > 1:
                    half = len(item) // 2
                    fallback(item[:half])
                    fallback(item[half:])

        pipelined(batches, work, store, fallback, workers if self.n_gpu >= 2 else 1, deadline, 'atlas lite', self.log)
        out = out.reshape(n, k)
        done = np.isfinite(out).all(1)
        self.log(f"atlas lite scored {int(done.sum())}/{n} rows ({n * k} sequences) in {time.time() - stat['t0']:.0f}s (oom retries {stat['oom']})")
        return out, done

    def candidate_logprobs(self, mean_lp, cand):
        z = self.lite['slope'] * mean_lp + np.asarray(self.lite['b'])[cand]
        z = z - z.max(1, keepdims=True)
        return z - np.log(np.exp(z).sum(1, keepdims=True))

    def calibrated(self, scores):
        feats = scores - scores.mean(1, keepdims=True)
        z = (feats / np.asarray(self.head['std'])) @ np.asarray(self.head['a']) + np.asarray(self.head['b'])
        z = z - z.max(1, keepdims=True)
        return z - np.log(np.exp(z).sum(1, keepdims=True))

    def close(self):
        del self.model
        gc.collect()
        torch.cuda.empty_cache()


## 11. Five Independent Readers — Native Qwen3 Embedding

The native Qwen3-Embedding-8B branch combines five trained fold heads. Each head transforms multidepth features into class probabilities; their calibrated outputs are averaged as a distinct expert before the final blend.

This branch has a **40% share** in the public fusion. Its fold-specific states are loaded from the mounted inference assets.


In [ ]:
%%writefile native_q8.py
import argparse
import gc
import json
import os
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd

INSTRUCTION = 'Instruct: Identify the language model that generated the text based on its writing style.\nQuery: '
DEPTHS = (3, 6, 9, 12, 18, 24, 30, 36)
SUFFIXES = ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')
RANK = 16
SCALE = 2.0
TOKEN_LIMIT = 512
TERMINAL = 151643


def log_softmax(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - z.max(axis=-1, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=-1, keepdims=True))


def calibrate(plain, centered, bias, groups, selections, use_centered=True):
    plain, centered, bias = (np.asarray(x, dtype=np.float64) for x in (plain, centered, bias))
    groups = np.asarray(groups)
    folds = plain.shape[0]
    assert plain.shape == centered.shape and plain.ndim == 3 and plain.shape[-1] == 18
    assert bias.shape == (folds, 18) and groups.shape == (plain.shape[1],) and len(selections) == folds
    assert all(np.isfinite(x).all() for x in (plain, centered, bias))
    _, inverse = np.unique(groups, return_inverse=True)
    count = np.bincount(inverse)
    eligible = (count[inverse] >= 2) & (count[inverse] <= 18)
    out = []
    for k, state in enumerate(selections):
        assert not state['outer_labels_used'] and state['centered_head_trained']
        assert state['plain_scale'] > 0 and state['centered_scale'] > 0
        means = np.zeros((len(count), 18))
        np.add.at(means, inverse, centered[k])
        means /= count[:, None]
        relative = centered[k] - means[inverse] + bias[k]
        a = log_softmax(plain[k] * state['plain_scale'])
        b = log_softmax(relative * state['centered_scale'])
        weight = float(state['selected_centered_weight'])
        assert 0 <= weight <= 0.5
        fraction = np.where(eligible & use_centered, weight, 0.0)[:, None]
        out.append(log_softmax((1 - fraction) * a + fraction * b))
    return np.stack(out)


def windows(tokenizer, texts):
    prefix = tokenizer(INSTRUCTION, add_special_tokens=False)['input_ids']
    assert tokenizer('', add_special_tokens=True)['input_ids'] == [TERMINAL], 'unexpected end-of-text handling'
    capacity = TOKEN_LIMIT - len(prefix) - 1
    first_capacity = TOKEN_LIMIT - 1
    full = tokenizer([INSTRUCTION + str(t) for t in texts], truncation=False, add_special_tokens=False)['input_ids']
    seqs, owners, weights = [], [], []
    for row, tokens in enumerate(full):
        assert len(tokens) > 0
        seqs.append(tokens[:first_capacity] + [TERMINAL])
        owners.append(row)
        weights.append(max(1, min(len(tokens), first_capacity) - len(prefix)))
        pos = first_capacity
        while pos < len(tokens):
            chunk = tokens[pos:pos + capacity]
            seqs.append(prefix + chunk + [TERMINAL])
            owners.append(row)
            weights.append(len(chunk))
            pos += len(chunk)
    return seqs, np.asarray(owners), np.asarray(weights, dtype=np.float64)


class NativeQ8:
    def __init__(self, base, assets, work, log=print, gpus=None):
        import torch
        from torch import nn
        from transformers import AutoModel, AutoTokenizer
        self.torch, self.nn, self.log = torch, nn, log
        torch.set_num_threads(2)
        for key in ('allow_tf32', 'allow_fp16_reduced_precision_reduction', 'allow_bf16_reduced_precision_reduction'):
            setattr(torch.backends.cuda.matmul, key, False)
        torch.backends.cudnn.allow_tf32 = False
        self.assets = Path(assets)
        self.labels = json.loads((self.assets / 'manifest.json').read_text())['classes']
        origin = Path(base)
        index = json.loads((self.assets / 'base' / 'model.safetensors.index.json').read_text())
        shards = sorted(set(index['weight_map'].values()))
        small = ['config.json', 'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json', 'merges.txt', 'vocab.json']
        self.staged = Path(work) / 'native-q8-base'
        self.staged.mkdir(parents=True, exist_ok=True)
        for name in small:
            src = self.assets / 'base' / name
            assert src.is_file(), str(src)
            dst = self.staged / name
            if not dst.exists():
                shutil.copyfile(src, dst)
        for name in shards:
            src, dst = origin / name, self.staged / name
            assert src.is_file(), str(src)
            if not dst.exists():
                try:
                    os.symlink(src, dst)
                except OSError:
                    shutil.copyfile(src, dst)
        for k in range(5):
            assert (self.assets / f'fold{k}' / 'trainables.safetensors').is_file(), k
        cfg = json.loads((self.staged / 'config.json').read_text())
        self.layers, self.hidden = cfg['num_hidden_layers'], cfg['hidden_size']
        assert self.layers == 36
        gpus = gpus if gpus is not None else torch.cuda.device_count()
        assert gpus >= 1
        self.head_device = 'cuda:0'
        place = {'embed_tokens': 0, 'norm': gpus - 1, 'rotary_emb': gpus - 1}
        cut = 16 if gpus >= 2 else self.layers
        place.update({f'layers.{i}': 0 if i < cut else 1 for i in range(self.layers)})
        self.tokenizer = AutoTokenizer.from_pretrained(self.staged, local_files_only=True)
        self.tokenizer.padding_side = 'right'
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        loaded = AutoModel.from_pretrained(self.staged, local_files_only=True, dtype=torch.float16, device_map=place,
                                           attn_implementation='sdpa', output_loading_info=True)
        if isinstance(loaded, tuple):
            self.model, info = loaded
            assert not any(info.get(k) for k in ('missing_keys', 'mismatched_keys', 'error_msgs', 'unexpected_keys')), info
        else:
            self.model = loaded
        self.model.config.use_cache = False
        self.loras = {}
        for name, mod in list(self.model.named_modules()):
            if isinstance(mod, nn.Linear) and name.rsplit('.', 1)[-1] in SUFFIXES:
                wrapped = self._lora(mod)
                parent, attr = name.rsplit('.', 1)
                setattr(self.model.get_submodule(parent), attr, wrapped)
                self.loras[name] = wrapped
        assert len(self.loras) == self.layers * len(SUFFIXES), len(self.loras)
        dev = self.head_device
        self.norms = nn.ModuleList([nn.LayerNorm(self.hidden, device=dev, dtype=torch.float32) for _ in DEPTHS])
        self.plain = nn.Linear(self.hidden * len(DEPTHS), 18, device=dev, dtype=torch.float32)
        self.centered = nn.Linear(self.hidden * len(DEPTHS), 18, device=dev, dtype=torch.float32)
        self.model.requires_grad_(False).eval()
        for m in (self.norms, self.plain, self.centered):
            m.requires_grad_(False).eval()
        self.expected = {}
        for name, mod in self.loras.items():
            self.expected[f'encoder.backbone.base_model.model.{name}.lora_A.default.weight'] = mod.a.weight
            self.expected[f'encoder.backbone.base_model.model.{name}.lora_B.default.weight'] = mod.b.weight
        for prefix, m in (('norms', self.norms), ('plain', self.plain), ('centered', self.centered)):
            self.expected.update({f'{prefix}.{n}': p for n, p in m.named_parameters()})
        assert len(self.expected) == 2 * len(self.loras) + 2 * len(DEPTHS) + 4
        self.current = None

    def _lora(self, base):
        torch, nn = self.torch, self.nn

        class TaskLoRA(nn.Module):
            def __init__(self, layer):
                super().__init__()
                self.base_layer = layer
                dev = layer.weight.device
                self.a = nn.Linear(layer.in_features, RANK, bias=False, device=dev, dtype=torch.float32)
                self.b = nn.Linear(RANK, layer.out_features, bias=False, device=dev, dtype=torch.float32)

            def forward(self, x):
                result = self.base_layer(x)
                return (result + self.b(self.a(x.float())) * SCALE).to(result.dtype)

        return TaskLoRA(base)

    def load_fold(self, fold):
        if self.current == fold:
            return
        from safetensors.torch import load_file
        torch = self.torch
        state = load_file(str(self.assets / f'fold{fold}' / 'trainables.safetensors'))
        assert set(state) == set(self.expected), (sorted(set(state) - set(self.expected))[:3], sorted(set(self.expected) - set(state))[:3])
        with torch.inference_mode():
            for name, param in self.expected.items():
                assert state[name].shape == param.shape and state[name].dtype == torch.float32, name
                assert torch.isfinite(state[name]).all(), name
                param.copy_(state[name].to(param.device))
        del state
        gc.collect()
        self.current = fold

    def _features(self, seqs, idx):
        torch = self.torch
        batch = self.tokenizer.pad({'input_ids': [seqs[i] for i in idx]}, padding=True, return_tensors='pt').to('cuda:0')
        out = self.model(**batch, output_hidden_states=True, use_cache=False, return_dict=True)
        assert len(out.hidden_states) == self.layers + 1
        pos = batch['attention_mask'].sum(1) - 1
        rows = torch.arange(len(idx))
        parts = []
        for depth, norm in zip(DEPTHS, self.norms):
            hs = out.last_hidden_state if depth == self.layers else out.hidden_states[depth]
            pooled = hs[rows.to(hs.device), pos.to(hs.device)].float()
            if depth == self.layers:
                pooled = self.nn.functional.normalize(pooled, dim=-1)
            parts.append(norm(pooled.to(self.head_device)))
        return torch.cat(parts, dim=-1)

    def predict(self, texts, folds=range(5), batch_size=4, deadline=None, on_fold=None):
        torch = self.torch
        seqs, owners, weights = windows(self.tokenizer, texts)
        order = sorted(range(len(seqs)), key=lambda i: (len(seqs[i]), seqs[i]))
        n = len(texts)
        res = {'plain': [], 'centered': [], 'centered_bias': [], 'folds': []}
        finite = np.ones(n, dtype=bool)
        for fold in folds:
            if deadline is not None and time.time() > deadline:
                self.log(f'native: deadline reached before fold {fold}')
                break
            self.load_fold(fold)
            t0 = time.time()
            vals = {'plain': np.zeros((n, 18)), 'centered': np.zeros((n, 18))}
            with torch.inference_mode():
                for s in range(0, len(order), batch_size):
                    idx = order[s:s + batch_size]
                    try:
                        feats = self._features(seqs, idx)
                    except torch.cuda.OutOfMemoryError:
                        torch.cuda.empty_cache()
                        feats = torch.cat([self._features(seqs, [i]) for i in idx], 0)
                    ok = torch.isfinite(feats).all(1).cpu().numpy()
                    if not ok.all():
                        for i, good in zip(idx, ok):
                            if not good:
                                finite[owners[i]] = False
                        feats = torch.nan_to_num(feats)
                    own = owners[idx]
                    for name, head in (('plain', self.plain), ('centered', self.centered)):
                        sc = head(feats).float().cpu().numpy().astype(np.float64)
                        np.add.at(vals[name], own, sc * weights[idx][:, None])
                    del feats
            denom = np.bincount(owners, weights=weights, minlength=n)[:, None]
            for name in ('plain', 'centered'):
                res[name].append(vals[name] / denom)
            res['centered_bias'].append(self.centered.bias.detach().cpu().numpy().astype(np.float64).copy())
            res['folds'].append(int(fold))
            self.log(f'native: fold {fold} done, {n} rows, {len(seqs)} windows, {time.time() - t0:.0f}s, non-finite rows {int((~finite).sum())}')
            if on_fold is not None:
                on_fold({k: np.stack(v) if k != 'folds' else np.asarray(v) for k, v in res.items()}, finite)
        return {k: np.stack(v) if k != 'folds' else np.asarray(v) for k, v in res.items()}, finite


def save(path, ids, labels, res, finite):
    tmp = str(path) + '.tmp.npz'
    np.savez(tmp, ids=np.asarray(ids, dtype=str), labels=np.asarray(labels), finite=finite, **res)
    os.replace(tmp, path)


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--base', required=True)
    p.add_argument('--assets', required=True)
    p.add_argument('--query', required=True)
    p.add_argument('--out', required=True)
    p.add_argument('--work', required=True)
    p.add_argument('--batch', type=int, default=4)
    p.add_argument('--deadline', type=float, default=0.0)
    p.add_argument('--limit', type=int, default=0)
    a = p.parse_args()
    T0 = time.time()

    def log(*x):
        print(f'[{time.time() - T0:7.0f}s]', *x, flush=True)

    frame = pd.read_csv(a.query, dtype={'id': str})
    assert frame.id.is_unique and len(frame)
    if a.limit:
        frame = frame.iloc[:a.limit]
    texts = frame.text.fillna('').astype(str).tolist()
    model = NativeQ8(a.base, a.assets, a.work, log=log)
    log('native: backbone and heads built')
    res, finite = model.predict(texts, batch_size=a.batch, deadline=a.deadline or None,
                                on_fold=lambda r, f: save(a.out, frame.id.values, model.labels, r, f))
    assert len(res['folds']), 'no fold finished'
    save(a.out, frame.id.values, model.labels, res, finite)
    log('native: saved', a.out, 'folds', res['folds'].tolist())


if __name__ == '__main__':
    main()


## 12. Sparse Evidence — Prefit Text Readouts

Even large representations can overlook simple textual habits. The prefit core provides LinearSVC scores, TF-IDF n-grams and distortion-sensitive features. These scores capture recurring local patterns that differ from the transformer-based evidence.


In [ ]:
%%writefile ntl660_readout.py
import json
import re
from pathlib import Path

import joblib
import numpy as np
from scipy.sparse import csr_matrix, diags, hstack

LABELS = sorted(['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
                 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
                 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
                 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'])
FUNCTION_WORDS = frozenset('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
VIEWS = [('char', 3, 5, 250000, 1.), ('word', 1, 3, 180000, .7),
         ('shape', 2, 6, 60000, .5), ('function', 1, 3, 50000, .5)]


def shape_view(text):
    return re.sub(r'[^\W\d_]+', lambda m: 'X' if m[0].isupper() else 'x', text)


def function_view(text):
    return ' '.join(t.lower() if t.lower() in FUNCTION_WORDS or not t.isalnum() else 'CONTENT'
                    for t in re.findall(r'\w+|[^\w\s]', text))


def prepare(texts, view, keep=()):
    if view == 'shape':
        return [shape_view(t) for t in texts]
    if view == 'function':
        return [function_view(t) for t in texts]
    if view == 'distort':
        keep = set(keep)
        return [re.sub(r"[A-Za-z’']+", lambda m: m[0] if m[0].lower() in keep else '*', t) for t in texts]
    return texts


def log_softmax(z):
    z = z - z.max(1, keepdims=True)
    return z - np.log(np.exp(z).sum(1, keepdims=True))


def softmax(z):
    z = z - z.max(1, keepdims=True)
    z = np.exp(z)
    return z / z.sum(1, keepdims=True)


def sparse_group_centre(X, groups, lo=2, hi=18):
    X = csr_matrix(X)
    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)
    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))
    if not len(keep):
        return X.copy()
    col = np.full(len(cnt), -1)
    col[keep] = np.arange(len(keep))
    rows = np.flatnonzero(col[inv] >= 0)
    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))
    means = diags(1.0 / cnt[keep]) @ (M.T @ X)
    return (X - M @ means).tocsr()


def style_groups(texts):
    from scipy.cluster.hierarchy import fcluster, linkage
    from scipy.spatial.distance import squareform
    from sklearn.feature_extraction.text import TfidfVectorizer
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    try:
        X = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=100000,
                            dtype=np.float32).fit_transform(texts)
        D = np.clip(1.0 - (X @ X.T).toarray(), 0., 1.)
    except ValueError as exc:
        if 'empty vocabulary' not in str(exc):
            raise
        D = np.ones((len(texts), len(texts)), dtype=np.float32) - np.eye(len(texts), dtype=np.float32)
    med = max(float(np.median(D[np.triu_indices(len(texts), 1)])), 1e-6)
    D = np.asarray(D / med, dtype=np.float64).copy()
    np.fill_diagonal(D, 0.)
    return fcluster(linkage(squareform(D, checks=False), method='average'), .90, criterion='distance')


class Readout:
    def __init__(self, folder):
        self.folder = Path(folder)
        self.meta = json.loads((self.folder / 'metadata.json').read_text())
        if self.meta['labels'] != LABELS:
            raise ValueError('Unexpected canonical class order')
        self.vectorizers = [joblib.load(self.folder / e['file']) for e in self.meta['views']]
        from sklearn.feature_extraction.text import TfidfVectorizer
        assert all(type(v) is TfidfVectorizer and not callable(v.analyzer) and
                   v.preprocessor is None and v.tokenizer is None for v in self.vectorizers)

    def matrix(self, texts):
        return hstack([v.transform(prepare(texts, e['view'], self.meta.get('keep', []))) * e['weight']
                       for v, e in zip(self.vectorizers, self.meta['views'])], format='csr')

    def raw(self, X, state='model'):
        with np.load(self.folder / (state + '.npz'), allow_pickle=False) as z:
            scores = np.asarray(X @ z['coef'].T) + z['intercept']
            classes = z['classes'].copy()
        if len(classes) == 2 and scores.shape[1] == 1:
            scores = np.column_stack([-scores[:, 0], scores[:, 0]])
        return scores, classes

    def scores(self, texts, state='model'):
        return self.raw(self.matrix(texts), state)


def core_votes(root, texts):
    votes = {}
    for key, weight in [('svc4', 12.0), ('tfidf', .25), ('distort', .10)]:
        r = Readout(Path(root) / 'core' / key)
        scores, classes = r.scores(texts)
        assert classes.tolist() == list(range(len(LABELS)))
        votes[key] = log_softmax(scores) * weight
    return votes


def complementary_style(root, texts):
    root = Path(root)
    r = Readout(root / 'comp' / 'stylegram')
    X = r.matrix(texts)
    d, classes = r.raw(X, 'plain')
    assert classes.tolist() == list(range(len(LABELS)))
    P = softmax(d)
    groups = style_groups(texts)
    sizes = np.bincount(groups)[groups]
    rows = (sizes >= 2) & (sizes <= 18)
    if rows.any():
        d, _ = r.raw(sparse_group_centre(X[rows], groups[rows]), 'twin')
        P[rows] = softmax(d)
    gates = {}
    for family in ['claude', 'gpt', 'gemini', 'glm']:
        g = Readout(root / 'comp' / 'gates' / family)
        d, classes = g.scores(texts)
        expected = [i for i, label in enumerate(LABELS) if label.startswith(family)]
        assert classes.tolist() == expected
        gates[family] = d
    return P, gates, groups


## 13. The Long-Context Witness — Qwen3.8-27B GSQ

A specialized Qwen3.8-27B branch examines passages with its own quantized inference engine and fitted classifier. It targets difficult boundaries where smaller readers disagree.

The public configuration gives the GSQ branch **3% of the parent mixture**. Its contribution should be interpreted alongside the much larger native embedding branch.


In [ ]:
%%writefile gsq_infer_model.py
import json
import math
import struct
import time
from pathlib import Path
import numpy as np
import torch
from torch import nn
from accelerate.utils import set_module_tensor_to_device
from transformers import AutoTokenizer
from transformers.models.qwen3_5.configuration_qwen3_5 import Qwen3_5TextConfig
from transformers.models.qwen3_5 import modeling_qwen3_5 as qm
from gsq_infer_layers import GSQLinear,DenseFrozenLinear
from delta_turing import triangular_chunk_gated_delta_rule

REVISION="b5ce0b76f60020a875dee4f6ec9d934cca4121e4"


class TensorReader:
    def __init__(self,base):
        self.base=Path(base)
        self.index=json.loads((self.base/"model.safetensors.index.json").read_text())["weight_map"]
        self.headers,self.maps={},{}
        for shard in sorted(set(self.index.values())):
            path=self.base/shard
            assert path.is_file(),f"Missing exact GSQ shard: {path}"
            with path.open("rb") as stream:
                size=struct.unpack("<Q",stream.read(8))[0];assert size<2**21
                self.headers[shard]=(8+size,json.loads(stream.read(size)))
            self.maps[shard]=np.memmap(path,dtype=np.uint8,mode="r")
        missing=[key for key,shard in self.index.items() if key not in self.headers[shard][1]]
        assert all(key.startswith("mtp.") for key in missing),missing
        self.stale_mtp_keys=missing

    def has(self,key):
        return key in self.index and key in self.headers[self.index[key]][1]

    def array(self,key):
        assert self.has(key),f"Required actual GSQ tensor is missing: {key}"
        shard=self.index[key];offset,header=self.headers[shard];item=header[key]
        lo,hi=item["data_offsets"]
        dtype={"I32":"<i4","I64":"<i8","BF16":"<u2","F16":"<f2","F32":"<f4"}[item["dtype"]]
        value=self.maps[shard][offset+lo:offset+hi].view(dtype).reshape(item["shape"])
        return value,item["dtype"]

    def tensor(self,key,device="cuda",bf16_dtype=None):
        value,dtype=self.array(key)
        if dtype=="BF16":
            assert bf16_dtype in (torch.float16,torch.float32)
            fp32=(value.astype(np.uint32)<<16).view(np.float32)
            result=torch.from_numpy(fp32).to(bf16_dtype)
        else:
            result=torch.from_numpy(value.copy())
        return result.to(device)


class QuantizedCPUEmbedding(nn.Module):
    def __init__(self,reader):
        super().__init__()
        prefix="model.language_model.embed_tokens."
        self.packed,dtype=reader.array(prefix+"weight_packed");assert dtype=="I32"
        self.scale_bits,dtype=reader.array(prefix+"weight_scale");assert dtype=="BF16"
        assert self.packed.shape==(248320,640) and self.scale_bits.shape==(248320,80)
        self.cpu_packed_bytes=self.packed.nbytes+self.scale_bits.nbytes
        self.last_lookup={}

    @torch.no_grad()
    def forward(self,ids):
        began=time.perf_counter()
        assert ids.device.type=="cpu"
        flat=ids.numpy().reshape(-1)
        unique,inverse=np.unique(flat,return_inverse=True)
        words=self.packed[unique].astype(np.uint32,copy=False)
        shifts=np.arange(8,dtype=np.uint32)*4
        q=((words[:,:,None]>>shifts)&15).reshape(len(unique),5120).astype(np.int16)-8
        scales=(self.scale_bits[unique].astype(np.uint32)<<16).view(np.float32)
        values=(q.reshape(len(unique),80,64).astype(np.float32)*scales[:,:,None]).reshape(len(unique),5120)
        decoded=torch.from_numpy(values).to(torch.float16)
        selected=decoded.index_select(0,torch.from_numpy(inverse)).reshape(*ids.shape,5120)
        result=selected.to("cuda",non_blocking=False)
        self.last_lookup=dict(unique_rows=len(unique),requested_rows=len(flat),
            transferred_bytes=selected.numel()*selected.element_size(),seconds=time.perf_counter()-began)
        return result


class GSQClassifier(nn.Module):
    def __init__(self,base,*,pad_multiple=8):
        super().__init__();began=time.perf_counter()
        assert torch.cuda.device_count()==1
        self.pad_multiple=pad_multiple
        self.depths=[8,16,24,32,40,48,56,64]
        config=json.loads((Path(base)/"config.json").read_text())
        quant=config["quantization_config"]
        assert quant["quant_method"]=="compressed-tensors" and quant["format"]=="pack-quantized"
        assert quant["config_groups"]["group_0"]["weights"]["num_bits"]==3
        assert quant["config_groups"]["group_0"]["weights"]["group_size"]==128
        cfg=Qwen3_5TextConfig(**config["text_config"])
        assert cfg.num_hidden_layers==64 and cfg.hidden_size==5120
        cfg._attn_implementation="sdpa";cfg.use_cache=False
        self.cfg=cfg
        delta=triangular_chunk_gated_delta_rule
        qm.FusedRMSNormGated=None
        with torch.device("meta"):self.model=qm.Qwen3_5TextModel(cfg)
        self.model.norm=nn.Identity()
        self.reader=TensorReader(base)
        prefix="model.language_model."
        packed_count=dense_count=packed_gpu_bytes=0
        for name,module in list(self.model.named_modules()):
            if not isinstance(module,nn.Linear):continue
            assert name.startswith("layers.") and module.bias is None
            layer=int(name.split(".")[1]);rank=16 if layer>=56 else 0
            key=prefix+name+"."
            if self.reader.has(key+"weight_packed"):
                shape,_=self.reader.array(key+"weight_shape")
                assert tuple(shape)==(module.out_features,module.in_features)
                packed=self.reader.tensor(key+"weight_packed")
                scales=self.reader.tensor(key+"weight_scale",bf16_dtype=torch.float32)
                replacement=GSQLinear(packed,scales,module.in_features,module.out_features,rank)
                packed_count+=1;packed_gpu_bytes+=packed.numel()*packed.element_size()+scales.numel()*scales.element_size()
            else:
                assert ".linear_attn.in_proj_a" in name or ".linear_attn.in_proj_b" in name,name
                replacement=DenseFrozenLinear(self.reader.tensor(key+"weight",bf16_dtype=torch.float16),module.in_features,module.out_features,rank)
                dense_count+=1
            parent,_,attribute=name.rpartition(".");setattr(self.model.get_submodule(parent),attribute,replacement)
        self.model.embed_tokens=QuantizedCPUEmbedding(self.reader)
        self.model.rotary_emb=qm.Qwen3_5TextRotaryEmbedding(cfg,device="cuda")
        for name,parameter in list(self.model.named_parameters()):
            if ".lora_A" in name or ".lora_B" in name:continue
            value=self.reader.tensor(prefix+name,bf16_dtype=torch.float16)
            set_module_tensor_to_device(self.model,name,"cuda",value=value,dtype=value.dtype)
        for name,parameter in self.model.named_parameters():parameter.requires_grad_(".lora_A" in name or ".lora_B" in name)
        assert not any(value.is_meta or value.device.type!="cuda" for value in self.model.parameters())
        assert not any(value.is_meta for value in self.model.buffers())
        for layer in self.model.layers:
            if hasattr(layer,"linear_attn"):
                layer.linear_attn.chunk_gated_delta_rule=delta
                layer.linear_attn.causal_conv1d_fn=None
        assert all(not p.requires_grad for layer in self.model.layers[:56] for p in layer.parameters())
        self.norms=nn.ModuleList([nn.LayerNorm(5120) for _ in self.depths]).to("cuda")
        self.plain=nn.Linear(40960,18).to("cuda");self.centered=nn.Linear(40960,18).to("cuda")
        self.tokenizer=AutoTokenizer.from_pretrained(base,local_files_only=True)
        self.tokenizer.pad_token_id=248044;self.tokenizer.padding_side="right"
        self.load_report=dict(revision=REVISION,seconds=time.perf_counter()-began,packed_linears=packed_count,
            ignored_original_BF16_linears=dense_count,packed_gpu_bytes=packed_gpu_bytes,
            CPU_packed_embedding_bytes=self.model.embed_tokens.cpu_packed_bytes,
            trainable_parameters=sum(p.numel() for p in self.parameters() if p.requires_grad),
            delta_backend=delta.__module__,causal_convolution="native PyTorch",pad_multiple=pad_multiple,
            all_decoder_layers=64,LoRA_target_layers=list(range(56,64)),
            dtype="FP16 compute/FP32 GSQ scales+LoRA+norms+heads; BF16 decoded on CPU",
            stale_unused_MTP_index_entries=len(self.reader.stale_mtp_keys))

    def forward(self,token_rows):
        assert all(ids and ids[-1]==248044 for ids in token_rows)
        batch=self.tokenizer.pad({"input_ids":token_rows},padding=True,pad_to_multiple_of=self.pad_multiple,return_tensors="pt")
        assert batch.attention_mask.sum(1).tolist()==[len(ids) for ids in token_rows]
        assert batch.input_ids.shape[1]<=self.cfg.max_position_embeddings
        raw=[]
        with torch.no_grad():
            hidden=self.model.embed_tokens(batch.input_ids)
            attention=batch.attention_mask.to("cuda")
            pos=torch.arange(hidden.shape[1],device="cuda")[None].expand(hidden.shape[0],-1)
            kwargs=dict(config=self.cfg,inputs_embeds=hidden,attention_mask=attention,past_key_values=None,position_ids=pos)
            masks={"full_attention":qm.create_causal_mask(**kwargs),"linear_attention":qm.create_recurrent_attention_mask(**kwargs)}
            rope=self.model.rotary_emb(hidden,pos[None].expand(3,-1,-1))
            mask=attention.unsqueeze(-1).float();denominator=mask.sum(1)
            for index in range(56):
                with torch.autocast("cuda",dtype=torch.float16):
                    hidden=self.model.layers[index](hidden,position_embeddings=rope,
                        attention_mask=masks[self.cfg.layer_types[index]],position_ids=pos,past_key_values=None,use_cache=False)
                if index+1 in self.depths:raw.append((hidden.float()*mask).sum(1)/denominator)
        assert len(raw)==7 and not hidden.requires_grad
        for index in range(56,64):
            layer=self.model.layers[index]
            def apply(value,block=layer,rotary=rope,position=pos,attention_mask=masks[self.cfg.layer_types[index]]):
                with torch.autocast("cuda",dtype=torch.float16):
                    return block(value,position_embeddings=rotary,attention_mask=attention_mask,
                                 position_ids=position,past_key_values=None,use_cache=False)
            hidden=apply(hidden)
        raw.append((hidden.float()*mask).sum(1)/denominator)
        features=torch.cat([norm(value) for norm,value in zip(self.norms,raw)],-1)
        assert features.shape==(len(token_rows),40960) and torch.isfinite(features).all()
        return features


In [ ]:
%%writefile gsq_infer_layers.py
import math
import torch
import triton
import triton.language as tl
from torch import nn
from torch.nn import functional as F


@triton.jit
def unpack3_kernel(P,S,W,N:tl.constexpr,K:tl.constexpr,PACKED_COLS:tl.constexpr,GROUPS:tl.constexpr,BLOCK:tl.constexpr):
    element=tl.program_id(0)*BLOCK+tl.arange(0,BLOCK)
    valid=element<N*K
    row=element//K;col=element%K
    bit=col*3;word=bit//32;shift=bit%32
    low=tl.load(P+row*PACKED_COLS+word,mask=valid,other=0).to(tl.uint32)
    crosses=shift>29
    high=tl.load(P+row*PACKED_COLS+word+1,mask=valid&crosses,other=0).to(tl.uint32)
    unsigned=((low>>shift)|(high<<((32-shift)&31)))&7
    signed=unsigned.to(tl.int32)-4
    scale=tl.load(S+row*GROUPS+col//128,mask=valid,other=0).to(tl.float32)
    tl.store(W+element,signed.to(tl.float32)*scale,mask=valid)


def dequantize(packed,scales):
    n,groups=scales.shape;k=groups*128
    output=torch.empty((n,k),device=packed.device,dtype=torch.float16)
    unpack3_kernel[(triton.cdiv(n*k,512),)](packed,scales,output,n,k,packed.shape[1],groups,512)
    return output


class GSQLinear(nn.Module):
    def __init__(self,packed,scales,in_features,out_features,rank):
        super().__init__()
        assert packed.dtype==torch.int32 and scales.dtype==torch.float32
        assert tuple(packed.shape)==(out_features,in_features*3//32)
        assert tuple(scales.shape)==(out_features,in_features//128)
        self.register_buffer("packed",packed,persistent=False)
        self.register_buffer("scales",scales,persistent=False)
        self.in_features,self.out_features=in_features,out_features
        self.bias=None
        initialize_lora(self,rank)
    def forward(self,value):
        base=F.linear(value,dequantize(self.packed,self.scales))
        return add_lora(self,value,base)


class DenseFrozenLinear(nn.Module):
    def __init__(self,weight,in_features,out_features,rank):
        super().__init__()
        assert tuple(weight.shape)==(out_features,in_features)
        self.register_buffer("base_weight",weight,persistent=False)
        self.in_features,self.out_features=in_features,out_features
        self.bias=None
        initialize_lora(self,rank)
    def forward(self,value):
        return add_lora(self,value,F.linear(value,self.base_weight))


def initialize_lora(module,rank):
    module.scale=2.
    module.dropout=nn.Identity()
    if rank:
        module.lora_A=nn.Parameter(torch.empty(rank,module.in_features,device="cuda",dtype=torch.float32))
        module.lora_B=nn.Parameter(torch.zeros(module.out_features,rank,device="cuda",dtype=torch.float32))
        nn.init.kaiming_uniform_(module.lora_A,a=math.sqrt(5))
    else:
        module.register_parameter("lora_A",None);module.register_parameter("lora_B",None)


def add_lora(module,value,base):
    if module.lora_A is None:return base
    with torch.autocast("cuda",dtype=torch.float16):
        extra=F.linear(F.linear(module.dropout(value),module.lora_A),module.lora_B)
    return base+module.scale*extra.to(base.dtype)


In [ ]:
%%writefile delta_turing.py
import torch
from torch.nn import functional as F
from transformers.models.qwen3_5.modeling_qwen3_5 import l2norm

def triangular_chunk_gated_delta_rule(
    query,
    key,
    value,
    g,
    beta,
    chunk_size=64,
    initial_state=None,
    output_final_state=False,
    use_qk_l2norm_in_kernel=False,
    **kwargs,
):
    initial_dtype = query.dtype
    if use_qk_l2norm_in_kernel:
        query = l2norm(query, dim=-1, eps=1e-6)
        key = l2norm(key, dim=-1, eps=1e-6)
    query, key, value, beta, g = [
        x.transpose(1, 2).contiguous().to(torch.float32) for x in (query, key, value, beta, g)
    ]

    batch_size, num_heads, sequence_length, k_head_dim = key.shape
    v_head_dim = value.shape[-1]
    pad_size = (chunk_size - sequence_length % chunk_size) % chunk_size
    query = F.pad(query, (0, 0, 0, pad_size))
    key = F.pad(key, (0, 0, 0, pad_size))
    value = F.pad(value, (0, 0, 0, pad_size))
    beta = F.pad(beta, (0, pad_size))
    g = F.pad(g, (0, pad_size))
    total_sequence_length = sequence_length + pad_size
    scale = 1 / (query.shape[-1] ** 0.5)
    query = query * scale

    v_beta = value * beta.unsqueeze(-1)
    k_beta = key * beta.unsqueeze(-1)
    query, key, value, k_beta, v_beta = [
        x.reshape(x.shape[0], x.shape[1], -1, chunk_size, x.shape[-1]) for x in (query, key, value, k_beta, v_beta)
    ]
    g = g.reshape(g.shape[0], g.shape[1], -1, chunk_size)
    mask = torch.triu(torch.ones(chunk_size, chunk_size, dtype=torch.bool, device=query.device), diagonal=0)

    g = g.cumsum(dim=-1)
    decay_mask = ((g.unsqueeze(-1) - g.unsqueeze(-2)).tril().exp().float()).tril()
    attn = -((k_beta @ key.transpose(-1, -2)) * decay_mask).masked_fill(mask, 0)
    with torch.autocast(device_type=query.device.type, enabled=False):
        identity = torch.eye(chunk_size, dtype=torch.float32, device=attn.device)
        coefficient = identity - attn.float()
        identity = identity.expand_as(coefficient)
        attn = torch.linalg.solve_triangular(coefficient, identity,
                                             upper=False, unitriangular=True)
    value = attn @ v_beta
    k_cumdecay = attn @ (k_beta * g.exp().unsqueeze(-1))
    last_recurrent_state = (
        torch.zeros(batch_size, num_heads, k_head_dim, v_head_dim, dtype=value.dtype, device=value.device)
        if initial_state is None
        else initial_state.to(value)
    )
    core_attn_out = torch.zeros_like(value)
    mask = torch.triu(torch.ones(chunk_size, chunk_size, dtype=torch.bool, device=query.device), diagonal=1)

    for i in range(0, total_sequence_length // chunk_size):
        q_i, k_i, v_i = query[:, :, i], key[:, :, i], value[:, :, i]
        attn = q_i @ k_i.transpose(-1, -2) * decay_mask[:, :, i]
        v_prime = (k_cumdecay[:, :, i]) @ last_recurrent_state
        v_new = v_i - v_prime
        attn_inter = (q_i * g[:, :, i, :, None].exp()) @ last_recurrent_state
        core_attn_out[:, :, i] = attn_inter + attn @ v_new
        last_recurrent_state = (
            last_recurrent_state * g[:, :, i, -1, None, None].exp()
            + (k_i * (g[:, :, i, -1, None] - g[:, :, i]).exp()[..., None]).transpose(-1, -2) @ v_new
        )

    if not output_final_state:
        last_recurrent_state = None
    core_attn_out = core_attn_out.reshape(core_attn_out.shape[0], core_attn_out.shape[1], -1, core_attn_out.shape[-1])
    core_attn_out = core_attn_out[:, :, :sequence_length]
    core_attn_out = core_attn_out.transpose(1, 2).contiguous().to(initial_dtype)
    return core_attn_out, last_recurrent_state


In [ ]:
%%writefile adapter.py
import json
from pathlib import Path

import torch
from safetensors import safe_open

REVISION = "b5ce0b76f60020a875dee4f6ec9d934cca4121e4"


def load_final(model, final_file, metadata_file, complete_file):
    final_file, metadata_file, complete_file = map(Path, (final_file, metadata_file, complete_file))
    for path in (final_file, metadata_file, complete_file):
        if not path.is_file():
            raise FileNotFoundError(f"Required final training artifact missing: {path}")
    metadata = json.loads(metadata_file.read_text(encoding="utf-8"))
    complete = json.loads(complete_file.read_text(encoding="utf-8"))
    assert complete["status"] == "TRAINED_FULLTEXT_GSQ3_PUBLIC119899_OFFICIAL2_EPOCHS"
    assert complete["rows_seen"] == {"external": 119899, "official": 5398}
    assert complete["final_artifact"]["path"] == final_file.name == "final.safetensors"
    assert complete["contract_hash"] == metadata["checkpoint_contract_hash"]
    assert metadata["source_revision"] == REVISION
    assert metadata["rank"] == 16 and metadata["alpha"] == 32
    assert metadata["depths"] == [8, 16, 24, 32, 40, 48, 56, 64]
    assert metadata["lora_target_layer_indices"] == list(range(56, 64))
    assert metadata["executed_decoder_layers"] == 64 and metadata["text_truncation"] is False
    assert len(metadata["labels"]) == 18
    assert final_file.stat().st_size == complete["final_artifact"]["bytes"]
    expected = {name: param for name, param in model.named_parameters() if param.requires_grad}
    export_keys = {"backbone." + name.removeprefix("model.") if name.startswith("model.") else name: name
                   for name in expected}
    with safe_open(str(final_file), framework="pt", device="cpu") as weights:
        assert set(weights.keys()) == set(export_keys), "Final adapter tensor names differ from model trainable parameters"
        with torch.no_grad():
            for key, name in export_keys.items():
                source = weights.get_tensor(key)
                target = expected[name]
                assert source.dtype == target.dtype and source.shape == target.shape and torch.isfinite(source).all(), key
                target.copy_(source.to(target.device))
    model.eval()
    return metadata


In [ ]:
%%writefile gsq_infer.py
import argparse
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def read_rows(path, text_column):
    frame = pd.read_parquet(path).reset_index(drop=True)
    assert text_column in frame and frame[text_column].notna().all()
    texts = frame[text_column].astype(str).tolist()
    assert all(texts)
    return frame, texts


def worker(args):
    assert os.environ.get("CUDA_VISIBLE_DEVICES") in ("0", "1")
    import torch
    from adapter import load_final
    from gsq_infer_model import GSQClassifier

    assert torch.cuda.device_count() == 1, "GSQ worker must see one isolated GPU"
    assert torch.cuda.get_device_capability(0) >= (7, 5), "GSQ T4 path requires SM 7.5 or newer"
    torch.set_grad_enabled(False)
    frame, texts = read_rows(args.input, args.text_column)
    indices = np.arange(len(texts), dtype=np.int64)[args.worker_id::2]
    assert len(indices)
    model = GSQClassifier(args.base)
    metadata = load_final(model, args.final, args.metadata, args.complete)
    tokenizer = model.tokenizer
    features = np.empty((len(indices), 40960), dtype=np.float32)
    plain = np.empty((len(indices), 18), dtype=np.float32)
    for begin in range(0, len(indices), args.batch_size):
        batch_ids = indices[begin:begin + args.batch_size]
        token_rows = tokenizer([texts[int(i)] for i in batch_ids], add_special_tokens=False,
                               truncation=False)["input_ids"]
        token_rows = [ids + [248044] for ids in token_rows]
        assert all(len(ids) <= model.cfg.max_position_embeddings for ids in token_rows), "Full text exceeds model context"
        with torch.inference_mode():
            value = model(token_rows)
            score = model.plain(value)
        stop = begin + len(batch_ids)
        features[begin:stop] = value.float().cpu().numpy()
        plain[begin:stop] = score.float().cpu().numpy()
        del value, score
    args.output.parent.mkdir(parents=True, exist_ok=True)
    np.savez(args.output, row_index=indices, features=features, plain_logits=plain)
    print(json.dumps(dict(worker=args.worker_id, rows=len(indices), labels=metadata["labels"],
                          load_report=model.load_report)), flush=True)


def group_codes(args, frame):
    assert (args.groups is None) != (args.prompt_column is None), "Provide exactly one group source"
    if args.groups is not None:
        codes = np.load(args.groups, allow_pickle=False)
        assert codes.ndim == 1 and len(codes) == len(frame) and np.issubdtype(codes.dtype, np.integer)
        assert np.all(codes >= 0), "Explicit group codes must be nonnegative"
        return codes
    assert args.prompt_column in frame, "Requested prompt column is missing"
    assert frame[args.prompt_column].notna().all(), "Missing prompt group is ambiguous"
    return pd.factorize(frame[args.prompt_column], sort=False)[0]


def combine(args, outputs):
    from safetensors import safe_open

    frame, _ = read_rows(args.input, args.text_column)
    n = len(frame)
    features = np.empty((n, 40960), dtype=np.float32)
    plain = np.empty((n, 18), dtype=np.float32)
    seen = np.zeros(n, dtype=bool)
    for path in outputs:
        with np.load(path, allow_pickle=False) as item:
            ids = item["row_index"]
            assert ids.ndim == 1 and np.issubdtype(ids.dtype, np.integer)
            assert len(np.unique(ids)) == len(ids)
            assert np.all((0 <= ids) & (ids < n)) and not np.any(seen[ids])
            assert item["features"].shape == (len(ids), 40960)
            assert item["plain_logits"].shape == (len(ids), 18)
            features[ids] = item["features"]
            plain[ids] = item["plain_logits"]
            seen[ids] = True
    assert seen.all() and np.isfinite(features).all() and np.isfinite(plain).all()
    metadata = json.loads(args.metadata.read_text(encoding="utf-8"))
    with safe_open(str(args.final), framework="pt", device="cpu") as weights:
        kernel = weights.get_tensor("centered.weight").numpy().astype(np.float32)
        bias = weights.get_tensor("centered.bias").numpy().astype(np.float32)
    assert kernel.shape == (18, 40960) and bias.shape == (18,)
    groups = group_codes(args, frame)
    centered = plain.copy()
    valid_centered = np.zeros(n, dtype=bool)
    for group in np.unique(groups):
        ids = np.flatnonzero(groups == group)
        if len(ids) == 1:
            continue
        centered[ids] = (features[ids] - features[ids].mean(0, keepdims=True)) @ kernel.T + bias
        valid_centered[ids] = True
    assert np.isfinite(centered).all() and len(groups) == n
    args.output.parent.mkdir(parents=True, exist_ok=True)
    np.savez(args.output, row_index=np.arange(n, dtype=np.int64), features=features,
             plain_logits=plain, centered_logits=centered, valid_centered=valid_centered,
             group_code=groups, labels=np.asarray(metadata["labels"]))


def parent(args):
    _, texts = read_rows(args.input, args.text_column)
    worker_count = min(2, len(texts))
    outputs = [args.work_dir / f"gsq-worker-{i}.npz" for i in range(worker_count)]
    args.work_dir.mkdir(parents=True, exist_ok=True)
    children = []
    for device in range(worker_count):
        command = [sys.executable, str(Path(__file__).resolve()), "--worker-id", str(device),
                   "--base", str(args.base), "--final", str(args.final), "--metadata", str(args.metadata),
                   "--complete", str(args.complete), "--input", str(args.input), "--output", str(outputs[device]),
                   "--text-column", args.text_column,
                   "--batch-size", str(args.batch_size)]
        env = os.environ.copy()
        env.update(CUDA_VISIBLE_DEVICES=str(device), HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                   TOKENIZERS_PARALLELISM="false")
        children.append(subprocess.Popen(command, env=env))
    codes = [child.wait() for child in children]
    if any(code != 0 for code in codes):
        raise RuntimeError(f"GSQ workers failed: exit codes {codes}; outputs were not combined")
    combine(args, outputs)


def main():
    p = argparse.ArgumentParser(description=__doc__)
    for name in ("base", "final", "metadata", "complete", "input", "output"):
        p.add_argument("--" + name, type=Path, required=True)
    p.add_argument("--work-dir", type=Path)
    p.add_argument("--text-column", default="text")
    p.add_argument("--prompt-column")
    p.add_argument("--groups", type=Path)
    p.add_argument("--batch-size", type=int, default=1)
    p.add_argument("--worker-id", type=int, choices=(0, 1))
    args = p.parse_args()
    assert args.batch_size >= 1
    if args.worker_id is None:
        assert args.work_dir is not None and os.environ.get("CUDA_VISIBLE_DEVICES") in (None, "", "0,1")
        assert (args.groups is None) != (args.prompt_column is None), "Provide --groups or --prompt-column"
        parent(args)
    else:
        worker(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile gsq_stage.py
import json
import os
import shutil
import signal
import subprocess
import sys
import tempfile
import time
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd

BASE_FILES = {'config.json': 5038, 'merges.txt': 3353259, 'model.safetensors.index.json': 198539, 'tokenizer.json': 12809320,
              'tokenizer_config.json': 17928, 'vocab.json': 6722759, 'model-00001-of-00003.safetensors': 2272369784,
              'model-00002-of-00003.safetensors': 4777704120, 'model-00003-of-00003.safetensors': 4777704376}


@contextmanager
def staged_tree(source, files, label, temp_root='/tmp'):
    source, temp_root = Path(source), Path(temp_root)
    assert source.is_dir()
    for name, size in files.items():
        item = source / name
        assert item.is_file() and item.stat().st_size == size, (label, name, size)
    total = sum(files.values())
    free = shutil.disk_usage(temp_root).free
    assert free > total + 2 * 2 ** 30, ('not enough local disk to stage', label, total, free)
    destination = Path(tempfile.mkdtemp(prefix='ntl-stage-' + label + '-', dir=temp_root))
    start = time.monotonic()
    try:
        for name, size in files.items():
            with (source / name).open('rb') as reader, (destination / name).open('wb') as writer:
                while block := reader.read(16 * 1024 * 1024):
                    writer.write(block)
            assert (destination / name).stat().st_size == size
        print(json.dumps({'stage': label, 'staged_bytes': total, 'seconds': round(time.monotonic() - start)}), flush=True)
        yield destination
    finally:
        shutil.rmtree(destination, ignore_errors=True)


def run(assets, ids, texts, groups, work, labels, log=print, timeout=None):
    assets, work = Path(assets), Path(work)
    inference = assets / 'gsq120k-trained' / 'gsq120k-inference' / 'gsq'
    final, metadata, complete = inference / 'final.safetensors', inference / 'final.json', inference / 'COMPLETE.json'
    for path in (final, metadata, complete):
        assert path.is_file(), path
    wheels = sorted((assets / 'name-that-llm-artifacts' / 'qwen38-offline-linux').glob('*.whl'))
    assert len(wheels) >= 6, wheels
    work.mkdir(parents=True, exist_ok=True)
    target = work / 'python'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--disable-pip-version-check', '--target', str(target)] + [str(w) for w in wheels],
                   check=True, capture_output=True)
    query = work / 'query.parquet'
    pd.DataFrame({'id': list(ids), 'text': list(texts)}).to_parquet(query, index=False)
    group_file = work / 'groups.npy'
    np.save(group_file, np.asarray(groups, dtype=np.int64), allow_pickle=False)
    output = work / 'gsq.npz'
    if output.exists():
        output.unlink()
    started = time.monotonic()
    with staged_tree(assets / 'gsq3-base' / 'qwen38-27b-gsq3', BASE_FILES, 'gsq3-base') as base:
        env = dict(os.environ, PYTHONPATH=str(target), CUDA_VISIBLE_DEVICES='', HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false')
        command = [sys.executable, '-u', str(Path(__file__).resolve().parent / 'gsq_infer.py'), '--base', str(base), '--final', str(final), '--metadata', str(metadata),
                   '--complete', str(complete), '--input', str(query), '--groups', str(group_file), '--output', str(output), '--work-dir', str(work / 'workers'), '--batch-size', '1']
        process = subprocess.Popen(command, env=env, start_new_session=True)
        try:
            code = process.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
            raise TimeoutError(f'GSQ stage exceeded {timeout:.0f} s')
        if code != 0:
            raise RuntimeError(f'GSQ stage exited with code {code}')
    with np.load(output, allow_pickle=False) as z:
        saved = [str(x) for x in z['labels'].tolist()]
        assert sorted(saved) == sorted(labels) and len(saved) == 18
        logits = z['plain_logits'][:, [saved.index(l) for l in labels]].astype(np.float64)
        assert len(logits) == len(ids) and np.isfinite(logits).all()
    logits -= logits.max(1, keepdims=True)
    probs = np.exp(logits)
    probs /= probs.sum(1, keepdims=True)
    log(f'GSQ 27B scored {len(ids)} rows in {time.monotonic() - started:.0f} s')
    return probs


## 14. The Shared Prompt — Dense Semantic Geometry

Different answers can share the same subject without sharing the same author. The prompt-geometry stage estimates related passages using Qwen3 embedding cosine similarity, with a threshold of **0.70**.

The resulting grouping signal helps reconcile predictions among semantically related passages, while the attribution heads continue to read each response independently.


In [ ]:
%%writefile qwen_groups.py
import argparse
import json
import os
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer


def stage(base, assets, work):
    base, assets, staged = Path(base), Path(assets), Path(work) / 'qgroup-base'
    staged.mkdir(parents=True, exist_ok=True)
    index = json.loads((assets / 'base' / 'model.safetensors.index.json').read_text())
    for name in ('config.json', 'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json', 'merges.txt', 'vocab.json'):
        target = staged / name
        if not target.exists():
            shutil.copyfile(assets / 'base' / name, target)
    for name in sorted(set(index['weight_map'].values())):
        source, target = base / name, staged / name
        assert source.is_file(), str(source)
        if not target.exists():
            try:
                os.symlink(source, target)
            except OSError:
                shutil.copyfile(source, target)
    return staged


def main():
    p = argparse.ArgumentParser()
    for key in ('base', 'assets', 'query', 'out', 'work'):
        p.add_argument('--' + key, required=True)
    p.add_argument('--max_len', type=int, default=512)
    p.add_argument('--tok_budget', type=int, default=8192)
    p.add_argument('--mem_frac', type=float, default=0.88)
    a = p.parse_args()
    started = time.time()
    frame = pd.read_csv(a.query, dtype={'id': str})
    texts = frame.text.fillna('').astype(str).tolist()
    staged = stage(a.base, a.assets, a.work)
    tok = AutoTokenizer.from_pretrained(staged, local_files_only=True)
    auto, plain = tok('hello')['input_ids'], tok('hello', add_special_tokens=False)['input_ids']
    eos = auto[-1] if (len(auto) > len(plain) and auto[-1] != plain[-1]) else (tok.eos_token_id if tok.eos_token_id is not None else 151643)
    pad = tok.pad_token_id if tok.pad_token_id is not None else eos
    gpus = torch.cuda.device_count()
    assert gpus >= 1
    max_memory = {i: f'{int(torch.cuda.get_device_properties(i).total_memory * a.mem_frac / 2 ** 30)}GiB' for i in range(gpus)}
    model = AutoModel.from_pretrained(staged, local_files_only=True, dtype=torch.float16, device_map='auto', max_memory=max_memory).eval()
    first = next(model.parameters()).device
    ids = []
    for t in texts:
        x = tok(t, truncation=True, max_length=a.max_len - 1)['input_ids']
        ids.append(x if x[-1] == eos else x + [eos])
    order = np.argsort([len(x) for x in ids], kind='stable')
    emb = np.zeros((len(texts), model.config.get_text_config().hidden_size), np.float16)
    budget, done, ooms = a.tok_budget, 0, 0
    with torch.inference_mode():
        while done < len(order):
            end = done + 1
            while end < len(order) and (end - done + 1) * len(ids[order[end]]) <= budget:
                end += 1
            idx = order[done:end]
            try:
                width = max(len(ids[i]) for i in idx)
                inp = torch.full((len(idx), width), pad)
                att = torch.zeros((len(idx), width), dtype=torch.long)
                for r, i in enumerate(idx):
                    inp[r, :len(ids[i])] = torch.tensor(ids[i])
                    att[r, :len(ids[i])] = 1
                hidden = model(input_ids=inp.to(first), attention_mask=att.to(first), use_cache=False).last_hidden_state
                vec = hidden[torch.arange(len(idx), device=hidden.device), (att.sum(1) - 1).to(hidden.device)].float()
                emb[idx] = torch.nn.functional.normalize(vec, dim=-1).cpu().numpy().astype(np.float16)
                done = end
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                ooms += 1
                budget = max(len(ids[order[done]]), budget // 2)
                print(f'qwen_groups: OOM, token budget -> {budget}', flush=True)
    assert np.isfinite(emb).all()
    os.makedirs(os.path.dirname(os.path.abspath(a.out)), exist_ok=True)
    np.save(a.out, emb)
    print(f'qwen_groups: {len(texts)} rows, dim {emb.shape[1]}, {ooms} OOM retries, {time.time() - started:.0f} s', flush=True)


if __name__ == '__main__':
    main()


## 15. Fusion — Balancing Eighteen Candidate Authors

Core LoRA, Gaussian, ATLAS and specialized GSQ scores form the parent distribution. The native embedding branch then supplies an independent probability view. Final adjustment uses **Sinkhorn strength 0.30** and **target mix 0.35**.

These steps balance the eighteen author classes while preserving the structure of the underlying model scores.


In [ ]:
%%writefile infer.py
import os, sys, time, gc, argparse, json, re
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np, pandas as pd, scipy.sparse as sp, torch
from pipe import pipelined, spans_devices

LABELS = ['claude-fable-5','claude-opus-4-6','claude-opus-4-8','claude-opus-5','claude-sonnet-5',
 'gemini-3.1-pro-preview','gemini-3.6-flash','gemini-3.7-flash','gemini-3.8-flash','glm-5.3','glm-5.3-flash',
 'gpt-5.5','gpt-5.6-luna','gpt-5.6-sol','gpt-5.6-terra','gpt-6-astra','kimi-k3','qwen3.8-max']
L2I = {l: i for i, l in enumerate(LABELS)}

p = argparse.ArgumentParser()
p.add_argument('--data', required=True)
p.add_argument('--models', required=True)
p.add_argument('--out', default='submission.parquet')
p.add_argument('--comp_path', default='')
p.add_argument('--comp_w', type=float, default=0.0)
p.add_argument('--tfidf_w', type=float, default=1.0)
p.add_argument('--distort_w', type=float, default=0.0)
p.add_argument('--adapt_thr', type=float, default=0.15)
p.add_argument('--adapt_w', type=float, default=0.0)
p.add_argument('--extra', default='')
p.add_argument('--extra_w', type=float, default=0.5)
p.add_argument('--svc_w', type=float, default=0.0)
p.add_argument('--pool', default='')
p.add_argument('--sk_div', type=int, default=1)
p.add_argument('--sinkhorn', type=float, default=0.0)
p.add_argument('--sinkhorn_target_mix', type=float, default=0.35)
p.add_argument('--atlas_dir', default='')
p.add_argument('--atlas_base', default='')
p.add_argument('--atlas_mode', default='off', choices=['off', 'head', 'lite', 'mix'])
p.add_argument('--atlas_member_w', type=float, default=0.0)
p.add_argument('--atlas_k', type=int, default=6)
p.add_argument('--atlas_w', type=float, default=0.0)
p.add_argument('--atlas_margin', type=float, default=0.0)
p.add_argument('--atlas_bs', type=int, default=4)
p.add_argument('--atlas_chunk', type=int, default=3)
p.add_argument('--atlas_workers', type=int, default=2)
p.add_argument('--atlas_deadline', type=float, default=0.0)
p.add_argument('--save_lp', default='')
p.add_argument('--native_dir', default='')
p.add_argument('--native_base', default='')
p.add_argument('--native_share', type=float, default=0.0)
p.add_argument('--native_cent', type=int, default=1)
p.add_argument('--native_deadline', type=float, default=0.0)
p.add_argument('--optional_deadline', type=float, default=0.0)
p.add_argument('--native_work', default='/kaggle/temp/native')
p.add_argument('--native_min_agree', type=float, default=0.35)
p.add_argument('--prefit_core', default='')
p.add_argument('--gsq_dir', default='')
p.add_argument('--gsq_share', type=float, default=0.0)
p.add_argument('--gsq_deadline', type=float, default=0.0)
p.add_argument('--gsq_timeout', type=float, default=0.0)
p.add_argument('--gsq_work', default='/kaggle/temp/gsq')
p.add_argument('--gsq_min_agree', type=float, default=0.30)
p.add_argument('--group_embed_base', default='')
p.add_argument('--group_embed_assets', default='')
p.add_argument('--group_work', default='/kaggle/temp/qgroups')
p.add_argument('--group_sim', type=float, default=0.70)
a = p.parse_args()
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
T0 = time.time()
def log(*x): print(f'[{time.time()-T0:7.0f}s]', *x, flush=True)

train = pd.read_csv(os.path.join(a.data, 'train.csv'))
test = pd.read_csv(os.path.join(a.data, 'test.csv'))
train['text'] = train.text.fillna(''); test['text'] = test.text.fillna('')
y = train.model.map(L2I).values; Gtr = train.prompt_id.values
log('train', len(train), 'test', len(test))

def capped_agglo(S, cap=18, thr=0.02):
    n = len(S); A = S.astype(np.float32).copy(); np.fill_diagonal(A, -np.inf)
    size = np.ones(n, np.int32); alive = np.ones(n, bool); lab = np.arange(n)
    while True:
        ok = (size[:, None] + size[None, :] <= cap) & alive[:, None] & alive[None, :]
        M = np.where(ok, A, -np.inf); k = int(np.argmax(M)); i, j = divmod(k, n)
        if not np.isfinite(M[i, j]) or M[i, j] < thr: break
        new = (size[i] * A[i] + size[j] * A[j]) / (size[i] + size[j])
        A[i] = new; A[:, i] = new; A[i, i] = -np.inf; A[j] = -np.inf; A[:, j] = -np.inf
        size[i] += size[j]; alive[j] = False; lab[lab == j] = i
    return lab

from sklearn.feature_extraction.text import TfidfVectorizer
try:
    v = TfidfVectorizer(sublinear_tf=True, stop_words='english', min_df=2, max_df=0.05)
    Xc = v.fit_transform(test.text.values)
    Gte = capped_agglo((Xc @ Xc.T).toarray(), 18, a.adapt_thr)
except ValueError as e:
    log('clustering fallback:', e)
    Gte = np.arange(len(test))
log('test clusters', len(np.unique(Gte)), 'sizes', np.bincount(np.bincount(Gte)[np.bincount(Gte) > 0]).tolist())

if a.group_embed_base:
    try:
        import subprocess
        os.makedirs(a.group_work, exist_ok=True)
        emb_path = os.path.join(a.group_work, 'group_emb.npy')
        if os.path.exists(emb_path):
            os.remove(emb_path)
        gcmd = [sys.executable, '-u', os.path.join(os.path.dirname(os.path.abspath(__file__)), 'qwen_groups.py'), '--base', a.group_embed_base, '--assets', a.group_embed_assets,
                '--query', os.path.join(a.data, 'test.csv'), '--out', emb_path, '--work', a.group_work]
        gcode = subprocess.run(gcmd, env=dict(os.environ, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false')).returncode
        E_q = np.load(emb_path).astype(np.float32)
        assert gcode == 0 and E_q.shape[0] == len(test) and np.isfinite(E_q).all()
        G_q = capped_agglo(E_q @ E_q.T, 18, a.group_sim)
        _sz_old, _sz_new = np.bincount(Gte)[Gte], np.bincount(G_q)[G_q]
        log('qwen groups', len(np.unique(G_q)), 'clusters (tf-idf', len(np.unique(Gte)), ') rows in groups of 2..18: qwen', round(float(((_sz_new >= 2) & (_sz_new <= 18)).mean()), 3),
            'tf-idf', round(float(((_sz_old >= 2) & (_sz_old <= 18)).mean()), 3))
        assert (np.bincount(G_q).max()) <= 18
        Gte = G_q
        del E_q
    except Exception as _e:
        log('QWEN GROUPS SKIPPED, keeping the TF-IDF groups:', repr(_e))
LP = {}
EXT_T, EXT_Y = np.array([], dtype=object), np.array([], dtype=int)
EXTRA_DF = pd.DataFrame(columns=['text', 'model'])
def _norm_text_key(value):
    import unicodedata
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', str(value or ''))).strip()
if a.extra:
    _E = pd.read_parquet(a.extra) if a.extra.endswith('.parquet') else pd.read_csv(a.extra)
    if not {'text', 'model'}.issubset(_E.columns):
        raise ValueError(f'External file must contain text/model columns: {a.extra}')
    _E = _E[_E.model.isin(LABELS)].copy()
    _E['text'] = _E.text.fillna('').astype(str)
    _official_keys = {_norm_text_key(t) for t in train.text.values}
    _seen = set(_official_keys)
    _keep = []
    for _t in _E.text.values:
        _k = _norm_text_key(_t)
        _ok = bool(_k) and _k not in _seen
        _keep.append(_ok)
        if _ok: _seen.add(_k)
    EXTRA_DF = _E.loc[_keep, ['text', 'model']].reset_index(drop=True)
    EXT_T = EXTRA_DF.text.astype(str).values
    EXT_Y = EXTRA_DF.model.map(L2I).values.astype(int)
    log('matched-clean extra rows after exact de-dup', len(EXT_T))
FIT_T = np.concatenate([train.text.values, EXT_T])
FIT_Y = np.concatenate([y, EXT_Y])
FIT_W = np.concatenate([np.ones(len(train), dtype=np.float32), np.full(len(EXT_T), a.extra_w, dtype=np.float32)])
if a.tfidf_w > 0:
    from sklearn.linear_model import LogisticRegression
    mats_tr, mats_te = [], []
    for kw in [dict(analyzer='char_wb', ngram_range=(2, 5), min_df=3, max_features=60000),
               dict(token_pattern=r"\w+|[^\w\s]", ngram_range=(1, 2), min_df=3, max_features=30000)]:
        vec = TfidfVectorizer(sublinear_tf=True, lowercase=False, **kw).fit(FIT_T)
        mats_tr.append(vec.transform(FIT_T)); mats_te.append(vec.transform(test.text.values))
    LP['tfidf'] = LogisticRegression(C=20, max_iter=3000).fit(sp.hstack(mats_tr).tocsr(), FIT_Y, sample_weight=FIT_W).predict_log_proba(sp.hstack(mats_te).tocsr()) * a.tfidf_w
    gc.collect(); log('tfidf done')
else:
    LP['tfidf'] = np.zeros((len(test), 18), dtype=np.float32)
    log('tfidf skipped (zero ensemble weight)')
if a.svc_w > 0:
    import re as _re
    from scipy.sparse import hstack as _hs
    from sklearn.svm import LinearSVC
    _FW = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
    _shape = lambda t: _re.sub(r"[^\W\d_]+", lambda m_: "X" if m_[0].isupper() else "x", t)
    _func = lambda t: " ".join(w.lower() if w.lower() in _FW or not w.isalnum() else "CONTENT" for w in _re.findall(r"\w+|[^\w\s]", t))
    _parts = []
    _official = train[['text', 'model']].copy(); _official['text'] = _official.text.fillna('').astype(str)
    _official['_srcw'] = 1.0
    _parts.append(_official)
    _seen = {_norm_text_key(t) for t in _official.text.values}
    if len(EXTRA_DF):
        _e = EXTRA_DF[['text', 'model']].copy(); _e['_srcw'] = float(a.extra_w)
        _keep = []
        for _t in _e.text.values:
            _k = _norm_text_key(_t); _ok = bool(_k) and _k not in _seen
            _keep.append(_ok)
            if _ok: _seen.add(_k)
        _e = _e.loc[_keep].copy(); _parts.append(_e)
    if a.pool and os.path.exists(a.pool):
        _P = pd.read_parquet(a.pool)
        if not {'text', 'model'}.issubset(_P.columns):
            raise ValueError(f'Pool must contain text/model columns: {a.pool}')
        _P = _P[_P.model.isin(LABELS)][['text', 'model']].copy()
        _P['text'] = _P.text.fillna('').astype(str); _P['_srcw'] = 0.35
        _keep = []
        for _t in _P.text.values:
            _k = _norm_text_key(_t); _ok = bool(_k) and _k not in _seen
            _keep.append(_ok)
            if _ok: _seen.add(_k)
        _P = _P.loc[_keep].copy(); _parts.append(_P)
    _fitdf = pd.concat(_parts, ignore_index=True)
    _fit = _fitdf.text.astype(str).tolist(); _fy = _fitdf.model.map(L2I).to_numpy(dtype=int); _sw = _fitdf['_srcw'].to_numpy(dtype=np.float32)
    log(f'svc4 fit rows={len(_fit)} source_weighted; external clean={len(EXTRA_DF)}')
    _A, _B = [], []
    for view, lo, hi, mf, wv in [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("function", 1, 3, 50000, 0.5)]:
        prep = _shape if view == "shape" else _func if view == "function" else (lambda s_: s_)
        an = "char" if view == "shape" else "word" if view == "function" else view
        o = dict(analyzer=an, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2, max_features=mf, dtype=np.float32)
        if an == "word": o["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
        _v = TfidfVectorizer(**o); _A.append(_v.fit_transform([prep(t) for t in _fit]) * wv); _B.append(_v.transform([prep(t) for t in test.text.values]) * wv)
    _svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight='balanced')
    _svc.fit(_hs(_A, format='csr'), _fy, sample_weight=_sw)
    _d = _svc.decision_function(_hs(_B, format='csr'))
    LP['svc4'] = (_d - _d.max(1, keepdims=True) - np.log(np.exp(_d - _d.max(1, keepdims=True)).sum(1, keepdims=True))) * a.svc_w
    del _A, _B, _parts, _fitdf, _svc; gc.collect(); log('svc4 done')
if a.distort_w > 0:
    import re, collections
    wre = re.compile(r"[A-Za-z’']+")
    cnt = collections.Counter(w.lower() for t in FIT_T for w in wre.findall(t))
    keep = {w for w, _ in cnt.most_common(100)}
    dist = lambda arr: [wre.sub(lambda m_: m_.group(0) if m_.group(0).lower() in keep else '*', t) for t in arr]
    Dtr, Dte = dist(FIT_T), dist(test.text.values)
    vs = [TfidfVectorizer(sublinear_tf=True, lowercase=False, analyzer='char', ngram_range=(2, 6), min_df=2),
          TfidfVectorizer(sublinear_tf=True, lowercase=True, token_pattern=r"[^\s]+", ngram_range=(1, 3), min_df=2)]
    Xa = sp.hstack([v_.fit_transform(Dtr) for v_ in vs]).tocsr(); Xb = sp.hstack([v_.transform(Dte) for v_ in vs]).tocsr()
    LP['distort'] = LogisticRegression(C=10, max_iter=3000).fit(Xa, FIT_Y, sample_weight=FIT_W).predict_log_proba(Xb) * a.distort_w
    log('distort done')
if a.prefit_core:
    from ntl660_readout import core_votes
    _pv = core_votes(a.prefit_core, test.text.tolist())
    assert set(_pv) == {'svc4', 'tfidf', 'distort'} and all(v.shape == (len(test), 18) and np.isfinite(v).all() for v in _pv.values())
    LP.update(_pv)
    log('prefit core votes loaded', {k: round(float(abs(v).max()), 2) for k, v in _pv.items()})
from transformers import AutoTokenizer, AutoModel
@torch.no_grad()
def l24_predict(spec, texts, groups=None, train_texts=None):
    from transformers import AutoConfig
    from safetensors.torch import load_file
    from concurrent.futures import ThreadPoolExecutor
    adapters = spec.get('adapters') or [spec['adapter']]
    aw = spec.get('adapter_weights')
    if aw is None or len(aw) != len(adapters): aw = [1.0 / len(adapters)] * len(adapters)
    tok = AutoTokenizer.from_pretrained(spec['path'])
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    cfg = AutoConfig.from_pretrained(spec['path']); tc = cfg.text_config if hasattr(cfg, 'text_config') else cfg
    tc.num_hidden_layers = spec['keep']; tc.use_cache = False
    if getattr(tc, 'layer_types', None) is not None: tc.layer_types = tc.layer_types[:spec['keep']]
    _ms = []
    if spec.get('dp') and torch.cuda.device_count() >= 2:
        try:
            for _d in range(2):
                _ms.append(AutoModel.from_pretrained(spec['path'], config=cfg, dtype=torch.float16, device_map={'': _d}).eval())
                for _n in ('vision_tower', 'multi_modal_projector'):
                    if getattr(_ms[-1], _n, None) is not None: setattr(_ms[-1], _n, None)
                gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(_d)
                _ms[-1](input_ids=torch.full((spec.get('bs', 16), spec.get('max_len', 384)), tok.pad_token_id, device=f'cuda:{_d}'), output_hidden_states=True, use_cache=False)
                assert torch.cuda.max_memory_allocated(_d) < torch.cuda.get_device_properties(_d).total_memory - (1 << 30), 'under 1 GiB to spare'
        except Exception as _e:
            log(f'  l24 {spec["name"]}: one split copy instead of one per GPU ({_e!r:.80})'); _ms = []
        gc.collect(); torch.cuda.empty_cache()
    if not _ms: _ms = [AutoModel.from_pretrained(spec['path'], config=cfg, dtype=torch.float16, device_map='auto').eval()]
    m = _ms[0]
    if spec.get('no_final_norm'):
        for _m in _ms: _m.norm = torch.nn.Identity()
    mods = dict(m.named_modules()); first = next(m.parameters()).device; L = spec['layers']
    _mods = [dict(_m.named_modules()) for _m in _ms]
    _eos = spec.get('eos_id')
    _tk = lambda tx: [x + ([_eos] if _eos is not None else []) for x in tok(list(tx), truncation=True, max_length=spec.get('max_len', 384) - (1 if _eos is not None else 0))['input_ids']]
    enc = _tk(texts)
    order = np.argsort([len(x) for x in enc]); bs = spec.get('bs', 16)
    def lsm(z): return z - z.max(1, keepdims=True) - np.log(np.exp(z - z.max(1, keepdims=True)).sum(1, keepdims=True))
    P_acc, C_acc = 0.0, 0.0; FEATS = None; OVF = [0, 0]
    if train_texts is not None:
        enc_tr = _tk(train_texts)
    def featurize(enc_, norms, hid):
        order_ = np.argsort([len(x) for x in enc_])
        Z_ = np.zeros((len(enc_), hid * len(L)), np.float32)
        batches = [order_[i:i + bs] for i in range(0, len(order_), bs)]

        def feats(mm, b, check=True):
            dev = next(mm.parameters()).device
            width = max(len(enc_[j]) for j in b)
            ids = torch.full((len(b), width), tok.pad_token_id)
            att = torch.zeros((len(b), width), dtype=torch.long)
            for r, j in enumerate(b):
                ids[r, :len(enc_[j])] = torch.tensor(enc_[j])
                att[r, :len(enc_[j])] = 1
            with torch.no_grad():
                hs = mm(input_ids=ids.to(dev), attention_mask=att.to(dev), output_hidden_states=True, use_cache=False).hidden_states
                if check and not all(torch.isfinite(hs[l]).all() for l in L):
                    return None
                mask = att.unsqueeze(-1).float()
                mask[:, 0] = 0
                mg = {d_: mask.to(d_) for d_ in {hs[l].device for l in L}}
                return torch.cat([nm(((hs[l].float() * mg[hs[l].device]).sum(1) / mg[hs[l].device].sum(1)).cpu()) for l, nm in zip(L, norms)], -1).numpy()

        def put(b, r):
            Z_[b] = r

        def checked(k, b):
            mm = _ms[k]
            r = feats(mm, b)
            if r is None:
                mm.to(torch.bfloat16)
                r = feats(mm, b, check=False)
                mm.to(torch.float16)
                OVF[k] += 1
            put(b, r)

        def run_copy(k, n):
            for b in batches[k::n]:
                checked(k, b)

        if len(_ms) == 1:
            if spans_devices(_ms[0]):
                pipelined(batches, lambda b: feats(_ms[0], b), put, lambda b: checked(0, b), workers=2, tag='l24 ' + spec['name'], log=log)
            else:
                for b in batches:
                    checked(0, b)
        else:
            try:
                with ThreadPoolExecutor(len(_ms)) as _ex:
                    for _f in [_ex.submit(run_copy, k, len(_ms)) for k in range(len(_ms))]:
                        _f.result()
            except Exception as _e:
                log(f'  l24 {spec["name"]}: two-copy pass failed ({_e!r:.80}), one copy from here on')
                del _ms[1:], _mods[1:]
                gc.collect()
                torch.cuda.empty_cache()
                for b in batches:
                    checked(0, b)
        return Z_
    for ai, ad in enumerate(adapters):
        adp = os.path.join(ad, 'adapter') if os.path.exists(os.path.join(ad, 'adapter', 'adapter_config.json')) else ad
        acfg = json.load(open(os.path.join(adp, 'adapter_config.json'))); scale = acfg['lora_alpha'] / acfg['r']
        sd = load_file(os.path.join(adp, 'adapter_model.safetensors')); deltas = []
        for k, v in sd.items():
            if '.lora_A.' not in k: continue
            name = k.replace('base_model.model.', '', 1).split('.lora_A.')[0]
            if name not in mods and ('model.' + name) in mods: name = 'model.' + name
            B = sd[k.replace('.lora_A.', '.lora_B.')]
            for _md in _mods:
                W = _md[name].weight
                W += ((B.to(W.device, torch.float32) @ v.to(W.device, torch.float32)) * scale).to(W.dtype)
                deltas.append((W, B, v))
        if os.path.exists(os.path.join(ad, 'heads.pt')): H = torch.load(os.path.join(ad, 'heads.pt'), map_location='cpu')
        else: H = dict(load_file(os.path.join(ad, 'heads.safetensors')))
        if os.path.exists(os.path.join(ad, 'meta.json')):
            _lab = json.load(open(os.path.join(ad, 'meta.json')))['labels']; _perm = [_lab.index(l) for l in LABELS]
            for _p in ('plain', 'centered'): H[f'{_p}.weight'] = H[f'{_p}.weight'][_perm]; H[f'{_p}.bias'] = H[f'{_p}.bias'][_perm]
        hid = H['norms.0.weight'].shape[0]
        norms = [torch.nn.LayerNorm(hid) for _ in L]
        for i, nm in enumerate(norms): nm.load_state_dict({'weight': H[f'norms.{i}.weight'], 'bias': H[f'norms.{i}.bias']})
        Z = featurize(enc, norms, hid)
        if train_texts is not None and ai == len(adapters) - 1:
            FEATS = (Z.copy(), featurize(enc_tr, norms, hid))
        for W, B, A_ in deltas:
            W -= ((B.to(W.device, torch.float32) @ A_.to(W.device, torch.float32)) * scale).to(W.dtype)
        torch.cuda.empty_cache()
        del deltas, sd; gc.collect()
        hd = lambda p, X: lsm(X @ H[f'{p}.weight'].float().numpy().T + H[f'{p}.bias'].float().numpy())
        P_acc = P_acc + hd('plain', Z) * aw[ai]
        if groups is not None:
            Zc = Z.copy()
            for g in np.unique(groups):
                ii = groups == g; Zc[ii] -= Zc[ii].mean(0)
            C_acc = C_acc + hd('centered', Zc) * aw[ai]
        log(f'  l24 {spec["name"]}: adapter {os.path.basename(ad)} done (bf16 redo batches {sum(OVF)})')
    del m, mods, _ms, _mods; gc.collect(); torch.cuda.empty_cache()
    if train_texts is not None: return P_acc, (C_acc if groups is not None else None), FEATS
    return P_acc, (C_acc if groups is not None else None)

def gpu_mem():
    return ' '.join(f'gpu{i}:{torch.cuda.memory_allocated(i)/1e9:.1f}GB' for i in range(torch.cuda.device_count()))

specs = json.loads(a.models)
failed = []


def run_l24(spec):
    log('l24', spec['name'])
    out = {}
    if spec.get('gauss', 0) > 0:
        plain, cent, (Zte_g, Ztr_g) = l24_predict(spec, test.text.values, Gte if a.adapt_w > 0 else None, train_texts=train.text.values)
        import gauss as _ga
        prm = _ga.fit(Ztr_g, y, Gtr, dict(red='wpca', d=128, se_shrink='lw', sp_shrink='lw', lam_floor=1e-4, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0))
        prm['T1'] = prm['Tg'] = 3.0
        _sz = np.bincount(Gte)[Gte]
        _rows = (_sz >= 2) & (_sz <= 18)
        _q = _ga.scores(Ztr_g[[0, 1]], np.zeros(2, dtype=int), prm)
        assert np.allclose(np.exp(_q).sum(1), 1)
        _G = np.zeros((len(test), 18))
        if _rows.any():
            _G[_rows] = _ga.scores(Zte_g[_rows], np.unique(Gte[_rows], return_inverse=True)[1], prm)
        out[spec['name'] + '_gauss'] = _G * spec['gauss']
        log(f"  gaussian head {spec['name']}: {int(_rows.sum())} grouped test rows, weight {spec['gauss']}")
    else:
        plain, cent = l24_predict(spec, test.text.values, Gte if a.adapt_w > 0 else None)
    out[spec['name']] = plain * spec.get('w', 1.0)
    if cent is not None:
        rows = np.bincount(Gte)[Gte] >= 2
        out[spec['name'] + '_twin'] = np.where(rows[:, None], cent, 0.0) * a.adapt_w * spec.get('w', 1.0) * spec.get('twin', 1.0)
    LP.update(out)


def run_enc(spec):
    log('enc', spec['name'])
    from safetensors.torch import load_file as _lf
    _ck = spec['ckpt']
    _meta = json.load(open(os.path.join(_ck, 'meta.json')))
    _perm = [_meta['labels'].index(l) for l in LABELS]
    _hd = _lf(os.path.join(_ck, 'head.safetensors'))
    _W = _hd['weight'].float().numpy()[_perm]
    _b = _hd['bias'].float().numpy()[_perm]
    _etok = AutoTokenizer.from_pretrained(os.path.join(_ck, 'encoder'))
    _enc = AutoModel.from_pretrained(os.path.join(_ck, 'encoder'), dtype=torch.float32).to('cuda:0').eval()
    _ids = _etok(list(test.text.values), truncation=True, max_length=_meta['max_len'])['input_ids']
    _ord = np.argsort([len(x) for x in _ids], kind='stable')
    Z_ = np.zeros((len(_ids), _enc.config.hidden_size), np.float32)
    with torch.inference_mode():
        for _s in range(0, len(_ord), 32):
            _idx = _ord[_s:_s + 32]
            _bt = _etok.pad({'input_ids': [_ids[i] for i in _idx]}, return_tensors='pt').to('cuda:0')
            _m = _bt['attention_mask'].unsqueeze(-1).float()
            Z_[_idx] = ((_enc(**_bt).last_hidden_state * _m).sum(1) / _m.sum(1).clamp(min=1)).cpu().numpy()
    del _enc
    _zz = Z_ @ _W.T + _b
    LP[spec['name']] = (_zz - _zz.max(1, keepdims=True) - np.log(np.exp(_zz - _zz.max(1, keepdims=True)).sum(1, keepdims=True))) * spec.get('w', 1.0)


RUNNERS = {'l24': run_l24, 'enc': run_enc}
for spec in specs:
    if spec.get('optional') and a.optional_deadline and time.time() > a.optional_deadline:
        log('MEMBER SKIPPED (time):', spec['name'])
        failed.append(spec['name'])
        continue
    try:
        RUNNERS[spec['type']](spec)
    except Exception as _e:
        failed.append(spec['name'])
        log(f"MEMBER FAILED {spec['name']}: {_e!r}")
    gc.collect()
    torch.cuda.empty_cache()
    log(spec['name'], 'done', gpu_mem())
log('members failed:', failed if failed else 'none')

tot = sum(LP.values())
if a.save_lp:
    np.save(a.save_lp, tot)
COMP_P = None
if a.comp_w > 0 and a.comp_path and os.path.exists(a.comp_path):
    try:
        _cm = json.load(open(a.comp_path.replace('comp_logit.npy', 'comp_meta.json')))
        assert _cm['ids'] == test['id'].astype(str).tolist(), 'comp stack row order differs'
        _cl = np.load(a.comp_path)[:, [_cm['labels'].index(l) for l in LABELS]]
        COMP_P = np.exp(_cl - _cl.max(1, keepdims=True))
        COMP_P /= COMP_P.sum(1, keepdims=True)
        assert np.isfinite(COMP_P).all()
        log(f"comp stack loaded ({_cm['seconds']:.0f}s)")
    except Exception as _e:
        log('COMP SKIPPED:', repr(_e))
        COMP_P = None
elif a.comp_w > 0:
    log('COMP SKIPPED: no stack output')


def blend(terms):
    total = sum(terms.values())
    active = sum(abs(v).max() > 0 for v in terms.values())
    lpn = total / max(1, active) if a.sk_div else total
    P = np.exp(lpn - lpn.max(1, keepdims=True))
    P /= P.sum(1, keepdims=True)
    if COMP_P is not None:
        P = (1 - a.comp_w) * P + a.comp_w * COMP_P
        P /= P.sum(1, keepdims=True)
    return P


def decide_scores(P):
    Q = P.copy()
    rho = float(np.clip(a.sinkhorn_target_mix, 0.0, 1.0))
    marg = np.clip(P.mean(0), 1e-6, None)
    marg /= marg.sum()
    prior = np.bincount(y, minlength=18).astype(np.float64) + 1.0
    prior /= prior.sum()
    target = np.clip((1.0 - rho) * marg + rho * prior, 1e-6, None)
    target /= target.sum()
    for _ in range(100):
        Q *= (len(Q) * target[None, :]) / np.clip(Q.sum(0, keepdims=True), 1e-12, None)
        Q /= np.clip(Q.sum(1, keepdims=True), 1e-12, None)
    return (1 - a.sinkhorn) * np.log(P + 1e-12) + a.sinkhorn * np.log(Q + 1e-12)


def decide(P):
    return decide_scores(P).argmax(1)


def write_pred(pred, tag):
    sub = pd.DataFrame({'id': test.id.values, 'model': [LABELS[i] for i in pred]})
    assert sub.model.isin(LABELS).all() and sub.id.is_unique and len(sub) == len(test)
    sub.to_parquet(a.out, index=False)
    log(f'wrote {tag}', sub.model.value_counts().to_dict())


def write(P, tag):
    write_pred(decide(P), tag)


P = blend(LP)
write(P, 'blend without ATLAS')


def atlas_lite(k):
    import atlas_infer
    scorer = atlas_infer.Scorer(a.atlas_base, a.atlas_dir, log=log)
    try:
        enc = scorer.encode(test.text.values)
        to_b = np.array([scorer.labels.index(l) for l in LABELS])
        cand = np.argsort(-P, 1)[:, :k]
        mean_lp, done = scorer.run_candidates(enc, to_b[cand], a.atlas_deadline or None, workers=a.atlas_workers)
        lp = scorer.candidate_logprobs(mean_lp[done], to_b[cand][done])
    finally:
        scorer.close()
    rows = np.repeat(lp.min(1, keepdims=True), 18, 1)
    np.put_along_axis(rows, cand[done], lp, 1)
    term = np.zeros((len(test), 18))
    term[done] = rows
    return term, done


def atlas_scores(rows, calibrate):
    import atlas_infer
    scorer = atlas_infer.Scorer(a.atlas_base, a.atlas_dir, log=log)
    try:
        enc = scorer.encode(test.text.values)
        out, done = scorer.run(enc, rows, deadline=a.atlas_deadline or None, batch=a.atlas_bs, chunk=a.atlas_chunk, workers=a.atlas_workers)
        order = [scorer.labels.index(l) for l in LABELS]
        if calibrate:
            res = np.zeros((len(test), 18))
            res[done] = scorer.calibrated(out[done])
        else:
            raw = out[..., 1:].mean(-1)
            res = np.divide(raw, raw.sum(1, keepdims=True), out=np.zeros_like(raw), where=raw.sum(1, keepdims=True) > 0)
    finally:
        scorer.close()
    return res[:, order], done


if a.atlas_mode != 'off' and a.atlas_dir:
    if a.atlas_deadline and time.time() > a.atlas_deadline - 600:
        log('ATLAS SKIPPED: not enough time left')
    else:
        try:
            if a.atlas_mode == 'lite':
                term, done = atlas_lite(a.atlas_k)
                P = blend({**LP, 'atlas': a.atlas_member_w * term})
                log(f'atlas lite top-{a.atlas_k} on {int(done.sum())}/{len(test)} rows at weight {a.atlas_member_w}', gpu_mem())
            elif a.atlas_mode == 'head':
                lp, done = atlas_scores(np.arange(len(test)), True)
                P = blend({**LP, 'atlas': a.atlas_member_w * lp})
                log(f'atlas head on {int(done.sum())}/{len(test)} rows at weight {a.atlas_member_w}', gpu_mem())
            else:
                top2 = np.sort(P, 1)
                margin = top2[:, -1] - top2[:, -2]
                rows = np.where(margin < a.atlas_margin)[0] if a.atlas_margin > 0 else np.arange(len(test))
                A_, done = atlas_scores(rows, False)
                use = rows[done[rows]]
                mixed = P.copy()
                mixed[use] += a.atlas_w * (A_[use] - P[use])
                P = mixed / mixed.sum(1, keepdims=True)
                log(f'atlas mixed on {len(use)}/{len(rows)} rows at weight {a.atlas_w}', gpu_mem())
        except Exception as _e:
            log('ATLAS SKIPPED:', repr(_e))
        write(P, 'final blend')


if a.gsq_share > 0 and a.gsq_dir:
    if a.gsq_deadline and time.time() > a.gsq_deadline:
        log('GSQ SKIPPED: not enough time left')
    else:
        try:
            import gsq_stage as _gs
            import shutil as _sh
            _sh.rmtree('/tmp/llama-3.1', ignore_errors=True)
            gc.collect()
            torch.cuda.empty_cache()
            Q27 = _gs.run(a.gsq_dir, test['id'].astype(str).tolist(), test.text.tolist(), Gte, a.gsq_work, LABELS, log, a.gsq_timeout or None)
            _agree = float((Q27.argmax(1) == P.argmax(1)).mean())
            log(f'GSQ 27B agreement with the blend {_agree:.3f}')
            if _agree < a.gsq_min_agree:
                raise RuntimeError(f'GSQ agreement {_agree:.3f} below {a.gsq_min_agree}: refusing to use it')
            P = (1 - a.gsq_share) * P + a.gsq_share * Q27
            P /= P.sum(1, keepdims=True)
            write(P, 'blend with GSQ 27B')
        except Exception as _e:
            log('GSQ SKIPPED:', repr(_e))


def rowsoftmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def native_stage(P):
    import subprocess
    import native_q8
    os.makedirs(a.native_work, exist_ok=True)
    out = os.path.join(a.native_work, 'native.npz')
    if os.path.exists(out):
        os.remove(out)
    gc.collect()
    torch.cuda.empty_cache()
    cmd = [sys.executable, '-u', os.path.join(os.path.dirname(os.path.abspath(__file__)), 'native_q8.py'), '--base', a.native_base, '--assets', a.native_dir,
           '--query', os.path.join(a.data, 'test.csv'), '--out', out, '--work', a.native_work, '--deadline', str(a.native_deadline)]
    env = dict(os.environ, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false')
    code = subprocess.run(cmd, env=env).returncode
    if not os.path.exists(out):
        raise RuntimeError(f'native worker produced nothing (exit code {code})')
    z = np.load(out, allow_pickle=False)
    folds = z['folds'].tolist()
    assert z['ids'].astype(str).tolist() == test['id'].astype(str).tolist(), 'native row order differs'
    labels = z['labels'].astype(str).tolist()
    order = [labels.index(l) for l in LABELS]
    states = [json.load(open(os.path.join(a.native_dir, 'calibration', f'fold{k}.json')))['models']['qwen8'] for k in folds]
    lp = native_q8.calibrate(z['plain'][:, :, order], z['centered'][:, :, order], z['centered_bias'][:, order], Gte, states, bool(a.native_cent))
    share = (a.native_share * z['finite'])[:, None]
    logP = np.log(np.maximum(P, 1e-15))
    agree = [float((lp[i].argmax(1) == P.argmax(1)).mean()) for i in range(len(folds))]
    log(f'native folds {folds} (worker exit {code}); agreement with the blend per fold', [round(v, 3) for v in agree],
        'rows without native', int((~z['finite']).sum()))
    if min(agree) < a.native_min_agree:
        raise RuntimeError(f'native agreement {min(agree):.3f} below {a.native_min_agree}: refusing to use it')
    probs = np.zeros_like(P)
    for i in range(len(folds)):
        probs += rowsoftmax(decide_scores(rowsoftmax((1 - share) * logP + share * lp[i]))) / len(folds)
    assert np.isfinite(probs).all() and np.allclose(probs.sum(1), 1)
    return probs


if a.native_share > 0 and a.native_dir and a.native_base:
    if a.native_deadline and time.time() > a.native_deadline - 1500:
        log('NATIVE SKIPPED: not enough time left')
    else:
        try:
            probs = native_stage(P)
            changed = int((probs.argmax(1) != decide(P)).sum())
            write_pred(probs.argmax(1), 'final blend with native q8')
            log(f'native q8 at share {a.native_share}: {changed}/{len(test)} decisions differ from the blend-only decision', gpu_mem())
        except Exception as _e:
            log('NATIVE SKIPPED:', repr(_e))


## 16. A Complementary Signature — Stylometric Stack

The complementary branch incorporates DeBERTa-v3 Large and sparse word, character and shape features. It provides additional sensitivity to local typography and vocabulary habits that may be muted in semantic embeddings.

Its coefficient is **0.15** in the public inference command.


In [ ]:
%%writefile comp_stack.py
import gc, glob, json, os, pathlib, re, time
import numpy as np, pandas as pd
try:
    import torch
except ImportError:
    torch = None
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.sparse import csr_matrix, diags, hstack, vstack
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from pipe import pipelined, spans_devices

SMOKE = os.environ.get('NTL_SMOKE') == '1'
T0 = time.time()
if torch is not None:
    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
LABELS = sorted(['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
                 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
                 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
                 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'])
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}
FAMILY_IDX = {f: np.array([i for i, l in enumerate(LABELS) if l.startswith(f)]) for f in ('claude', 'gpt', 'gemini', 'glm')}

GROUP_THRESH, GROUP_LAYER, GROUP_MIN, GROUP_MAX = 0.60, 24, 2, 18
STYLE_GROUP_THRESH = 0.90

W = {'stylegram': 1.516, 'stylometry_lgb': 0.063, 'ultrabert': 0.095,
     'q3_8b': 0.0, 'q25_7b': 0.0, 'q3_4b': 0.0, 'glm9b': 0.0, 'debv3l': 0.05, 'electl': 0.032, 'desk': 0.132,
     'lora24_public': 0.177, 'lora24_seed7': 0.113, 'lora24_claude': 0.177, 'lora20_q25_instyle': 0.17,
     'tokattn_q35': 0.153, 'tokattn_deb': 0.185}
TWIN_W = {'q3_8b': 1.5 * 0.219, 'q25_7b': 1.5 * 0.141, 'q3_4b': 1.5 * 0.083, 'glm9b': 1.5 * 0.110, 'debv3l': 0.25}
LORA_TWIN_W = {'lora24_public': 0.15, 'lora24_claude': 0.15}
GATE_LAM = {'claude_style': 1.5, 'claude_lora': 1.0, 'family': 0.25}
EXTRA_MIN_LABELS, EXTRA_SHARE = 12, 0.5

GAUSS_HEADS = {'gauss_q3_8b_L22': dict(src='q3_8b', part='L22_mean', d=384, T1=2.351, Tg=2.306, w=TWIN_W['q3_8b']),
               'gauss_glm9b_L8': dict(src='glm9b', part='L8_mean', d=384, T1=2.722, Tg=2.596, w=TWIN_W['glm9b']),
               'gauss_q3_4b_L4': dict(src='q3_4b', part='L4_mean', d=384, T1=2.725, Tg=2.578, w=TWIN_W['q3_4b']),
               'gauss_debv3l_L24': dict(src='debv3l', part='L24_mean', d=384, T1=3.112, Tg=3.069, w=TWIN_W['debv3l'])}
GAUSS_SETTING = dict(red='wpca', se_shrink='lw', sp_shrink='lw', lam_floor=1e-4, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0)

LM_HEADS = [
    ('glm9b', 'glm4', 4096, [(8, 'mean'), (3, 'mean')], [(GROUP_LAYER, 'mean')], 0.0025),
    ('q3_8b', 'qwen3', 4096, [(7, 'mean'), (22, 'mean')], [], 0.0025),
    ('q25_7b', 'qwen2', 3584, [(6, 'mean'), (20, 'mean')], [], 0.0025),
    ('q3_4b', 'qwen3', 2560, [(4, 'style'), (7, 'style')], [(4, 'mean')], 0.0025)]
ENC_HEADS = [
    ('debv3l', 'deberta-v3-large', [(24, 'mean'), (6, 'mean')], 0.005),
    ('electl', 'electra-large-discriminator', [(21, 'mean'), (24, 'style')], 0.005),
    ('desk', 'desklib-detector', [(23, 'mean'), (23, 'style')], 0.005)]
LORA_RUNS = [
    (('qwen3', 4096, 24), [2, 4, 6, 8, 12, 16, 20, 24], [
        ('lora24_public', '**/lora24/final/COMPLETE.json', 'heads.safetensors', 'anchor_features.npz'),
        ('lora24_seed7', '**/name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json', 'norms.safetensors', 'valid_features.npz'),
        ('lora24_claude', '**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json', 'heads.safetensors', 'anchor_features.npz')]),
    (('qwen2', 3584, 20), [2, 4, 6, 8, 10, 12, 16, 20], [
        ('lora20_q25_instyle', '**/ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json', 'norms.safetensors', 'valid_features.npz')])]
LORA_MAX_LEN, LORA_BS = 447, 8
TOKATTN_HEADS = [('tokattn_q35', 'tokattn_final.pt', 'qwen3_5'), ('tokattn_deb', 'tokattn_debl23_final.pt', 'deberta')]
if os.environ.get('NTL_DROP_Q35') == '1':
    TOKATTN_HEADS = [t for t in TOKATTN_HEADS if t[0] != 'tokattn_q35']

if SMOKE:
    _all = pd.read_csv(os.environ['NTL_DATA'] + '/train.csv')
    _p = _all['prompt_id'].drop_duplicates().sample(frac=1, random_state=0).tolist()
    train = _all[_all['prompt_id'].isin(_p[:30])].reset_index(drop=True)
    test = _all[_all['prompt_id'].isin(_p[30:42])].sample(frac=1, random_state=1).reset_index(drop=True)
    test = pd.concat([test[test['prompt_id'] == p].head(k) for p, k in zip(_p[30:42], [1, 1, 2, 2, 2, 3, 3, 4, 6, 9, 18, 18])]).reset_index(drop=True)
    if os.environ.get('NTL_TINY') == '1':
        test = test.head(3).reset_index(drop=True)
    SMOKE_Y = test['model'].map(LAB2I).to_numpy(dtype=int)
else:
    DATA_DIR = next(p for p in [pathlib.Path('/kaggle/input/competitions/name-that-llm'), pathlib.Path('/kaggle/input/name-that-llm')]
                    if (p / 'train.csv').exists())
    train, test = pd.read_csv(DATA_DIR / 'train.csv'), pd.read_csv(DATA_DIR / 'test.csv')
    if os.environ.get('NTL_FAKE_TEST'):
        test = pd.read_parquet(os.environ['NTL_POOL']).sample(int(os.environ['NTL_FAKE_TEST']), random_state=0)[['text']].reset_index(drop=True)
        test['id'] = [f'fake{i}' for i in range(len(test))]
test['text'] = test['text'].fillna('').astype(str)
TR, TE = train['text'].tolist(), test['text'].tolist()
y = train['model'].map(LAB2I).to_numpy(dtype=int)
PROMPT = train['prompt_id'].to_numpy(dtype=object)
TINY = len(TE) < 50
DEV = 'cuda' if torch is not None and torch.cuda.is_available() else 'cpu'
print(f'train {train.shape}  test {test.shape}  gpus {torch.cuda.device_count() if torch is not None else 0}  smoke {SMOKE}', flush=True)

LOGIT = np.zeros((len(TE), K))


def add_member(key, prob, rows=None, weight=None):
    w = W[key] if weight is None else weight
    lp = np.log(np.clip(prob, 1e-9, 1.0))
    if rows is None:
        LOGIT[:] += w * lp
    else:
        LOGIT[rows] += w * lp
    print(f'  + {key:20s} weight {w:.3f} on {len(TE) if rows is None else int(np.sum(rows))} rows  ({time.time() - T0:.0f}s)', flush=True)


def softmax_rows(S):
    S = S - S.max(1, keepdims=True)
    P = np.exp(S)
    return P / P.sum(1, keepdims=True)


def full_prob(P_part, classes):
    P = np.zeros((len(P_part), K))
    P[:, classes] = P_part
    return P


def group_centre_dense(X, groups):
    X = np.asarray(X, dtype=np.float32).copy()
    for g in np.unique(groups):
        i = groups == g
        X[i] -= X[i].mean(0)
    return X


def self_test_rows():
    idx = np.where(np.isin(PROMPT, pd.unique(PROMPT)[:3]))[0]
    return idx, np.unique(PROMPT[idx], return_inverse=True)[1]

def average_linkage_groups(D, thresh):
    if len(D) < 2:
        return np.ones(len(D), dtype=int)
    D = np.asarray(D, dtype=np.float64).copy()
    np.fill_diagonal(D, 0.0)
    return fcluster(linkage(squareform(D, checks=False), method='average'), thresh, criterion='distance')


def content_cosine_distance(texts, max_features=None):
    X = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=max_features, dtype=np.float32).fit_transform(texts)
    return np.clip(1.0 - (X @ X.T).toarray(), 0.0, 1.0)


def mixed_groups(texts, E, thresh=GROUP_THRESH):
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    Z = np.asarray(E, dtype=np.float64)
    Z = Z - Z.mean(0)
    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12
    D = 0.5 * (content_cosine_distance(texts) + np.clip(1.0 - Z @ Z.T, 0.0, 2.0))
    return average_linkage_groups(D, thresh)


def normalised_content_groups(texts, thresh=STYLE_GROUP_THRESH):
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    D = content_cosine_distance(texts, max_features=100_000)
    med = max(float(np.median(D[np.triu_indices(len(texts), 1)])), 1e-6)
    return average_linkage_groups(D / med, thresh)


def eligible(groups, lo=GROUP_MIN, hi=GROUP_MAX):
    size = np.bincount(groups)[groups]
    return (size >= lo) & (size <= hi)


def describe_groups(name, groups):
    size = np.bincount(groups)[groups]
    print(f'{name}: {len(groups)} rows | singletons {int((size == 1).sum())}  2-3: {int(((size >= 2) & (size <= 3)).sum())}'
          f'  4-18: {int(((size >= 4) & (size <= 18)).sum())}  >18: {int((size > 18).sum())}', flush=True)


G_TE, GROUP_ROWS = None, None

FUNCTION_WORDS = frozenset("a an the and or but if while although because as so than that which who whom whose this these those it its "
                           "they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with "
                           "without for of by over under between through into about is are was were be been being have has had do does did "
                           "can could will would shall should may might must not no nor all any some each every both either neither other "
                           "another such only also just very too more most less least here there when where why how what whether then now "
                           "still yet even much many few little".split())
TOKEN_RE = re.compile(r"\w+|[^\w\s]")


def shape_view(text):
    return re.sub(r"[^\W\d_]+", lambda m: 'X' if m[0].isupper() else 'x', text)


def function_view(text):
    return ' '.join(t.lower() if (t.lower() in FUNCTION_WORDS or not t.isalnum()) else 'CONTENT' for t in TOKEN_RE.findall(text))


class StyleGrams:
    VIEWS = [('char', str, (3, 5), 250_000, 1.0), ('word', str, (1, 3), 180_000, 0.7),
             ('char', shape_view, (2, 6), 60_000, 0.5), ('word', function_view, (1, 3), 50_000, 0.5)]

    def fit(self, texts):
        self.vecs = []
        for analyzer, view, ngr, maxf, w in self.VIEWS:
            opts = dict(analyzer=analyzer, ngram_range=ngr, lowercase=False, sublinear_tf=True, min_df=2, max_features=maxf, dtype=np.float64)
            if analyzer == 'word':
                opts['token_pattern'] = r"(?u)\b\w+\b|[^\w\s]"
            self.vecs.append((TfidfVectorizer(**opts).fit([view(t) for t in texts]), view, w))
        return self

    def transform(self, texts):
        return hstack([v.transform([view(t) for t in texts]) * w for v, view, w in self.vecs], format='csr')


def sparse_group_centre(X, groups, lo=GROUP_MIN, hi=GROUP_MAX):
    X = csr_matrix(X)
    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)
    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))
    if not len(keep):
        return X.copy()
    col = np.full(len(cnt), -1); col[keep] = np.arange(len(keep))
    rows = np.flatnonzero(col[inv] >= 0)
    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))
    means = diags(1.0 / cnt[keep]) @ (M.T @ X)
    return (X - M @ means).tocsr()


def svc_probs(svc, X):
    return full_prob(softmax_rows(svc.decision_function(X)), svc.classes_)


def load_extra_passages():
    sources = [('**/ntl_extra_passages_claude.csv', 'model', None), ('**/gen_passages.csv', 'model', None),
               ('**/ntl_generations.csv', 'label', 'finish_reason'), ('**/ntl_proxy_generations.csv', 'label', None),
               ('**/ntl_generated*.csv', 'model', 'finish'), ('**/passages_clean.csv', 'model', None)]
    parts = []
    for pattern, label_col, finish_col in sources:
        for f in sorted(glob.glob('/kaggle/input/' + pattern, recursive=True)):
            d = pd.read_csv(f)
            _pin = {'ntl_generations.csv': lambda d: d['batch'].astype(str).isin(
                        {'b001', 'b001g', 'b003', 'b003a', 'd1028', 'd1029', 'd1030', 'pilot', 'pilot2', 'pilot3', 'pilotpro'}),
                    'gen_passages.csv': lambda d: d['batch'].astype(str).isin(
                        {f'b{i}' for i in (1, 2, *range(5, 16))} | {f'v2b{i:02d}' for i in range(1, 21)}),
                    'passages_clean.csv': lambda d: pd.to_numeric(d['dataset_version_added'], errors='coerce').isin([1, 2, 4, 5])
                    }.get(os.path.basename(f))
            if _pin is not None:
                try:
                    d = d[_pin(d)]
                except KeyError as _e:
                    print('extra-passage pin skipped for', f, repr(_e), flush=True)
            if finish_col and finish_col in d:
                d = d[d[finish_col].eq('stop')]
            d = d.rename(columns={label_col: 'model'})[['prompt_id', 'model', 'text']].copy()
            d['prompt_id'] = ('pv_' if f.endswith('passages_clean.csv') else '') + d['prompt_id'].astype(str)
            parts.append(d)
    for f in sorted(glob.glob('/kaggle/input/**/ntl_open_pool.parquet', recursive=True))[:1]:
        d = pd.read_parquet(f, columns=['prompt_group', 'model', 'text']).rename(columns={'prompt_group': 'prompt_id'})
        d['prompt_id'] = 'op_' + d['prompt_id'].astype(str)
        parts.append(d)
    if not parts:
        return pd.DataFrame(columns=['prompt_id', 'model', 'text'])
    x = pd.concat(parts, ignore_index=True)
    x = x[x['model'].isin(LABELS)].copy()
    x['text'] = x['text'].astype(str).map(lambda t: re.sub(r'\s+', ' ', re.sub(r'^\*\*(.*)\*\*$', r'\1', t.strip())).strip())
    x = x[x['text'].str.split().str.len().between(30, 300)].drop_duplicates(subset=['model', 'text'])
    x = x.drop_duplicates(subset=['prompt_id', 'model'])
    x = x[x.groupby('prompt_id')['model'].transform('nunique') >= EXTRA_MIN_LABELS].reset_index(drop=True)
    return x


EXTRA = load_extra_passages()
EX_T, EX_Y, EX_G = EXTRA['text'].tolist(), EXTRA['model'].map(LAB2I).to_numpy(dtype=int), EXTRA['prompt_id'].to_numpy(dtype=object)
print(f'extra passages: {len(EXTRA)} rows, {EXTRA["prompt_id"].nunique()} prompts', flush=True)


def extra_sample_weights():
    n_tr, n_ex = np.bincount(y, minlength=K).astype(float), np.bincount(EX_Y, minlength=K).astype(float)
    w_tr = np.where(n_ex[y] > 0, 1.0 - EXTRA_SHARE, 1.0)
    w_ex = EXTRA_SHARE * n_tr[EX_Y] / np.maximum(n_ex[EX_Y], 1.0)
    return np.concatenate([w_tr, w_ex])


grams = StyleGrams().fit(TR + EX_T)
X_tr, X_ex, X_te = grams.transform(TR), grams.transform(EX_T) if len(EX_T) else None, grams.transform(TE)


def fit_svc(X, labels, C=0.5, **kw):
    return LinearSVC(C=C, random_state=42, max_iter=4000).fit(X, labels, **kw)


if X_ex is not None:
    svc_plain = fit_svc(vstack([X_tr, X_ex]).tocsr(), np.concatenate([y, EX_Y]), sample_weight=extra_sample_weights())
else:
    svc_plain = fit_svc(X_tr, y)
P_style = svc_probs(svc_plain, X_te)

X_tr_c = sparse_group_centre(X_tr, PROMPT)
if X_ex is not None:
    svc_centred = fit_svc(vstack([X_tr_c, sparse_group_centre(X_ex, EX_G, GROUP_MIN, 10 ** 6)]).tocsr(), np.concatenate([y, EX_Y]),
                          sample_weight=extra_sample_weights())
else:
    svc_centred = fit_svc(X_tr_c, y)
G_style = normalised_content_groups(TE)
rows_style = eligible(G_style)
describe_groups('stylegram groups', G_style)
if rows_style.any():
    P_style[rows_style] = svc_probs(svc_centred, sparse_group_centre(X_te[rows_style], G_style[rows_style]))
if TINY:
    st_idx, st_g = self_test_rows()
    acc = np.mean(svc_centred.classes_[svc_centred.decision_function(sparse_group_centre(X_tr[st_idx], st_g)).argmax(1)] == y[st_idx])
    print(f'  centred stylegram self-test on {len(st_idx)} training rows (seen in training): acc {acc:.2f}', flush=True)
add_member('stylegram', P_style)
del X_tr_c, svc_plain, svc_centred, P_style


def family_gate(texts, labels, classes, C, sample_weight=None, balanced=False):
    g = StyleGrams().fit(texts)
    svc = LinearSVC(C=C, class_weight='balanced' if balanced else None, random_state=42, max_iter=4000).fit(
        g.transform(texts), labels, sample_weight=sample_weight)
    assert list(svc.classes_) == list(classes)
    d = svc.decision_function(g.transform(TE))
    return np.column_stack([-d, d]) if len(classes) == 2 else d


GATES = {}
api_claude = sorted(glob.glob('/kaggle/input/**/ntl_extra_passages_claude.csv', recursive=True))
if api_claude:
    api = pd.read_csv(api_claude[0])
    m = np.isin(y, FAMILY_IDX['claude'])
    GATES['claude'] = family_gate([t for t, k in zip(TR, m) if k] + api['text'].astype(str).tolist(),
                                  np.concatenate([y[m], api['model'].map(LAB2I).to_numpy(dtype=int)]), FAMILY_IDX['claude'], 0.5)
    print(f'claude gate: {int(m.sum())} official + {len(api)} API rows', flush=True)
    del api
else:
    print('Claude API passages not attached: Claude gate off', flush=True)
for fam, C in [('gpt', 1.0), ('gemini', 1.0), ('glm', 0.5)]:
    m, me = np.isin(y, FAMILY_IDX[fam]), np.isin(EX_Y, FAMILY_IDX[fam])
    GATES[fam] = family_gate([t for t, k in zip(TR, m) if k] + [t for t, k in zip(EX_T, me) if k], np.concatenate([y[m], EX_Y[me]]),
                             FAMILY_IDX[fam], C, sample_weight=np.concatenate([np.ones(int(m.sum())), np.full(int(me.sum()), 0.25)]), balanced=True)
    print(f'{fam} gate: {int(m.sum())} official + {int(me.sum())} extra rows', flush=True)
del grams, X_tr, X_ex, X_te

PUNCT = list(',.;:!?()[]"\'-') + ['—', '–', '‘', '’', '“', '”', '…', '*', '/', '%']
STOPS = sorted(ENGLISH_STOP_WORDS)
SENT_RE, WORD_RE = re.compile(r'(?<=[.!?])\s+'), re.compile(r"[A-Za-z’']+")


def stylometry(t):
    words = WORD_RE.findall(t)
    low = [w.lower() for w in words]
    nw, nc = max(len(words), 1), max(len(t), 1)
    sents = [s for s in SENT_RE.split(t.strip()) if s]
    sl = np.array([len(WORD_RE.findall(s)) for s in sents] or [0], dtype=float)
    wl = np.array([len(w) for w in words] or [0], dtype=float)
    f = [np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(), wl.mean(), wl.std(), (wl >= 9).mean(), len(set(low)) / nw,
         sum(c.isupper() for c in t) / nc, sum(c.isdigit() for c in t) / nc, sum(ord(c) > 127 for c in t) / nc,
         t.count(', ') / nw, t.count(' and ') / nw, t.count(' but ') / nw,
         sum(1 for s in sents if s[:1].isupper() and s.split()[0].lower() in ('this', 'these', 'it', 'they')) / max(len(sents), 1),
         sum(w.endswith('ly') for w in words) / nw, sum(w.endswith('ing') for w in words) / nw, sum(w.endswith('tion') for w in words) / nw]
    f += [t.count(q) / nw for q in PUNCT]
    counts = pd.Series(low).value_counts() if low else pd.Series(dtype=float)
    f += [counts.get(s, 0) / nw for s in STOPS]
    return f


F_tr, F_te = (np.array([stylometry(t) for t in txt], dtype=np.float32) for txt in (TR, TE))
try:
    import lightgbm as lgb
    booster = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=10, subsample=0.8, subsample_freq=1,
                                 colsample_bytree=0.5, reg_lambda=1.0, random_state=0, verbose=-1, n_jobs=4).fit(F_tr, y)
    add_member('stylometry_lgb', full_prob(booster.predict_proba(F_te), booster.classes_))
    del booster
except ImportError:
    assert SMOKE, 'lightgbm missing'
    print('smoke: lightgbm unavailable, stylometry member skipped', flush=True)
del F_tr, F_te

import itertools
K = 18
DEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,
           "iters": 5, "repel": True, "sp_scale": 1.0}

def _lse(A, axis=None, keepdims=False):
    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
    if keepdims: return out
    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

def log_softmax(L): return L - _lse(L, axis=1, keepdims=True)

def _lw_shrinkage(X):
    from sklearn.covariance import ledoit_wolf_shrinkage
    return float(ledoit_wolf_shrinkage(X, assume_centered=True))

def _shrunk(S, a):
    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)

def _two_way(Z, y, pinv, G, iters=4):
    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)
    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))
    for _ in range(iters):
        R = Z - P[pinv]
        for c in cls: mu[c] = R[y == c].mean(0)
        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]
    return mu, P, cnt_p

def fit(X, y, prompts, cfg=None):
    c = dict(DEFAULT); c.update(cfg or {})
    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape
    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)
    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx
    if c["red"] == "none":
        V = np.eye(D)
    else:
        if c["red"] == "wpca":
            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]
        elif c["red"] == "pca": R = Xs
        else: raise ValueError(c["red"])
        d = int(min(c["d"], N - G - 1, D))
        if N < D:
            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]
            V = (R.T @ U) / np.sqrt(w)
        else:
            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T
    Z = Xs @ V; d = Z.shape[1]
    mu, P, ng = _two_way(Z, y, pinv, G)
    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = E.T @ E / dof
    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)
    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]
    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)
    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li
    W = V @ A.T
    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,
            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}

def transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]

_MASK = {}
def _inj_mask(n, C):
    if (n, C) not in _MASK:
        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)
        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]
        _MASK[(n, C)] = bad
    return _MASK[(n, C)]

def singleton_ll(U, m, lam):
    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

def joint_terms(U, m, lam):
    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]
    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)
    return a, B

def joint_table(a, B):
    n, C = a.shape; Q = np.zeros([C] * n)
    for i in range(n):
        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)
    for i, j in itertools.combinations(range(n), 2):
        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)
    return Q

def exact_marginals(Q, T=1.0, inj=True):
    n = Q.ndim; C = Q.shape[0]; Qt = Q / T
    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)
    lz = _lse(Qt); out = np.empty((n, C))
    for i in range(n):
        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz
    return out

def em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):
    n = len(U); v = lam / (1 + (n - 1) * lam)
    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
    q = np.exp(log_softmax(L))
    for _ in range(iters):
        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v
        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
        if repel:
            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)
        q = np.exp(log_softmax(L))
    return log_softmax(L)

LAM_DIFF = 1e6
def lam_g(prm, cfg=None):
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})
    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]

def group_logpost(U, prm, T1=None, Tg=None, cfg=None):
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)
    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg
    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)
    if n <= c["n_exact"]:
        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])
    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])

def scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))
    sizes = np.bincount(lab); s1 = sizes[lab] == 1
    T1 = prm["T1"] if T1 is None else T1
    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)
    for g in np.where(sizes >= 2)[0]:
        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)
    return out

if not SMOKE:
    from transformers import AutoConfig, AutoModel, AutoTokenizer
CACHED_PARTS = {}


def find_checkpoint(model_type, hidden=None, dir_name=None):
    pattern = f'/kaggle/input/**/{dir_name}/config.json' if dir_name else '/kaggle/input/**/config.json'
    for cfg in sorted(glob.glob(pattern, recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if dir_name or (c.get('model_type') == model_type and (hidden is None or c.get('hidden_size') == hidden)):
            return os.path.dirname(cfg)
    return None


def load_truncated_decoder(path, keep, dtype):
    cfg = AutoConfig.from_pretrained(path)
    cfg.num_hidden_layers = keep
    if getattr(cfg, 'layer_types', None):
        cfg.layer_types = cfg.layer_types[:keep]
    kw = dict(config=cfg, dtype=dtype)
    if DEV == 'cuda':
        kw['device_map'] = 'auto'
    return AutoModel.from_pretrained(path, **kw).eval()


def style_token_mask(tok):
    pieces = [tok.decode([i]) for i in range(len(tok))]
    return torch.tensor([(p.strip().lower() in ENGLISH_STOP_WORDS) or bool(re.fullmatch(r'\s*[^\w\s]+\s*', p)) for p in pieces])


@torch.no_grad()
def pooled_states(model, tok, texts, parts, decoder, bs, max_len=512):
    H = model.config.hidden_size
    out = {f'L{L}_{pool}': np.zeros((len(texts), H), dtype=np.float32) for L, pool in parts}
    smask = style_token_mask(tok) if any(pool == 'style' for _, pool in parts) else None
    dev = next(model.parameters()).device
    order = np.argsort([len(t) for t in texts])
    retried = [0]
    batches = [order[b0:b0 + bs] for b0 in range(0, len(texts), bs)]

    def compute(idx, check):
        if decoder:
            batch = [(tok.bos_token or '') + texts[i] for i in idx]
            enc = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=max_len, add_special_tokens=False).to(dev)
        else:
            enc = tok([texts[i] for i in idx], return_tensors='pt', padding=True, truncation=True, max_length=max_len).to(dev)
        with torch.no_grad():
            hs = model(**enc, output_hidden_states=True).hidden_states
            if check and decoder and not all(torch.isfinite(hs[L]).all() for L, _ in parts):
                return None
            m_all = enc['attention_mask'].float()
            res = {}
            for L, pool in parts:
                m = m_all * smask.to(m_all.device)[enc['input_ids']].float() if pool == 'style' else m_all
                m = m.unsqueeze(-1).to(hs[L].device)
                res[f'L{L}_{pool}'] = ((hs[L].float() * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
        return res

    def put(idx, res):
        for k, v in res.items():
            out[k][idx] = v

    def checked(idx):
        res = compute(idx, True)
        if res is None:
            retried[0] += 1
            model.to(torch.bfloat16)
            res = compute(idx, False)
        put(idx, res)

    if decoder and spans_devices(model):
        pipelined(batches, lambda idx: compute(idx, True), put, checked, workers=2, tag='pooled states', log=print)
    else:
        for idx in batches:
            checked(idx)
    if retried[0]:
        print(f'  {retried[0]} batches redone in bf16', flush=True)
    return {k: np.nan_to_num(v) for k, v in out.items()}


def pseudo_parts(parts, dim=48):
    from sklearn.decomposition import TruncatedSVD
    tf = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), sublinear_tf=True, min_df=2).fit(TR + TE)
    out = {}
    for j, (L, pool) in enumerate(parts):
        svd = TruncatedSVD(dim, random_state=L + 7 * j).fit(tf.transform(TR))
        out[f'L{L}_{pool}'] = (svd.transform(tf.transform(TR)).astype(np.float32), svd.transform(tf.transform(TE)).astype(np.float32))
    return out


def extract_parts(key, parts, loader):
    if SMOKE:
        return pseudo_parts(parts)
    model, tok, decoder, bs = loader()
    tr, te = pooled_states(model, tok, TR, parts, decoder, bs), pooled_states(model, tok, TE, parts, decoder, bs)
    del model, tok
    gc.collect()
    if DEV == 'cuda':
        torch.cuda.empty_cache()
    return {k: (tr[k], te[k]) for k in tr}


def decoder_loader(model_type, hidden, keep):
    def load():
        path = find_checkpoint(model_type, hidden)
        assert path, f'no {model_type} checkpoint with hidden size {hidden} under /kaggle/input'
        tok = AutoTokenizer.from_pretrained(path)
        tok.padding_side = 'right'
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        return load_truncated_decoder(path, keep, torch.float16 if DEV == 'cuda' else torch.float32), tok, True, 8
    return load


def encoder_loader(dir_name):
    def load():
        path = find_checkpoint(None, dir_name=dir_name)
        assert path, f'encoder directory {dir_name} not found under /kaggle/input'
        return AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval(), AutoTokenizer.from_pretrained(path), False, 16
    return load


def concat_parts(cache, parts):
    return (np.concatenate([cache[f'L{L}_{p}'][0] for L, p in parts], 1), np.concatenate([cache[f'L{L}_{p}'][1] for L, p in parts], 1))


def plain_and_twin(key, Etr, Ete, C):
    if W[key] > 0:
        sc = StandardScaler().fit(Etr)
        clf = LogisticRegression(C=C, max_iter=5000).fit(sc.transform(Etr), y)
        Xte = (Ete - Ete.mean(0)) / (Ete.std(0) + 1e-6) if len(Ete) >= 2 else sc.transform(Ete)
        add_member(key, full_prob(clf.predict_proba(Xte), clf.classes_))
    else:
        print(f'  {key}: plain head has weight 0 in the 0.563 blend, not fitted', flush=True)
    if key in TWIN_W:
        csc = StandardScaler().fit(group_centre_dense(Etr, PROMPT))
        twin = LogisticRegression(C=C, max_iter=5000).fit(csc.transform(group_centre_dense(Etr, PROMPT)), y)
        if GROUP_ROWS.any():
            pc = twin.predict_proba(csc.transform(group_centre_dense(Ete[GROUP_ROWS], G_TE[GROUP_ROWS])))
            add_member(key + '_twin', full_prob(pc, twin.classes_), rows=GROUP_ROWS, weight=TWIN_W[key])
        if TINY:
            st_idx, st_g = self_test_rows()
            ps = twin.predict_proba(csc.transform(group_centre_dense(Etr[st_idx], st_g)))
            print(f'  {key} twin self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(twin.classes_[ps.argmax(1)] == y[st_idx]):.2f}', flush=True)


def gaussian_group_head(name, spec, Xtr, Xte):
    prm = fit(Xtr, y, PROMPT, dict(GAUSS_SETTING, d=spec['d']))
    prm['T1'], prm['Tg'] = spec['T1'], spec['Tg']
    print(f'  {name}: d {prm["info"]["d"]}, shrinkage e/p {prm["info"]["a_e"]}/{prm["info"]["a_p"]}, lam pct {prm["info"]["lam_pct"]}', flush=True)
    if GROUP_ROWS.any():
        LP = scores(Xte[GROUP_ROWS], np.unique(G_TE[GROUP_ROWS], return_inverse=True)[1], prm)
        add_member(name, np.exp(LP), rows=GROUP_ROWS, weight=spec['w'])
    if TINY:
        st_idx, st_g = self_test_rows()
        LP = scores(Xtr[st_idx], st_g, prm)
        print(f'  {name} self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(LP.argmax(1) == y[st_idx]):.2f}', flush=True)


for key, model_type, hidden, parts, extra_parts, C in LM_HEADS:
    print(f'[{key}] {model_type} hidden {hidden} parts {parts + extra_parts}', flush=True)
    cache = extract_parts(key, parts + extra_parts, decoder_loader(model_type, hidden, max(L for L, _ in parts + extra_parts) + 1))
    if G_TE is None:
        G_TE = mixed_groups(TE, cache[f'L{GROUP_LAYER}_mean'][1])
        GROUP_ROWS = eligible(G_TE)
        describe_groups('test groups (content + GLM L24)', G_TE)
        if SMOKE:
            same_g, same_p = G_TE[:, None] == G_TE[None, :], test['prompt_id'].to_numpy(dtype=object)[:, None] == test['prompt_id'].to_numpy(dtype=object)[None, :]
            off = ~np.eye(len(TE), dtype=bool)
            print(f'  smoke grouping: pair precision {(same_g & same_p & off).sum() / max((same_g & off).sum(), 1):.2f} '
                  f'recall {(same_g & same_p & off).sum() / max((same_p & off).sum(), 1):.2f}', flush=True)
    CACHED_PARTS[key] = cache
    plain_and_twin(key, *concat_parts(cache, parts), C)
    for name, spec in GAUSS_HEADS.items():
        if spec['src'] == key:
            gaussian_group_head(name, spec, *cache[spec['part']])
    del cache
    CACHED_PARTS.pop(key)
    gc.collect()

for key, dir_name, parts, C in ENC_HEADS:
    print(f'[{key}] {dir_name} parts {parts}', flush=True)
    cache = extract_parts(key, parts, encoder_loader(dir_name))
    plain_and_twin(key, *concat_parts(cache, parts), C)
    for name, spec in GAUSS_HEADS.items():
        if spec['src'] == key:
            gaussian_group_head(name, spec, *cache[spec['part']])
    del cache
    gc.collect()

@torch.no_grad()
def score_lora_checkpoints(trunk_spec, pool_layers, checkpoints):
    import importlib.util as iu
    _find = iu.find_spec
    iu.find_spec = lambda name, *a, **k: None if name.split('.')[0] == 'torchao' else _find(name, *a, **k)
    from peft import PeftModel
    import peft.import_utils as pi
    pi.is_torchao_available = lambda *a, **k: False
    from safetensors.torch import load_file
    model_type, H, keep = trunk_spec
    base = find_checkpoint(model_type, H)
    assert base, f'no {model_type} base checkpoint under /kaggle/input'
    cfg = AutoConfig.from_pretrained(base)
    cfg.num_hidden_layers, cfg.use_cache = keep, False
    if getattr(cfg, 'layer_types', None) is not None:
        cfg.layer_types = cfg.layer_types[:keep]
    trunk = AutoModel.from_pretrained(base, config=cfg, dtype=torch.float16, device_map='auto').eval()
    for key, marker, norms_file, anchor_file in checkpoints:
        hits = sorted(glob.glob('/kaggle/input/' + marker, recursive=True))
        if not hits:
            print(f'  SKIP {key}: checkpoint not attached ({marker})', flush=True); continue
        ck = pathlib.Path(os.path.dirname(hits[0]))
        print(f'[{key}] {ck}', flush=True)
        tok = AutoTokenizer.from_pretrained(ck / 'tokenizer')
        tok.padding_side = 'right'
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        lm = PeftModel.from_pretrained(trunk, ck / 'adapter').eval()
        state = load_file(str(ck / norms_file))
        norms = torch.nn.ModuleList([torch.nn.LayerNorm(H) for _ in pool_layers])
        norms.load_state_dict({k[len('norms.'):]: v for k, v in state.items() if k.startswith('norms.')})
        ro = np.load(ck / 'readouts.npz')
        assert list(ro['classes']) == LABELS and list(ro['pool_layers']) == pool_layers
        dev = lm.get_input_embeddings().weight.device
        norms = norms.to(dev).float()

        def token_ids(texts):
            return [ids + [tok.eos_token_id] for ids in tok(list(texts), add_special_tokens=False, truncation=True, max_length=LORA_MAX_LEN)['input_ids']]

        def features(texts, bs):
            ids = token_ids(texts)
            Z = np.zeros((len(ids), H * len(pool_layers)), dtype=np.float32)
            order = np.argsort([len(t) for t in ids], kind='stable')
            batches = [order[b0:b0 + bs] for b0 in range(0, len(order), bs)]

            def work(idx):
                batch = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(dev)
                with torch.no_grad():
                    with torch.autocast('cuda', dtype=torch.float16):
                        hs = lm(**batch, output_hidden_states=True, use_cache=False).hidden_states
                    m = batch['attention_mask'].unsqueeze(-1).float()
                    m[:, 0] = 0
                    return torch.cat([nm((hs[L].float() * m.to(hs[L].device)).sum(1).to(dev) / m.sum(1).to(dev))
                                      for L, nm in zip(pool_layers, norms)], -1).cpu().numpy()

            def put(idx, res):
                Z[idx] = res

            pipelined(batches, work, put, lambda idx: put(idx, work(idx)), workers=2 if spans_devices(lm) else 1, tag='lora ' + key, log=print)
            return Z

        if (ck / anchor_file).exists():
            saved = np.load(ck / anchor_file)['features'].astype(np.float32)
            ours = features(TR[:3], 3)
            rel = float(np.abs(ours - saved).mean() / (np.abs(saved).mean() + 1e-9))
            print(f'  parity with the checkpoint anchors: max abs {float(np.abs(ours - saved).max()):.4f}, mean rel {rel:.5f}', flush=True)
            assert rel < 0.02, f'{key}: our features do not match the checkpoint'
        Z = features(TE, LORA_BS)
        add_member(key, softmax_rows(Z @ ro['plain_coef'].T + ro['plain_intercept']))
        if 'claude_coef' in ro.files:
            lc = Z @ ro['claude_coef'].T + ro['claude_intercept']
            GATES['claude_lora'] = lc - np.log(np.exp(lc - lc.max(1, keepdims=True)).sum(1, keepdims=True)) - lc.max(1, keepdims=True)
        if key in LORA_TWIN_W and GROUP_ROWS.any():
            Zc = group_centre_dense(Z[GROUP_ROWS], G_TE[GROUP_ROWS])
            add_member(key + '_twin', softmax_rows(Zc @ ro['center_coef'].T + ro['center_intercept']), rows=GROUP_ROWS, weight=LORA_TWIN_W[key])
        trunk = lm.unload()
        del lm, Z
        gc.collect()
        torch.cuda.empty_cache()
    del trunk
    gc.collect()
    torch.cuda.empty_cache()


if not SMOKE:
    for trunk_spec, pool_layers, checkpoints in LORA_RUNS:
        score_lora_checkpoints(trunk_spec, pool_layers, checkpoints)

class TokenAttention(torch.nn.Module if torch is not None else object):
    def __init__(self, k, h=4096):
        super().__init__()
        self.drop, self.proj, self.score, self.out = torch.nn.Dropout(0.2), torch.nn.Linear(h, 64), torch.nn.Linear(64, 1), torch.nn.Linear(128, k)

    def forward(self, x, m):
        h = torch.tanh(self.proj(self.drop(x)))
        a = torch.softmax(self.score(h).squeeze(-1).masked_fill(~m, -1e4), -1).unsqueeze(-1)
        mf = m.unsqueeze(-1).float()
        return self.out(self.drop(torch.cat([(h * mf).sum(1) / mf.sum(1).clamp(min=1), (a * h).sum(1)], -1)))


@torch.no_grad()
def token_attention_probs(texts, head_file, backbone, bs=8):
    hits = sorted(glob.glob('/kaggle/input/**/' + head_file, recursive=True))
    assert hits, f'training-kernel output holding {head_file} not attached'
    d = torch.load(hits[0], map_location='cpu')
    assert list(d['classes']) == LABELS
    decoder = d.get('decoder', True)
    path = find_checkpoint('qwen3_5') if backbone == 'qwen3_5' else find_checkpoint(None, dir_name='deberta-v3-large')
    assert path, f'backbone {backbone} not attached'
    cfg = AutoConfig.from_pretrained(path)
    if decoder:
        tc = getattr(cfg, 'text_config', None) or cfg
        tc.num_hidden_layers = d['keep']
        if getattr(tc, 'layer_types', None):
            tc.layer_types = tc.layer_types[:d['keep']]
        lm = AutoModel.from_pretrained(path, config=cfg, dtype=torch.float16, device_map='auto').eval()
    else:
        lm = AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval()
    tok = AutoTokenizer.from_pretrained(path)
    tok.padding_side = 'right'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    H = d.get('hidden', 4096)
    net = TokenAttention(K, H)
    net.load_state_dict(d['state_dict'])
    dev = lm.get_input_embeddings().weight.device
    net, mu, sd = net.to(dev).eval(), d['mu'].float().to(dev), d['sd'].float().to(dev)
    ids = tok(list(texts), add_special_tokens=not decoder, truncation=True, max_length=d['max_len'])['input_ids']
    P = np.zeros((len(texts), K))
    order = np.argsort([len(t) for t in ids])
    for b0 in range(0, len(order), bs):
        idx = order[b0:b0 + bs]
        batch = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(dev)
        out = lm(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], output_hidden_states=True)
        hs = (out.hidden_states if getattr(out, 'hidden_states', None) is not None else out.language_model_outputs.hidden_states)[d['layer']]
        n = batch['attention_mask'].sum(1)
        T = max(int(n.max()) - 1, 1)
        x = torch.zeros(len(idx), T, H, device=dev)
        m = torch.zeros(len(idx), T, dtype=torch.bool, device=dev)
        for r in range(len(idx)):
            k = int(n[r]) - 1
            if k > 0:
                x[r, :k] = ((torch.nan_to_num(hs[r, 1:k + 1].to(dev).float()).half().float() - mu) / sd).clamp(-8, 8)
                m[r, :k] = True
        P[idx] = torch.softmax(net(x, m), -1).cpu().numpy()
        del hs, out, batch
    del lm
    gc.collect()
    torch.cuda.empty_cache()
    return P


@torch.no_grad()
def ultrabert_probs(texts):
    from transformers import AutoModelForSequenceClassification
    folds = None
    for root in glob.glob('/kaggle/input/**/ultrabert-380m-ntl', recursive=True):
        fs = [f for f in sorted(glob.glob(os.path.join(root, 'fold*'))) if os.path.exists(os.path.join(f, 'config.json'))
              and json.load(open(os.path.join(f, 'config.json'))).get('model_type') == 'ultrabert']
        if len(fs) == 5:
            folds = fs
            break
    assert folds, 'ultrabert-380m-ntl (5 folds) not attached'
    tok = AutoTokenizer.from_pretrained(folds[0], trust_remote_code=True)
    ids = tok(texts, truncation=True, max_length=512)['input_ids']
    order = np.argsort([len(e) for e in ids])
    P = np.zeros((len(texts), K))
    for fp in folds:
        em = AutoModelForSequenceClassification.from_pretrained(fp, trust_remote_code=True, dtype=torch.float16, attn_implementation='sdpa').to(DEV).eval()
        cols = [LAB2I[em.config.id2label[i]] for i in range(len(em.config.id2label))]
        for b0 in range(0, len(order), 32):
            idx = order[b0:b0 + 32]
            b = tok.pad({'input_ids': [ids[j] for j in idx]}, return_tensors='pt').to(DEV)
            P[np.ix_(idx, cols)] += em(**b).logits.float().softmax(-1).cpu().numpy() / len(folds)
        del em
        gc.collect()
        torch.cuda.empty_cache()
    return P


if not SMOKE:
    _available_tokattn = []
    for _key, _head_file, _backbone in TOKATTN_HEADS:
        _head_hits = glob.glob('/kaggle/input/**/' + _head_file, recursive=True)
        _base_hit = find_checkpoint('qwen3_5') if _backbone == 'qwen3_5' else find_checkpoint(None, dir_name='deberta-v3-large')
        if _head_hits and _base_hit:
            _available_tokattn.append((_key, _head_file, _backbone))
        else:
            print(f'  OPTIONAL HEAD SKIPPED {_key}: head={bool(_head_hits)} backbone={bool(_base_hit)}', flush=True)
    TOKATTN_HEADS = _available_tokattn
    for key, head_file, backbone in TOKATTN_HEADS:
        if TINY:
            acc = float(np.mean(token_attention_probs(TR[:40], head_file, backbone).argmax(1) == y[:40]))
            print(f'  {key} in-sample check on 40 training rows: acc {acc:.2f}', flush=True)
            assert acc > 0.5, f'{key} does not reproduce its training fit'
        add_member(key, token_attention_probs(TE, head_file, backbone))
    add_member('ultrabert', ultrabert_probs(TE))


def _np_logsoftmax(z):
    from scipy.special import logsumexp as _lse_np
    z = np.asarray(z, dtype=np.float64)
    return z - _lse_np(z, axis=1, keepdims=True)

def _apply_conditional_family_gate(gate_key, family, lam, scores_are_logprob=False):
    score = GATES.get(gate_key)
    if score is None:
        print(f'  family gate skipped {gate_key}: no fitted scores', flush=True)
        return
    idx = FAMILY_IDX[family]
    score = np.asarray(score, dtype=np.float64)
    if score.shape != (len(TE), len(idx)):
        print(f'  family gate skipped {gate_key}: shape {score.shape} expected {(len(TE), len(idx))}', flush=True)
        return
    if scores_are_logprob:
        gate_lp = _np_logsoftmax(score)
    else:
        score = score - score.mean(axis=1, keepdims=True)
        scale = np.sqrt(np.mean(score * score, axis=1, keepdims=True))
        score = score / np.maximum(scale, 0.25)
        gate_lp = _np_logsoftmax(score)
    base = LOGIT[:, idx].copy()
    base_lp = _np_logsoftmax(base)
    base_p = np.exp(base_lp)
    sp = np.sort(base_p, axis=1)
    margin = sp[:, -1] - sp[:, -2]
    confidence_gate = np.clip((0.45 - margin) / 0.45, 0.0, 1.0)
    strength = min(0.45, float(lam) / (1.0 + float(lam))) * confidence_gate
    mixed_lp = _np_logsoftmax((1.0 - strength[:, None]) * base_lp + strength[:, None] * gate_lp)
    from scipy.special import logsumexp as _lse_np
    family_mass = _lse_np(base, axis=1)
    LOGIT[:, idx] = family_mass[:, None] + mixed_lp
    print(f'  family gate applied {gate_key}: mean strength={strength.mean():.4f}, active rows={int((strength > 0).sum())}', flush=True)

if not SMOKE:
    _apply_conditional_family_gate('claude', 'claude', GATE_LAM['claude_style'])
    _apply_conditional_family_gate('claude_lora', 'claude', GATE_LAM['claude_lora'], scores_are_logprob=True)
    for _fam in ('gpt', 'gemini', 'glm'):
        _apply_conditional_family_gate(_fam, _fam, GATE_LAM['family'])

np.save('/kaggle/working/comp_logit.npy', LOGIT.astype(np.float32))
json.dump({'labels': LABELS, 'ids': test['id'].astype(str).tolist(), 'seconds': time.time() - T0,
           'family_gates_applied': bool(not SMOKE)}, open('/kaggle/working/comp_meta.json', 'w'))
print(f'COMP STACK DONE {time.time() - T0:.0f}s', flush=True)


## 17. The Complete Examination — Execution and Checkpoints

The pipeline stages the trained adapters and large immutable weights, runs the complementary and primary inference processes, and produces a single prediction per passage. The key public coefficients are **native 0.40**, **GSQ 0.03**, **complementary 0.15**, **Sinkhorn 0.30 / 0.35**, and **clean LoRA adapters 0.72 / 0.28**.

The same source code contains a lightweight path for the public sample and the full model execution path for competition scoring.


In [ ]:
def run(cmd, env=None):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
    for line in proc.stdout:
        if not re.search(r'Warning|warn', line):
            print(line, end='')
    return proc.wait()


COMP_SEC = 0.0
if not FAST_COMMIT:
    t0 = time.time()
    code = run([sys.executable, 'comp_stack.py'], env=dict(os.environ, NTL_DROP_Q35='1', NTL_PIPE_WORKERS='1'))
    COMP_SEC = time.time() - t0
    print('comp stack exit code', code, 'wall', round(COMP_SEC), 's')


In [ ]:
LAYERS = [2, 4, 6, 8, 12, 16, 20, 24]
MODELS = [
    {"name": "q14v5", "type": "l24", "path": B14, "adapters": AQ14V5, "keep": 27, "layers": LAYERS, "w": 3.0, "bs": 8, "max_len": 384, "twin": 0.35, "gauss": 0.70},
    {"name": "q8v5", "type": "l24", "path": B8, "adapters": AQ8V5, "keep": 24, "layers": LAYERS, "w": 2.0, "bs": 16, "max_len": 384, "twin": 0.35, "gauss": 0.44, "dp": True},
    {"name": "ll8v5", "type": "l24", "path": BL, "adapters": ALL8V5, "keep": 24, "layers": LAYERS, "w": 1.0, "bs": 16, "max_len": 384, "twin": 0.35, "dp": True},
    {"name": "n14b_clean", "type": "l24", "path": B14, "adapters": [OQ4, N14B], "adapter_weights": [0.72, 0.28], "keep": N14BC['keep'], "layers": N14BC['layers'],
     "w": 2.5, "bs": 8, "max_len": N14BC['max_len'], "twin": 0.35, "no_final_norm": True, "eos_id": N14BC['eos_id']},
    {"name": "q25_14", "type": "l24", "path": B25_14, "adapters": [Q25_14], "keep": Q25_14C['keep'], "layers": Q25_14C['layers'], "w": 1.0, "bs": 8,
     "max_len": Q25_14C['max_len'], "twin": 0.35, "no_final_norm": True, "eos_id": Q25_14C['eos_id']},
    {"name": "deb", "type": "enc", "ckpt": DEB, "w": 1.0},
]
OUT = '/kaggle/working/submission.parquet'
cmd = [sys.executable, 'infer.py', '--data', DATA, '--models', json.dumps(MODELS),
       '--prefit_core', PREFIT, '--tfidf_w', '0', '--distort_w', '0', '--svc_w', '0',
       '--adapt_w', '1.0', '--adapt_thr', '0.15', '--sinkhorn', '0.30', '--sinkhorn_target_mix', '0.35',
       '--native_dir', NATIVE, '--native_base', NATIVE_BASE, '--native_share', '0.40', '--native_cent', '1',
       '--native_deadline', str(NB_T0 + RUN_LIMIT_S),
       '--group_embed_base', NATIVE_BASE, '--group_embed_assets', NATIVE, '--group_sim', '0.70',
       '--gsq_dir', GSQ_ASSETS, '--gsq_share', '0.03', '--gsq_deadline', str(NB_T0 + RUN_LIMIT_S - 9000), '--gsq_timeout', '5400',
       '--comp_path', '/kaggle/working/comp_logit.npy', '--comp_w', '0.15', '--out', OUT]
if FAST_COMMIT:
    ids = pd.read_csv(os.path.join(DATA, 'test.csv')).id
    pd.DataFrame({'id': ids, 'model': 'gpt-5.5'}).to_parquet(OUT, index=False)
    sys.path.insert(0, os.getcwd())
    from ntl660_readout import core_votes
    smoke = core_votes(PREFIT, pd.read_csv(os.path.join(DATA, 'test.csv')).text.fillna('').tolist())
    assert set(smoke) == {'svc4', 'tfidf', 'distort'} and all(len(v) == len(ids) and np.isfinite(v).all() for v in smoke.values())
    print('prefit core votes smoke ok', {k: v.shape for k, v in smoke.items()})
else:
    print('infer exit code', run(cmd, env=dict(os.environ, NTL_PIPE_WORKERS='1')))


## 18. The Final Verdict — Submission Integrity

The finished prediction table must have exactly two columns, `id` and `model`, with one non-null class label for every test ID. The last stage checks uniqueness, row-count agreement and class distribution before writing `submission.parquet`.


In [ ]:
submission = pd.read_parquet(OUT)
test_ids = pd.read_csv(os.path.join(DATA, 'test.csv')).id
assert len(submission) == len(test_ids) and set(submission.id) == set(test_ids)
assert list(submission.columns) == ['id', 'model'] and submission.id.is_unique and submission.model.notna().all()
print(submission.model.value_counts().to_string())
for path in ['/kaggle/working/open_pool.parquet', '/kaggle/working/extra_matched.parquet']:
    if os.path.exists(path):
        os.remove(path)
shutil.rmtree('/kaggle/working/deberta_v3_large', ignore_errors=True)
print('total', round(time.time() - NB_T0), 's')


---
<a id="version-trail"></a>
## 19. How the Public Solution Evolved

Our central investigation remains the same: five forensic experiments ask whether a model's writing habits survive changes in prompt and topic. The inference architecture has changed as new public model combinations became available.

### Earlier releases

| Release | Public LB | Inference lineage |
|:--|--:|:--|
| V10 | **0.631** | Early multimodel reproduction |
| V14 | **0.661** | Improved ensemble and Open Pool correction |
| [V15](https://www.kaggle.com/code/evgendvorkin/name-that-llm?scriptVersionId=365897799) | **0.668** | Stable Open Pool and checked inference |
| V16 | **0.668** | Five original forensic experiments and figures |
| V17 | **0.669** | [Abood PS's Sinkhorn](https://www.kaggle.com/code/aboodps/ntl-672-sinkhorn-050/notebook) |
| V18 | **0.670** | [Areeb Ali's DeBERTa ensemble](https://www.kaggle.com/code/areebbb/0-670-this-actually-works/notebook?scriptVersionId=366234454) |
| [V19](https://www.kaggle.com/code/evgendvorkin/name-that-llm-deep-forensic-eda-with-analogie?scriptVersionId=366613626) | **0.673** | [Areeb Ali's ModernBERT blend](https://www.kaggle.com/code/areebbb/0-673-i-m-better/notebook) |
| V24–V25 | **0.680–0.681** | [Vladislav Lavrentev's parent and native embedding](https://www.kaggle.com/code/lightningv08/0-681-qwen3-embedding-lora-qwen3-8-27b) |
| V26 | **0.682** | [Omer Faruk Merey's GSQ and target mix](https://www.kaggle.com/code/omerfarukmerey/0-682-0-681-qwen3-8-27b-gsq-12-mix-0-50?scriptVersionId=367805055) |
| V27 | **0.683** | [Omkar Pattanaik's Sovereign Titan Apex](https://www.kaggle.com/code/omkarpattanaik123/name-that-llm-v28-sovereign-titan-apex) |

### Clean LoRA and the native five-fold expert

The next public implementation combines five backbone readers with a **0.72 / 0.28 clean Qwen3-14B adapter blend**, a **40% Qwen3-Embedding native expert** and a **3% GSQ branch**. The source pipeline also includes dense prompt geometry and **0.30 / 0.35 Sinkhorn balancing**. The independent forensic experiments above remain unchanged.

The original 0.684 implementation is documented in the linked sources below.


 <a id="credits"></a>

## 20. The People Behind the Evidence — Credits & Attribution

This notebook combines our original forensic investigation with a credited reproduction of a public inference ensemble. The forensic EDA, visual analysis and research narrative are distinct from the externally developed inference architecture and pretrained components.

<div style="border:1px solid #D8E5EC;background:#F7FAFC;border-radius:12px;padding:15px 16px;margin:16px 0 20px">
  <div style="font-size:11px;letter-spacing:1.4px;color:#54758B;font-weight:800;margin-bottom:10px">OPEN RESEARCH · CONTRIBUTORS AND SOURCES</div>
  <div style="display:flex;flex-wrap:wrap;gap:8px">
    <a href="https://www.kaggle.com/lbjyang" style="flex:1 1 155px;text-decoration:none;background:white;border:1px solid #D3E2E9;border-radius:8px;padding:11px 12px;color:#225F76;font-size:13px;font-weight:700">lbjyang</a>
    <a href="https://www.kaggle.com/lightningv08" style="flex:1 1 155px;text-decoration:none;background:white;border:1px solid #D3E2E9;border-radius:8px;padding:11px 12px;color:#225F76;font-size:13px;font-weight:700">Vladislav Lavrentev</a>
    <a href="https://www.kaggle.com/concyclics" style="flex:1 1 155px;text-decoration:none;background:white;border:1px solid #D3E2E9;border-radius:8px;padding:11px 12px;color:#225F76;font-size:13px;font-weight:700">CHEN Han</a>
    <a href="https://www.kaggle.com/omkarpattanaik123" style="flex:1 1 155px;text-decoration:none;background:white;border:1px solid #D3E2E9;border-radius:8px;padding:11px 12px;color:#225F76;font-size:13px;font-weight:700">Omkar Pattanaik</a>
    <a href="https://www.kaggle.com/omerfarukmerey" style="flex:1 1 155px;text-decoration:none;background:white;border:1px solid #D3E2E9;border-radius:8px;padding:11px 12px;color:#225F76;font-size:13px;font-weight:700">Omer Faruk Merey</a>
    <a href="https://www.kaggle.com/areebbb" style="flex:1 1 155px;text-decoration:none;background:white;border:1px solid #D3E2E9;border-radius:8px;padding:11px 12px;color:#225F76;font-size:13px;font-weight:700">Areeb Ali</a>
    <a href="https://www.kaggle.com/dhruvtibarewal" style="flex:1 1 155px;text-decoration:none;background:white;border:1px solid #D3E2E9;border-radius:8px;padding:11px 12px;color:#225F76;font-size:13px;font-weight:700">Dhruv Tibarewal</a>
  </div>
</div>

### Direct inference reference

**Clean LoRA + Qwen3-Embedding Ensemble · 0.684**

The inference implementation in this notebook is reproduced from the publicly shared **[Qwen Embedding + Clean LoRA Ensemble v14](https://drive.google.com/file/d/1XN-laE1rKj6DrDqxW60au_zHu9Xp8voZ/view)**.

The same executable implementation appears in the following public notebook references:

- **[lbjyang](https://www.kaggle.com/lbjyang)** — **[NTL 684 Ozertuu Mirror](https://www.kaggle.com/code/lbjyang/ntl-684-ozertuu-mirror)**. Public inference notebook reporting 0.684.
- **[0.684 Clean Baseline](https://drive.google.com/file/d/1W04B6owxk56Rx0wXAYXqAzIHnR-JCXRF/view)** — another supplied version containing the same executable code.

The three supplied versions have identical inference code. The ensemble uses a **72/28 Clean LoRA blend**, **40% native Qwen3 embedding**, **3% GSQ**, **15% complementary component weight**, and **Sinkhorn adjustment with parameters 0.30/0.35**.

These sources provide the inference implementation reproduced here. Their shared code does not, by itself, establish which publication originally developed each component.

### Model lineage and prior published releases

**[Vladislav Lavrentev](https://www.kaggle.com/lightningv08)** — [**0.681 | Qwen3-Embedding LoRA + Qwen3.8-27B**](https://www.kaggle.com/code/lightningv08/0-681-qwen3-embedding-lora-qwen3-8-27b); [**0.680 ModernBERT LoRA Sparse Ensemble**](https://www.kaggle.com/code/lightningv08/0-680-modernbert-lora-sparse-ensemble); and the earlier [**0.666 weighted parent**](https://www.kaggle.com/code/lightningv08/0-666-qwen3-8-27b-lora-weighted-ensemble?scriptVersionId=365000313). His public architecture and trained components are major predecessors of the current ensemble.

**[CHEN Han (concyclics)](https://www.kaggle.com/concyclics)** — [**0.679 · Version 1**](https://www.kaggle.com/code/concyclics/0-679?scriptVersionId=365653660), for the trained multi-depth Qwen3-Embedding-8B five-fold expert, its heads and calibration.

**[Omer Faruk Merey](https://www.kaggle.com/omerfarukmerey)** — [**0.682 · GSQ 12% + Target Mix 0.50**](https://www.kaggle.com/code/omerfarukmerey/0-682-0-681-qwen3-8-27b-gsq-12-mix-0-50?scriptVersionId=367805055); public investigation of alternative GSQ and Sinkhorn settings in an earlier parent ensemble. His earlier [**0.668 Verifier + Matched Rows**](https://www.kaggle.com/code/omerfarukmerey/0-668-verifier-matched-rows-pool-prompts?scriptVersionId=365209484) also belongs to our notebook's early history.

**[Omkar Pattanaik](https://www.kaggle.com/omkarpattanaik123)** — [**V28 Sovereign Titan Apex**](https://www.kaggle.com/code/omkarpattanaik123/name-that-llm-v28-sovereign-titan-apex), the direct source of the preceding V27's **0.683** inference implementation. His WeMM embedding branch and sibling-style corrections belong to that earlier release rather than the Clean LoRA pipeline above.

**[aboodps](https://www.kaggle.com/aboodps)** — [**NTL 672 Sinkhorn 050**](https://www.kaggle.com/code/aboodps/ntl-672-sinkhorn-050), referenced in earlier releases.

**[heliosli](https://www.kaggle.com/heliosli)** — [**lv681 a15**](https://www.kaggle.com/code/heliosli/name-that-llm-lv681-a15), mount-path portability work in earlier public lineages.

**[Areeb Ali](https://www.kaggle.com/areebbb)** — [**0.673 | I'm Better!**](https://www.kaggle.com/code/areebbb/0-673-i-m-better/notebook) and [**0.670 | This Actually Works**](https://www.kaggle.com/code/areebbb/0-670-this-actually-works/notebook?scriptVersionId=366234454); ModernBERT/DeBERTa work used in our earlier releases.

### Contributors named in the broader public ensemble

| Contributor | Public contribution and reference |
|:--|:--|
| [Roman Tamrazov](https://www.kaggle.com/romantamrazov) | Style and model-family investigation · [Who Am I?](https://www.kaggle.com/code/romantamrazov/name-that-llm-who-am-i) |
| [mercurydust](https://www.kaggle.com/mercurydust) | Complementary text features · [Submission](https://www.kaggle.com/code/mercurydust/name-that-llm-submission) |
| [Dhruv Tibarewal](https://www.kaggle.com/dhruvtibarewal) | V5 LoRA adapters and public inference · [NLM v27](https://www.kaggle.com/code/dhruvtibarewal/nlm-v27) |
| [Nishant Dahal](https://www.kaggle.com/nishantdahal) | Two-stage LoRA and public adapters · [Two-Stage LoRA](https://www.kaggle.com/code/nishantdahal/two-stage-lora-part-two) |
| [Joseph Adamski](https://www.kaggle.com/josephadamski91) | ATLAS adapters and inference resources |
| [prvsiyan](https://www.kaggle.com/prvsiyan), [kirilltym](https://www.kaggle.com/kirilltym), [Lucas Boesen](https://www.kaggle.com/lucasboesen), [sangam21](https://www.kaggle.com/sangam21) | Public model-family, Gaussian and prompt-effect techniques |
| [pjmathematician](https://www.kaggle.com/pjmathematician), [vnton37nguyn](https://www.kaggle.com/vnton37nguyn), [zbigniewkirchner](https://www.kaggle.com/zbigniewkirchner), [crtproduction](https://www.kaggle.com/crtproduction) | Public pretrained assets and infrastructure across the broader lineage |

**Our contribution:** the five forensic experiments, prompt-controlled analysis, diagnostic validation, original figures, explanations, editorial composition and integration of the published ensemble into this research notebook. The fitted weights, trained adapters and inference architecture are credited to their public sources.